In [ ]:
# general imports
import os
import scipy.stats as stats
import itertools
from itertools import product, combinations_with_replacement
import pickle
from unidecode import unidecode
import pandas as pd
import json
import igraph as ig
import re

# plotting imports
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go

# kaleido is necessary for saving plots made with plotly
# this needs to be version 0.1 (pip install --upgrade kaleido=="0.1.0.post1") because higher versions crash, at least currently
import kaleido
import nbformat

# local imports
from unicode_utils import apply_greek_labels_heatmap, introduce_greek_letters, replace_greek_letters, apply_greek_labels
from helper_functions import get_common_index, leiden_clustering
from plotting_utils import cm_to_inches, setup_plotting_environment, plot_overview_labels_along_y_axis

setup_plotting_environment()

# ------- CRITICAL -------
alt_flag = False # use an alternative cell type set (more fine-grained)
# ------- CRITICAL -------
if not alt_flag:
    from constants import CELL_TYPE_COLORS, ALL_CELL_TYPES
else:
    from constants import CELL_TYPE_COLORS_ALT, ALL_CELL_TYPES_ALT
    ALL_CELL_TYPES = ALL_CELL_TYPES_ALT
    CELL_TYPE_COLORS = CELL_TYPE_COLORS_ALT

ALL_CELL_TYPES_EXCLUDE = [cell for cell in ALL_CELL_TYPES if cell not in ["ILC", "pDC"]] # sort of deprecated

# set up folders
input_data_folder = f"../../data/"
os.makedirs(input_data_folder, exist_ok=True)
overall_output_table_name = "SI_tables.xlsx"

if not alt_flag:
    base_plot_folder = f"./plots/"
    os.makedirs(base_plot_folder, exist_ok=True)
    intermediate_output_folder = f"./output/"
    os.makedirs(intermediate_output_folder, exist_ok=True)
    output_tables_folder = f"./output_tables/"
    os.makedirs(output_tables_folder, exist_ok=True)
    overall_output_table_name = os.path.join(output_tables_folder, overall_output_table_name)
if alt_flag:
    base_plot_folder = f"./plots_alt/"
    os.makedirs(base_plot_folder, exist_ok=True)
    intermediate_output_folder = f"./output_alt/"
    os.makedirs(intermediate_output_folder, exist_ok=True)
    output_tables_folder = f"./output_tables_alt/"
    os.makedirs(output_tables_folder, exist_ok=True)
    overall_output_table_name = os.path.join(output_tables_folder, overall_output_table_name)

def write_to_SI_table(df, sheet_name, mode="a"):
    if not os.path.exists(overall_output_table_name):
        mode = "w"
    else:
        mode = "a"
    with pd.ExcelWriter(overall_output_table_name, mode=mode, engine="openpyxl", if_sheet_exists="replace" if mode == "a" else None) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

# 1 - Load the data

In [ ]:
plot_folder = os.path.join(base_plot_folder, "1_data_loading")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# info on cytokines
cytokine_info = pd.read_excel("../annotations/cytokine_info.xlsx", index_col=0, sheet_name="all_cytokines")
cytokine_info_parse = cytokine_info[cytokine_info["parse_cytokine"]]
cytokine_info_order = pd.read_excel("../annotations/cytokine_info.xlsx", index_col=0, sheet_name="suggested_order")
gene_to_cytokine = cytokine_info.set_index("gene")['gene_to_cytokine'].dropna().to_dict()
cytokine_to_gene = {v: k for k, v in gene_to_cytokine.items()}
ALL_CYTOKINES_ORDER = cytokine_info['gene_to_cytokine'].dropna().tolist()

CYTOKINE_ORDER = list(cytokine_info_order["name"])
CYTOKINE_ORDER_WITH_PBS = ["PBS"] + CYTOKINE_ORDER
SINGLE_FAMILY_ORDER = cytokine_info_order["DEG_plot_family"].unique()

In [ ]:
curr_info = cytokine_info_parse.loc[CYTOKINE_ORDER].copy().reset_index()
curr_info['name'] = curr_info['name'].apply(introduce_greek_letters)
curr_info = curr_info[['name', 'gene', 'DEG_plot_family', 'receptor gene']]
curr_info = curr_info.rename(columns={
    'name': 'Cytokine',
    'gene': 'Gene(s)',
    'DEG_plot_family': 'Family',
    'receptor gene': 'Receptor gene(s)'
})
write_to_SI_table(curr_info, sheet_name="cytokine_screen")

curr_info = cytokine_info.copy().reset_index()
curr_info['name'] = curr_info['name'].apply(introduce_greek_letters)
curr_info = curr_info[['name', 'gene', 'family_level_1', 'family_level_2']]
curr_info = curr_info.rename(columns={
    'name': 'Cytokine',
    'gene': 'Gene(s)',
    'family_level_1': 'Family',
    'family_level_2': 'Subfamily'
})
write_to_SI_table(curr_info, sheet_name="all_cytokines")


In [ ]:
cytokine_name_to_level_1_family = cytokine_info.reset_index().set_index("name")["family_level_1"].to_dict()
cytokine_name_to_level_2_family = cytokine_info.reset_index().set_index("name")["family_level_2"].to_dict()

cytokine_level_1_order = list(dict.fromkeys(cytokine_name_to_level_1_family.values()))
cytokine_level_2_order = list(dict.fromkeys(cytokine_name_to_level_2_family.values()))

level2_to_level1 = cytokine_info.set_index("family_level_2")["family_level_1"].to_dict()

In [ ]:
# renaming files in case they don't have the correct names
# # --- IGNORE ---
# import re
# from pathlib import Path
# import sys

# def rename_replace(
#     directory: Path,
#     old: str,
#     new: str,
#     pattern: str = "*",
#     recursive: bool = False,
#     dry_run: bool = True,
#     overwrite: bool = False,
#     case_sensitive: bool = True):
#         if not directory.exists():
#                 raise FileNotFoundError(f"Directory does not exist: {directory}")
#         globber = directory.rglob if recursive else directory.glob
#         files = list(globber(pattern))

#         replaced = []
#         skipped = []
#         collisions = []

#         for f in files:
#                 if not f.is_file():
#                         continue
#                 name = f.name
#                 if case_sensitive:
#                         new_name = name.replace(old, new)
#                 else:
#                         # do a case-insensitive replace preserving original case where possible
#                         new_name = re.sub(re.escape(old), new, name, flags=re.IGNORECASE)

#                 if new_name == name:
#                         continue

#                 target = f.with_name(new_name)
#                 if target.exists():
#                         if overwrite:
#                                 action = "overwriting"
#                         else:
#                                 collisions.append((f, target))
#                                 skipped.append((f, "exists"))
#                                 continue

#                 if dry_run:
#                         print(f"[DRY RUN] {f} -> {target}")
#                 else:
#                         if target.exists() and overwrite:
#                                 target.unlink()
#                         try:
#                                 f.rename(target)
#                                 print(f"RENAMED: {f} -> {target}")
#                         except Exception as e:
#                                 print(f"ERROR renaming {f} -> {target}: {e}", file=sys.stderr)
#                                 skipped.append((f, str(e)))
#                                 continue

#                 replaced.append((f, target))

#         return {"renamed": replaced, "skipped": skipped, "collisions": collisions}

# rename_replace(Path("../../data/pseudobulks/"), old="other", new="Granulocyte", overwrite=True, dry_run=False)

## 1.1 - Process pseudobulks

#### Convert pseudobulk files into a usable format

In [ ]:
if not alt_flag:
    relevant_cell_types = [
        "B cell",
		"CD4 T cell",
        "CD8 T cell",
        "Treg",
        "MAIT",
		"NKT",
		"NK CD56hi",
        "NK CD56low",
        "CD14 Mono",
        "CD16 Mono", 
        "cDC",
        "pDC",
        "granulocyte",
        "ILC",
        "HSPC",
        "plasmablast",
    ]
else:
    relevant_cell_types = [
		"B Naive",
        "B Intermediate",
        "CD4 Naive",
        "CD4 Memory",
        "CD8 Naive",
        "CD8 Memory",
        "Treg",
        "MAIT",
        "CD14 Mono",
        "CD16 Mono",
		"NK CD56hi",
        "NK CD56low",
        "NKT",
        "cDC",
        "pDC",
        "granulocyte",
        "ILC",
        "HSPC",
        "plasmablast",
    ]
    
relevant_cell_type_conversion = {
        "B": "B cell",
        "Naive_B_cell": "B Naive",
		"Intermediate_B_cell": "B Intermediate",
        "CD4": "CD4 T cell",
        "CD4_Naive_T_cell": "CD4 Naive",
        "CD4_Memory_T_cell": "CD4 Memory",
        "CD8": "CD8 T cell",
        "CD8_Naive_T_cell": "CD8 Naive",
        "CD8_Memory_T_cell": "CD8 Memory",
        "Treg": "Treg",
        "MAIT": "MAIT",
        "NKT": "NKT",
        "cDC": "cDC",
        "pDC": "pDC",
        "NK": "Natural Killer",
        "NK_CD56hi": "NK CD56hi",
        "NK_CD56low": "NK CD56low",
        "Mono": "Monocyte",
        "CD14_Mono": "CD14 Mono",
        "CD16_Mono": "CD16 Mono",
        "HSPC": "HSPC",
        "Granulocyte": "granulocyte",
        "ILC": "ILC",
        "Plasmablast": "plasmablast",
    }
relevant_cell_to_pseudobulk = {v: k for k, v in relevant_cell_type_conversion.items()}
relevant_cells_pseudobulk = [relevant_cell_to_pseudobulk[c] for c in relevant_cell_types]

assert all([c in relevant_cell_type_conversion.values() for c in relevant_cell_types])

In [ ]:
pseudobulk_files = os.listdir(os.path.join(input_data_folder, "pseudobulks"))

cell_types = []
cytokines = []
cell_type_counts = {}
for file in pseudobulk_files:
    cell_type = "_".join(file.split("_")[1:-1])
    cell_types.append(cell_type)
    cytokines.append(file.split("_")[-1].split(".")[0])
    cell_type_counts[cell_type] = cell_type_counts.get(cell_type, 0) + 1
cell_types = list(set(cell_types))
cytokines = sorted(list(set(cytokines)))

# make sure everything is in order
for cell_type in cell_types:
    for cytokine in cytokines:
        if not cytokine in CYTOKINE_ORDER_WITH_PBS:
            print("Not in cytokine_info_parse:", cytokine)

In [ ]:
# optional code to load in PBS pseudobulks and combine them all into a single dataframe (for inspection purposes)
# # --- IGNORE ---
# import anndata as ad
# from processing_utils import normalize_df_to_rpm, convert_to_two_level_multiindex

# all_pbs_dfs = []
# all_counts_by_condition_pbs = []
# for cell_type in relevant_cell_types:
#     cell_type_pseudobulk = relevant_cell_to_pseudobulk[cell_type]
#     adata = ad.read_h5ad(input_data_folder + f"pseudobulks/pbulk_{cell_type_pseudobulk}_PBS.h5ad")
    
#     df = adata.to_df()
#     # Split the index at _ to create a multiindex
#     df.index = pd.MultiIndex.from_tuples([idx.split('_') for idx in df.index])
#     # prepend the cell_type to the second level of the index
#     df.index = df.index.set_levels([cell_type + "_" + x for x in df.index.levels[1]], level=1)
#     all_pbs_dfs.append(df)
    
#     # also save counts_by_condition
#     counts = adata.obs["psbulk_n_cells"]
#     # prepend the cell_type to the second level of the index
#     counts.index = [i + "_" + cell_type for i in counts.index]
#     all_counts_by_condition_pbs.append(counts)
    
# sum_gene_count_df_raw_by_donor_pbs = pd.concat(all_pbs_dfs, axis=0)
# sum_gene_count_df_raw_by_donor_pbs.fillna(0, inplace=True)

# all_counts_by_condition_pbs = pd.concat(all_counts_by_condition_pbs, axis=0)
# all_counts_by_condition_pbs.index = pd.MultiIndex.from_tuples([idx.split('_') for idx in all_counts_by_condition_pbs.index])
# all_counts_by_condition_pbs.index = all_counts_by_condition_pbs.index.droplevel(1)

# sum_gene_count_df_raw_by_donor_pbs.index = pd.MultiIndex.from_tuples([
#     (i[0], i[1].replace("_PBS", "")) for i in sum_gene_count_df_raw_by_donor_pbs.index
# ])

In [ ]:
import anndata as ad
from processing_utils import normalize_df_to_rpm, convert_to_two_level_multiindex

all_dfs = []
all_counts_by_condition = []
for cell_type in relevant_cell_types:
    for cytokine in CYTOKINE_ORDER_WITH_PBS:
        cell_type_pseudobulk = relevant_cell_to_pseudobulk[cell_type]
        adata = ad.read_h5ad(input_data_folder + f"pseudobulks/pbulk_{cell_type_pseudobulk}_{cytokine}.h5ad")
        
        df = adata.to_df()
        # Split the index at _ to create a multiindex
        df.index = pd.MultiIndex.from_tuples([idx.split('_') for idx in df.index])
        # prepend the cell_type to the second level of the index
        df.index = df.index.set_levels([cell_type + "_" + x for x in df.index.levels[1]], level=1)
        all_dfs.append(df)
        
        # also save counts_by_condition
        counts = adata.obs["psbulk_n_cells"]
        # prepend the cell_type to the second level of the index
        counts.index = [i + "_" + cell_type for i in counts.index]
        all_counts_by_condition.append(counts)
        
sum_gene_count_df_raw_by_donor = pd.concat(all_dfs, axis=0)
sum_gene_count_df_raw_by_donor.fillna(0, inplace=True)

average_sum_gene_count_df = sum_gene_count_df_raw_by_donor.groupby(level=1).mean() # median?
average_sum_gene_count_df = convert_to_two_level_multiindex(average_sum_gene_count_df)

sum_gene_count_df_norm_by_donor = normalize_df_to_rpm(sum_gene_count_df_raw_by_donor)
average_norm_sum_gene_count_df = sum_gene_count_df_norm_by_donor.groupby(level=1).mean() # median?
average_norm_sum_gene_count_df = convert_to_two_level_multiindex(average_norm_sum_gene_count_df)
average_norm_sum_gene_count_df = average_norm_sum_gene_count_df.loc[:, ALL_CELL_TYPES]

# std_norm_sum_gene_count_df = sum_gene_count_df_norm_by_donor.groupby(level=1).std() # don't end up using this one and it's pretty big
# std_norm_sum_gene_count_df = convert_to_two_level_multiindex(std_norm_sum_gene_count_df)
# std_norm_sum_gene_count_df = std_norm_sum_gene_count_df.loc[:, ALL_CELL_TYPES]

all_counts_by_condition = pd.concat(all_counts_by_condition)
all_counts_by_condition.index = pd.MultiIndex.from_tuples([idx.split('_') for idx in all_counts_by_condition.index])
all_counts_by_condition = all_counts_by_condition.swaplevel(0, 2)

if alt_flag:
    sum_gene_count_name = "pbulk_average_sum_gene_count_df_alt.parquet"
    norm_sum_gene_count_name = "pbulk_average_norm_sum_gene_count_df_alt.parquet"
    norm_sum_gene_count_name_std = "pbulk_std_norm_sum_gene_count_df_alt.parquet"
else:
    sum_gene_count_name = "pbulk_average_sum_gene_count_df.parquet"
    norm_sum_gene_count_name = "pbulk_average_norm_sum_gene_count_df.parquet"
    norm_sum_gene_count_name_std = "pbulk_std_norm_sum_gene_count_df.parquet"

if not os.path.exists(os.path.join(input_data_folder, sum_gene_count_name)):
    average_sum_gene_count_df.to_parquet(os.path.join(input_data_folder, sum_gene_count_name))
    average_norm_sum_gene_count_df.to_parquet(os.path.join(input_data_folder, norm_sum_gene_count_name))
    # std_norm_sum_gene_count_df.to_parquet(os.path.join(input_data_folder, norm_sum_gene_count_name_std))

In [ ]:
all_counts_by_condition = all_counts_by_condition.unstack().fillna(0).stack()

In [ ]:
# get by donor values raw
sum_gene_count_df_raw_by_donor = convert_to_two_level_multiindex(sum_gene_count_df_raw_by_donor, flip_indices_first=True)
sum_gene_count_df_raw_by_donor.index.names = ['cell_type', 'cytokine', 'donor']
sum_gene_count_df_raw_by_donor = sum_gene_count_df_raw_by_donor.T

# get by donor values normalized
sum_gene_count_df_norm_by_donor = convert_to_two_level_multiindex(sum_gene_count_df_norm_by_donor, flip_indices_first=True)
sum_gene_count_df_norm_by_donor.index.names = ['cell_type', 'cytokine', 'donor']
sum_gene_count_df_norm_by_donor = sum_gene_count_df_norm_by_donor.T

In [ ]:
# can read from file instead of recomputing but it's not much faster. could probably write out as .parquet instead
# average_norm_sum_gene_count_df = pd.read_csv(os.path.join(input_data_folder, norm_sum_gene_count_name), index_col=0, header=[0, 1])
# average_sum_gene_count_df = pd.read_csv(os.path.join(input_data_folder, sum_gene_count_name), index_col=0, header=[0, 1])

average_norm_sum_gene_count_df = average_norm_sum_gene_count_df[ALL_CELL_TYPES]
average_sum_gene_count_df = average_sum_gene_count_df[ALL_CELL_TYPES]

# separate out the control data
control_norm_gene_count_df = average_norm_sum_gene_count_df.loc[:, (slice(None), "PBS")]
control_norm_gene_count_df.columns = control_norm_gene_count_df.columns.droplevel(1) # drop the "PBS" from the index

In [ ]:
questionable_genes = set(pd.read_csv(os.path.join(output_tables_folder, "questionable_genes.csv"))['gene'])
unfiltered_genes = average_norm_sum_gene_count_df.index

inclusive_filters_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    df = average_sum_gene_count_df[cell_type]
    df = df[df.max(axis=1) > 20]
    df = df[~df.index.isin(questionable_genes)]
    print(cell_type, len(df))
    inclusive_filters_by_cell_type[cell_type] = df.index

In [ ]:
# write out per-donor normalized data for each cell type for loading into other downstream analyses
# --- IGNORE ---
# for cell_type in ALL_CELL_TYPES:
#     print(f"Processing {cell_type}...")
    
#     subset = sum_gene_count_df_norm_by_donor.xs(cell_type, level=0, axis=1)
#     subset = subset.loc[subset.index.intersection(inclusive_filters_by_cell_type[cell_type])]

#     long_subset = subset.stack(level=[0, 1])
#     long_subset = long_subset.reset_index()
#     long_subset.columns = ['gene', 'cytokine', 'donor', 'count']
#     long_subset['cell_type'] = cell_type
    
#     long_subset.to_parquet(os.path.join(output_tables_folder, f"pb_by_donor_{cell_type}.parquet"))

## 1.2 - Load DEGs

In [ ]:
edger_results = pd.read_csv(os.path.join(input_data_folder, "DEGs_all_with_pbs_DE_count.csv"), index_col=0)
edger_results = edger_results.rename(columns={"celltype": "cell_type"})

In [ ]:
# for cell_type in edger_results['cell_type'].unique():
#     if cell_type not in relevant_cell_types:
#         print("Missing cell type in edgeR results:", cell_type)
#         continue
#     else:
#         print("Retaining cell type in edgeR results:", cell_type)

In [ ]:
# set p-values for genes that fail the well test to 1
# don't do this for the log_fc, creates confusing results
mask = edger_results['pbs_well'] < 3.5
edger_results.loc[mask, "p_value"] = 1
edger_results.loc[mask, "adj_p_value"] = 1

# remove questionable genes from edger results
mask = edger_results['gene'].isin(questionable_genes)
edger_results.loc[mask, ["log_fc", "p_value", "adj_p_value"]] = np.nan

In [ ]:
def create_edger_val_df(result_df, val_type):
    """
    Creates a DataFrame from EdgeR results with a MultiIndex of (cell_type, cytokine) columns.
    
    Parameters:
    -----------
    result_df : pandas.DataFrame
        The EdgeR results DataFrame containing gene, cell_type, cytokine, and value columns.
    val_type : str
        The column name in result_df to use as values (e.g., 'adj_p_value', 'log_fc', 'logCPM').
        
    Returns:
    --------
    pandas.DataFrame
        A pivoted DataFrame with genes as index and MultiIndex columns of (cell_type, cytokine),
        containing the specified values. Missing combinations are filled with NaN.
    """
    # rename 
    df = result_df[[val_type, "gene", "cell_type", "cytokine"]]
    df = df.pivot(index="gene",columns=["cell_type", "cytokine"], values=val_type)
    # Create tuples of (cell_type, cytokine) for all combinations
    column_tuples = [(cell_type, cytokine) 
                    for cell_type in df.columns.get_level_values(0).unique() 
                    for cytokine in df.columns.get_level_values(1).unique()]
    df = df.reindex(pd.MultiIndex.from_tuples(column_tuples), axis=1)
    return df

edger_results["cell_type"] = edger_results["cell_type"].map(relevant_cell_type_conversion)

In [ ]:
# edger_results_filtered = edger_results[
#     (edger_results["adj_p_value"] < 0.05) &
#     (edger_results["log_fc"].abs() > 0.25)
#     ].copy()

# edger_results_filtered = edger_results_filtered[['cytokine', 'cell_type', 'gene', 'log_fc', 'logCPM', 'F', 'p_value', 'adj_p_value', 'num_donors']]
# edger_results_filtered.groupby("cell_type").size(
# ).sort_values()
# if not os.path.exists(os.path.join(output_tables_folder, "EdgeR_DEGs_filtered.xlsx")):
#     edger_results_filtered.to_excel(os.path.join(output_tables_folder, "EdgeR_DEGs_filtered.xlsx"), index=False)

In [ ]:
edger_results = edger_results[edger_results["cell_type"].isin(relevant_cell_types)].copy()
edger_results.groupby("cell_type").size().sort_values()

In [ ]:
if not os.path.exists(os.path.join(output_tables_folder, "EdgeR_Log2FC.parquet")):
    edger_df_log2FoldChange = create_edger_val_df(edger_results, val_type="log_fc")
    edger_df_log2FoldChange.to_parquet(os.path.join(output_tables_folder, "EdgeR_Log2FC.parquet"))
    
if not os.path.exists(os.path.join(output_tables_folder, "EdgeR_all_processed.parquet")):
    edger_results.to_parquet(os.path.join(output_tables_folder, "EdgeR_all_processed.parquet"))

In [ ]:
# not valid for multi-factor design
# edger_results['SE_logFC'] = edger_results['log_fc'].abs() / np.sqrt(edger_results['F']+0.01)
# edger_results['adjusted_fc'] = edger_results.apply(shrink_FCs, axis=1)

# plt.figure(figsize=(4, 3))
# plt.hist(edger_results["adjusted_fc"], bins = np.arange(-4, 4, 0.1))
# plt.axvline(x=0.5, color='red', linestyle='--')
# plt.axvline(x=-0.5, color='red', linestyle='--')
# plt.show()

# len(edger_results[edger_results["log_fc"].abs() > 0.5]) / len(edger_results)

In [ ]:
edger_df_padj = create_edger_val_df(edger_results, "adj_p_value")
edger_df_log2CPM = create_edger_val_df(edger_results, "logCPM")
edger_df_log2FoldChange = create_edger_val_df(edger_results, "log_fc")

In [ ]:
# sanity check to see that fold changes ~match what would be expected naively (also need this later)
raw_fc_values = average_norm_sum_gene_count_df.groupby(level=0, axis=1).apply(
    lambda g: g.div(g.xs("PBS", level=1, axis=1).iloc[:, 0], axis=0)
)
raw_fc_values.columns = raw_fc_values.columns.droplevel(0)

plt.figure(figsize=cm_to_inches((4, 3)))
x = edger_df_log2FoldChange["CD4 T cell", "IFN-beta"].dropna()
y = np.log2(raw_fc_values["CD4 T cell", "IFN-beta"].dropna())
x, y = get_common_index(x, y)

r = stats.pearsonr(x, y)
plt.scatter(x, y, s=1)
plt.xlabel("EdgeR Log2FC")
plt.ylabel("Naive Log2FC")
plt.title(f"CD4 T cell, IFN-β\nPearson r={r.statistic:.2f}")
plt.show()

## 1.3 - Get gene filters

In [ ]:
import pybiomart as pbm

# hg38
dataset_human = pbm.Dataset(name='hsapiens_gene_ensembl',  host='http://www.ensembl.org')
annot = dataset_human.query(attributes=['chromosome_name', 'transcription_start_site', 'strand', 'external_gene_name', 'transcript_biotype'])

# remove mitochondrial genes
annot['Chromosome/scaffold name'] = annot['Chromosome/scaffold name'].astype(str)
annot = annot[~annot['Chromosome/scaffold name'].str.contains('MT')]

transcript_filter = annot['Transcript type'].isin(['protein_coding'])
annot_filtered = annot[transcript_filter]
protein_coding_genes = set(annot_filtered['Gene name'].unique())
len(protein_coding_genes)

# ribosomal genes were taken from https://www.genenames.org/data/genegroup/#!/group/1054
with open("../annotations/ribosomal_genes.json", "r") as f:
    ribosomal_gene_info = json.load(f)
    
ribosomal_gene_info = pd.DataFrame(ribosomal_gene_info)
ribosomal_genes = set(ribosomal_gene_info["approvedSymbol"].unique())

protein_coding_genes = protein_coding_genes.difference(ribosomal_genes)
len(protein_coding_genes)

In [ ]:
# here, we could apply the filtering to only protein-coding genes
# don't do this except for looking at donor-specific gene regulation
# edger_df_padj = edger_df_padj[edger_df_padj.index.isin(protein_coding_genes)]
# edger_df_log2FoldChange = edger_df_log2FoldChange[edger_df_log2FoldChange.index.isin(protein_coding_genes)]
# average_norm_sum_gene_count_df = average_norm_sum_gene_count_df[average_norm_sum_gene_count_df.index.isin(protein_coding_genes)]
# average_sum_gene_count_df = average_sum_gene_count_df[average_sum_gene_count_df.index.isin(protein_coding_genes)]

# for cell_type in ALL_CELL_TYPES:
#     inclusive_filters_by_cell_type[cell_type] = inclusive_filters_by_cell_type[cell_type][inclusive_filters_by_cell_type[cell_type].isin(protein_coding_genes)]

## 1.4 - Overview of significance and fold change distribution

None of these plots are particularly interesting, so don't put in final publication

In [ ]:
edger_df_log2FoldChange_filtered = edger_df_log2FoldChange[edger_df_padj < 0.05]

positive_values = (edger_df_log2FoldChange_filtered > 0.5).sum(axis=0)
negative_values = (edger_df_log2FoldChange_filtered < -0.5).sum(axis=0)
total_values = positive_values + negative_values

plt.figure(figsize=cm_to_inches((5, 3)))
plt.hist(edger_df_log2FoldChange_filtered.values.flatten(), bins=np.arange(-3, 3, 0.05))

# Calculate total positive and negative values
total_positive = np.sum(positive_values)
total_negative = np.sum(negative_values)

# Add text annotations for counts
plt.text(0.05, 0.95, f"log2FC < -0.5:\n{total_negative}", 
         transform=plt.gca().transAxes, 
         verticalalignment='top', 
         horizontalalignment='left',
         fontsize=6)

plt.text(0.95, 0.95, f"log2FC > 0.5:\n{total_positive}", 
         transform=plt.gca().transAxes, 
         verticalalignment='top', 
         horizontalalignment='right',
         fontsize=6)

plt.xlabel("Log2 Fold Change")
plt.ylabel("Count")
plt.title("Distribution of significant log2 fold changes")

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"1.4_edger_df_log2FoldChange_filtered_distribution.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
edger_df_log2FoldChange_filtered = edger_df_log2FoldChange[edger_df_padj < 0.05]

# Create subplots for each cell type
if not alt_flag:
    fig, axes = plt.subplots(3, 5, figsize=cm_to_inches((12, 7)))
else:
    fig, axes = plt.subplots(4, 5, figsize=cm_to_inches((12, 7)))
fig.tight_layout(pad=2.0)

axes = axes.flatten()
for i, cell_type in enumerate(ALL_CELL_TYPES):
    print(cell_type)
    # Filter data for this cell type
    cell_type_data = edger_df_log2FoldChange_filtered.xs(cell_type, level=0, axis=1)
    
    # Calculate positive and negative values for this cell type
    positive_values_cell = (cell_type_data > 0.5).sum(axis=0)
    negative_values_cell = (cell_type_data < -0.5).sum(axis=0)
    
    # Calculate totals
    total_positive_cell = np.sum(positive_values_cell)
    total_negative_cell = np.sum(negative_values_cell)
    
    # Plot histogram
    n, bins, patches = axes[i].hist(cell_type_data.values.flatten(), bins=np.arange(-3, 3, 0.05), density=True, color=CELL_TYPE_COLORS[cell_type])
    
    # Add labels
    axes[i].set_xlabel(f"log$_{2}$FC")
    axes[i].set_ylabel("density")
    axes[i].set_ylim(0, max(n) * 1.25)
    # Add title in the upper left corner
    axes[i].text(0.05, 0.95, f"{cell_type}", transform=axes[i].transAxes, 
                 va='top', ha='left', fontsize=7)
    axes[i].label_outer()
    
    # Set tick length to 0 for inner plots
    axes[i].tick_params(length=0)
    
    # Remove y ticks
    axes[i].set_yticks([])
    
# Delete unused plots
for i in range(len(ALL_CELL_TYPES), len(axes)):
    fig.delaxes(axes[i])
    
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"1.4_edger_df_log2FoldChange_filtered_distribution_by_cell_type.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
positive_values = positive_values[total_values > 10]
negative_values = negative_values[total_values > 10]
ratio = positive_values / (positive_values + negative_values)
ratio_mean = ratio.groupby(level=1).mean()
ratio_std = ratio.groupby(level=1).std()
ratio_mean = ratio_mean.sort_values(ascending=False)
ratio_std = ratio_std.loc[ratio_mean.index]

# Create a bar plot of the ratio of positive to total DEGs for each cytokine with error bars
plt.figure(figsize=cm_to_inches((18, 4)))
x_pos = range(len(ratio_mean))
bars = plt.bar(x_pos, ratio_mean.values, yerr=ratio_std.values, capsize=2, error_kw={'elinewidth': 1})
# Apply Greek letters to x-tick labels
x_labels = [introduce_greek_letters(lbl) for lbl in ratio_mean.index]
plt.xticks(x_pos, x_labels, rotation=90, fontsize=5.5)
plt.yticks(fontsize=5.5)
plt.ylabel(r'mean($\frac{\mathrm{DEG}(log2FC>0.5)}{\mathrm{DEG}(log2FC>0.5) + \mathrm{DEG}(log2FC<-0.5)}$)')

plt.ylim(0, 1)
plt.xlim(-1, len(ratio_mean))  # Reduce space on left and right of x-axis
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"1.4_ratio_of_positive_to_total_DEGs.{format}"), bbox_inches="tight", dpi=300)

# 2 - Summarize results

## 2.1 -  Calculate the magnitude of the response

In [ ]:
plot_folder = os.path.join(base_plot_folder, "2_result_summary")
os.makedirs(plot_folder, exist_ok=True)

### 2.1.1 - Calculate Euclidean distances

In [ ]:
from scipy.spatial.distance import euclidean
from scipy.stats import mstats

def compute_response_euclidean_distance(df, cell_type, cytokine, control="PBS",
                                        dropna=False, min_genes=None, print_min_gene_warning=False,
                                        length_norm=False):
    """
    Compute the Euclidean distance between the treatment and control cytokine response for a given cell type.

    Parameters:
    - df (pd.DataFrame): A DataFrame containing gene expression data with a MultiIndex of genes,(cell_type, cytokine).
    - cell_type (str): The specific cell type for which the distance is to be computed.
    - cytokine (str): The treatment cytokine whose response is being compared.
    - control (str, optional): The control cytokine to compare against, default is "PBS".

    Returns:
    - float: The Euclidean distance between the treatment and control responses for the specified cell type and cytokine.
    """
    df_treatment = df.loc[:, (cell_type, cytokine)]
    df_control = df.loc[:, (cell_type, control)]
    
    if dropna:
        df_treatment = df_treatment.dropna()
        df_control = df_control.dropna()

    common_index = df_treatment.index.intersection(df_control.index)
    df_treatment = df_treatment.loc[common_index]
    df_control = df_control.loc[common_index]
    
    # compute distance
    if min_genes is not None:
        if len(common_index) < min_genes:
            if print_min_gene_warning:
                print(f"Not enough genes for {cell_type} {cytokine}")
            return np.nan
    
    distance = euclidean(df_treatment.values.flatten(), df_control.values.flatten())
    if length_norm:
        distance = distance / np.sqrt(len(df_treatment))
    return distance

def winsorize_and_normalize_distances(distances: pd.Series):
    """
    Winsorize and normalize the distances to a scale of 0 to 100.

    Parameters:
    - distances (pd.Series): A pandas Series containing the distances to be winsorized and normalized.

    Returns:
    - pd.Series: A pandas Series of the same index as the input, containing the normalized distances.
                 The distances are first winsorized to limit extreme values, then scaled to a range of 0 to 100.
    """
    # winsorize the distances (first limit: increase smallest values, second limit: decrease largest values)
    winsorized_distances = mstats.winsorize(distances, limits=[0, 0.05])
    # make it a pandas series with the same index
    winsorized_distances = pd.Series(winsorized_distances, index=distances.index)
    
    min_val, max_val = winsorized_distances.min(), winsorized_distances.max()
    normalized_distances = 100 * (winsorized_distances - min_val) / (max_val - min_val)
    
    return normalized_distances

In [ ]:
all_normalized_euclidean_distances = {}
for cell_type in ALL_CELL_TYPES:
    distances = {}
    
    # get the inclusive filter and average normalized log2 data
    all_inclusive_filter = inclusive_filters_by_cell_type[cell_type] # this probably does very little
    average_norm_filter = np.log2(average_norm_sum_gene_count_df.loc[all_inclusive_filter, :] + 1)
    
    for cytokine in CYTOKINE_ORDER:
        distance = compute_response_euclidean_distance(average_norm_filter, cell_type, cytokine)
        distances[cytokine] = distance
        
    distances_series = pd.Series(distances)
    all_normalized_euclidean_distances[cell_type] = winsorize_and_normalize_distances(distances_series)
all_normalized_euclidean_distances = pd.DataFrame(all_normalized_euclidean_distances)

In [ ]:
%%capture output
for cell_type in ALL_CELL_TYPES:
    plt.figure(figsize=(3, 2.4))
    plt.hist(all_normalized_euclidean_distances[cell_type])
    plt.xlabel("Euclidean distance")
    plt.ylabel("Count")
    plt.title(cell_type)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"2.1_normalized_euclidean_distance_{cell_type}.png"), bbox_inches="tight", dpi=300)

### 2.1.2 - Calculate p-value distances

R_{\text{strength}} = \sum_{i=1}^{N} \left| \log_2 \text{FC}_i \right| \cdot -\log_{10}(p_{\text{adj},i})

In [ ]:
# get the absolute value of the log2 fold change
# the filled values will drop out in the sum anyway, need this to avoid errors
edger_df_log2FoldChange_abs = edger_df_log2FoldChange.fillna(0).abs()
edger_df_padj_fillna = edger_df_padj.fillna(1).copy()

# cap the p-values at 1e-10 and log10-transform
edger_df_padj_fillna = edger_df_padj_fillna.clip(lower=1e-10)
edger_df_padj_fillna = -np.log10(edger_df_padj_fillna)

# this implements the formula above
all_logfoldchange_padj_distances = edger_df_log2FoldChange_abs * edger_df_padj_fillna
all_logfoldchange_padj_distances = all_logfoldchange_padj_distances.sum()

# make it a dataframe (cytokine x cell_type)
all_logfoldchange_padj_distances = all_logfoldchange_padj_distances.unstack(level=0)
all_logfoldchange_padj_distances = all_logfoldchange_padj_distances.loc[CYTOKINE_ORDER]

# apply normalization and winsorization
for cell_type in ALL_CELL_TYPES:
    distances_series = all_logfoldchange_padj_distances[cell_type]
    all_logfoldchange_padj_distances[cell_type] = winsorize_and_normalize_distances(distances_series)

### 2.1.3 - Merge distance measurements and calculate thresholds for a strong response

In [ ]:
plt.figure(figsize=cm_to_inches((5, 4)))
for cell_type in ALL_CELL_TYPES:
    r = np.corrcoef(all_normalized_euclidean_distances[cell_type],
                all_logfoldchange_padj_distances[cell_type])
    plt.scatter(all_normalized_euclidean_distances[cell_type],
                all_logfoldchange_padj_distances[cell_type],
                label=f"{cell_type}: r={r[0, 1]:.2f}",
                color=CELL_TYPE_COLORS[cell_type],
                s = 2.5,
                alpha=0.75)
    
plt.xlabel("response magnitude\n(Euclidean)")
plt.ylabel("response magnitude\n(padj-log2FC)")
plt.tight_layout()
plt.legend(loc=[1,0], fontsize=6, ncol=1)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_normalized_distance_comparison.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
# merge the two distance measurements
all_normalized_distances_mean = (all_normalized_euclidean_distances + all_logfoldchange_padj_distances) / 200
all_normalized_distances_mean = all_normalized_distances_mean[ALL_CELL_TYPES] # reset to the correct order
# calculate relative distances (divide by the max distance for each cytokine)
all_relative_distances_mean = all_normalized_distances_mean.div(all_normalized_distances_mean.max(axis=1), axis=0)

In [ ]:
def convert_to_percentile_along_axis(df, axis, granularity=5):
    """Convert values to percentile ranks along specified axis with given granularity.
    
    Args:
        df: pandas DataFrame to convert
        axis: 0 for columns, 1 for rows 
        granularity: Size of percentile bins (e.g. 5 means bins of 0-5%, 5-10%, etc)
        
    Returns:
        DataFrame with values converted to percentile ranks
    """
    # Calculate percentile rank along axis
    percentile_df = df.rank(axis=axis, pct=True) * 100
    
    # Round to nearest granularity value
    percentile_df = (percentile_df / granularity).round() * granularity
    
    return percentile_df

all_normalized_distances_percentile = convert_to_percentile_along_axis(all_normalized_distances_mean, axis=0, granularity=5)

In [ ]:
# calculate the magntiude threshold for each cell type
magnitude_threshold_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    # get the mean of the low percentile cytokines
    low_percentile = all_normalized_distances_percentile[cell_type] < 35
    low_percentile_cytokines = low_percentile.index[low_percentile]
    low_percentile_mean = all_normalized_distances_mean[cell_type][low_percentile_cytokines].mean()
    
    # set the threshold to 3x the mean of the low percentile cytokines
    magnitude_threshold_by_cell_type[cell_type] = 3*low_percentile_mean
    
strong_impact_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    strong_impact_by_cell_type[cell_type] = all_normalized_distances_mean[cell_type] > magnitude_threshold_by_cell_type[cell_type]
strong_impact_by_cell_type = pd.DataFrame(strong_impact_by_cell_type, index=all_normalized_distances_mean.index)

In [ ]:
# Create a figure with subplots arranged in a grid
n_cell_types = len(ALL_CELL_TYPES)
n_rows = 3 if not alt_flag else 4
n_cols = 6
fig, axes = plt.subplots(n_rows, n_cols, figsize=cm_to_inches((17, 7)))

# Flatten axes array for easier iteration
axes_flat = axes.flatten()

for idx, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes_flat[idx]
    ax.hist(all_normalized_distances_mean[cell_type].values, bins=np.arange(0, 1.05, 0.05), color='skyblue')
    ax.axvline(magnitude_threshold_by_cell_type[cell_type], color="red", linestyle="--", label="impact threshold" if idx == 0 else "")
    if idx % n_cols == 0:  # Only show y-label on leftmost plots
        ax.set_ylabel("cytokine count")
    else:
        ax.set_yticks([])
    if idx >= len(ALL_CELL_TYPES) - n_cols:  # Only show x-label on bottom plots
        ax.set_xlabel("response magnitude")
        ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])
        ax.set_xticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1'])
    else:
        ax.set_xticks([])
    ax.set_title(cell_type, fontsize=7)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 30)
    if idx == 0:
        ax.legend(fontsize=6)

# Remove any empty subplots
for idx in range(len(ALL_CELL_TYPES), len(axes_flat)):
    fig.delaxes(axes_flat[idx])

plt.tight_layout()

# Save the figure
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_distance_distributions.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
all_normalized_distances_mean.to_csv(os.path.join(output_tables_folder, "2_response_magnitudes.csv"))
strong_impact_by_cell_type.to_csv(os.path.join(output_tables_folder, "2_strong_impact.csv"))
pd.Series(magnitude_threshold_by_cell_type).to_csv(os.path.join(output_tables_folder, "2_response_magnitude_thresholds.csv"))

In [ ]:
df = all_normalized_distances_mean.reset_index().copy()
df = df.rename(columns={"index": "Cytokine"})
df['Cytokine'] = df['Cytokine'].apply(introduce_greek_letters)
write_to_SI_table(df, sheet_name="response_magnitudes")

df = strong_impact_by_cell_type.reset_index().copy()
df = df.rename(columns={"index": "Cytokine"})
df['Cytokine'] = df['Cytokine'].apply(introduce_greek_letters)
write_to_SI_table(df, sheet_name="strong_impact")

In [ ]:
response_magnitudes = pd.read_csv(os.path.join(output_tables_folder, "2_response_magnitudes.csv"), index_col=0)

## 2.2 - Create an overview plot for DEGs and the response magnitude

In [ ]:
significant_results = pd.DataFrame(index=ALL_CELL_TYPES, columns=CYTOKINE_ORDER, data=0)
for cell_line in ALL_CELL_TYPES:
    padj_df = edger_df_padj[cell_line]
    log2fc_df = edger_df_log2FoldChange[cell_line]
    num_significant = ((padj_df < 0.05) & (log2fc_df.abs() > 0.25)).sum(axis=0)
    significant_results.loc[cell_line, CYTOKINE_ORDER] = num_significant[CYTOKINE_ORDER].values

In [ ]:
donor_independent_counts = all_counts_by_condition.groupby(level=[0,1]).sum()

# Create a figure for histograms of log10(significant genes+1) per cell type
plt.figure(figsize=cm_to_inches((5, 7)))
ax = plt.gca()

# Calculate log10 of significant results (adding 1 to avoid log(0))
log_counts = np.log10(donor_independent_counts + 1)

# Define vertical offset between histograms
offset = 0.5

# Plot histograms for each cell type with vertical offset
for i, cell_type in enumerate(ALL_CELL_TYPES[::-1]):
    # Get data for this cell type
    data = log_counts.loc[cell_type]
    data = data[data.index != "PBS"].values
    
    # Create histogram
    counts, bins = np.histogram(data, bins=100, range=(1, 5))
    
    # Normalize counts to have similar heights
    if counts.max() > 0:
        counts = counts / counts.max() * 0.4
    
    # Plot histogram with vertical offset
    plt.bar(bins[:-1], counts, width=bins[1]-bins[0], bottom=i*offset, 
            alpha=0.7, label=cell_type if i == 0 else "", color=CELL_TYPE_COLORS[cell_type])
    
    # Add cell type label
    plt.text(1.3, i*offset + 0.2, cell_type, ha='right', va='center', fontsize=6)
    
    # Add a horizontal line at the bottom of each histogram instead of a box
    plt.plot([1.5, 5], [i*offset, i*offset], color='black', linestyle='-', linewidth=0.5)

# Set axis labels and title
plt.xlabel(r'log$_{10}$(cell type # by per stim. + 1)')
plt.yticks([])
plt.xlim(1.5, 5)
plt.ylim(0, len(ALL_CELL_TYPES)*offset)

plt.tight_layout()
# # Save the figure
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_count_distributions.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
# Create a figure for histograms of log10(significant genes+1) per cell type
plt.figure(figsize=cm_to_inches((5, 7)))
ax = plt.gca()

# Calculate log10 of significant results (adding 1 to avoid log(0))
log_significant = np.log10(significant_results + 1)

# Define vertical offset between histograms
offset = 0.5

# Plot histograms for each cell type with vertical offset
for i, cell_type in enumerate(ALL_CELL_TYPES[::-1]):
    # Get data for this cell type
    data = log_significant.loc[cell_type].values
    
    # Create histogram
    counts, bins = np.histogram(data, bins=20, range=(0, 4))
    
    # Normalize counts to have similar heights
    if counts.max() > 0:
        counts = counts / counts.max() * 0.4
    
    # Plot histogram with vertical offset
    plt.bar(bins[:-1], counts, width=bins[1]-bins[0], bottom=i*offset, 
            alpha=0.7, label=cell_type if i == 0 else "", color=CELL_TYPE_COLORS[cell_type])
    
    # Add cell type label
    plt.text(-0.2, i*offset + 0.2, cell_type, ha='right', va='center', fontsize=6)
    
    # Add a horizontal line at the bottom of each histogram instead of a box
    plt.plot([0, 4], [i*offset, i*offset], color='black', linestyle='-', linewidth=0.5)

# Set axis labels and title
plt.xlabel(r'log$_{10}$(DEG # by per stim. + 1)')
plt.yticks([])
plt.xlim(0, 4)
plt.ylim(0, len(ALL_CELL_TYPES)*offset)

plt.tight_layout()
# # Save the figure
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_DEG_distributions.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
# Create a figure for histograms of log10(significant genes+1) per cell type
plt.figure(figsize=cm_to_inches((4, 3.2)))
ax = plt.gca()

# Calculate log10 of significant results (adding 1 to avoid log(0))
log_counts_cells = np.log10(donor_independent_counts + 1).groupby(level=0).mean()
log_significant = np.log10(significant_results + 1).mean(axis=1)

log_counts_cells, log_significant = get_common_index(log_counts_cells, log_significant)
for cell_type in ALL_CELL_TYPES:
    plt.scatter(log_counts_cells[cell_type], log_significant[cell_type], color=CELL_TYPE_COLORS[cell_type], s=15, edgecolors="none", alpha=0.75)

# Set axis labels and title
plt.xlabel(r'mean(log$_{10}$(cell type # by per stim.))')
plt.ylabel(r'mean(log$_{10}$(DEG #+ 1))')
plt.xlim(2, 5)
plt.ylim(0, 3)

plt.tight_layout()
# # Save the figure
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_deg_vs_count.{format}"), bbox_inches="tight", dpi=300)

### Corr eval for strong response magnitudes

In [ ]:
if alt_flag:
    means_by_cell_type = {}
    stds_by_cell_type = {}
    for base_cell_type in ALL_CELL_TYPES:
        corrwith_vals = {}
        for cell_type in ALL_CELL_TYPES:
            cytokines_strong_base = strong_impact_by_cell_type.index[strong_impact_by_cell_type[base_cell_type]]
            cytokines_strong_cell = strong_impact_by_cell_type.index[strong_impact_by_cell_type[cell_type]]
            common_cytokines = cytokines_strong_base.intersection(cytokines_strong_cell)

            if len(common_cytokines) < 3:
                continue

            # subset the two DataFrames
            df_base = edger_df_log2FoldChange[base_cell_type][common_cytokines]
            df_cell = edger_df_log2FoldChange[cell_type][common_cytokines]

            # align them and compute correlations columnwise
            corr_series = df_base.corrwith(df_cell, method='pearson')
            corrwith_vals[cell_type] = corr_series
        corrwith_vals = pd.DataFrame(corrwith_vals)
        means_by_cell_type[base_cell_type] = corrwith_vals.mean()
        stds_by_cell_type[base_cell_type] = corrwith_vals.std()
    means_by_cell_type = pd.DataFrame(means_by_cell_type)   
    means_by_cell_type = means_by_cell_type.loc[ALL_CELL_TYPES, ALL_CELL_TYPES]    

In [ ]:
if alt_flag:
    plt.figure(figsize=cm_to_inches((12, 12)))
    sns.heatmap(means_by_cell_type, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, annot_kws={"size":5.5},
                cbar_kws={"label": "mean Pearson correlation of log2FC\nin cell types with strong impact", "shrink": 0.5},
                square=True)
    plt.tick_params(axis='both', length=0)
    plt.savefig(os.path.join(plot_folder, f"2_correlation_of_strong_impact_log2FC_mean.{format}"), bbox_inches="tight", dpi=300)
    
if alt_flag:
    plt.figure(figsize=cm_to_inches((12, 12)))
    sns.heatmap(all_normalized_distances_mean.corr(), annot=True, annot_kws={"size":5.5},
                fmt=".2f", cmap="viridis", vmin=0.5, vmax=1,
                cbar_kws={"label": "mean Pearson correlation of response magnitudes", "shrink": 0.5},
                square=True)
    plt.tick_params(axis='both', length=0)
    plt.savefig(os.path.join(plot_folder, f"2_correlation_of_response_magnitude.{format}"), bbox_inches="tight", dpi=300)

#### Create a long df for scatter plots

In [ ]:
# assert (significant_results.sum(axis=1).sort_values() == edger_results_filtered[
#     edger_results_filtered['cell_type'].isin(ALL_CELL_TYPES)].groupby("cell_type").size().sort_values()).all()

# edger_results_filtered[
#     edger_results_filtered['cell_type'].isin(ALL_CELL_TYPES)].groupby("cell_type").size()

In [ ]:
data_long = significant_results.reset_index().rename(columns={'index': 'cell_type'})
data_long = data_long.melt(id_vars=['cell_type'], var_name='cytokine', value_name='DEGs')

# Convert Cytokine column to a categorical type with specified order
data_long['cytokine'] = pd.Categorical(data_long['cytokine'], categories=CYTOKINE_ORDER, ordered=True)

# Sort data_long by the categorical order to ensure plotting follows this order
data_long = data_long.sort_values(by='cytokine', ascending=False)

In [ ]:
bin_sizes = [10, 33, 100, 333, 1000, 3333, 10000, 10E6]
bin_text = ["<10", "<33", "<100", "<333", "<1000", "<3333", "<10000", ">10000"]

def bin_DEG_numbers(x):
    if x <= bin_sizes[0]:
        return 1
    if x <= bin_sizes[1]:
        return 2
    elif x <= bin_sizes[2]:
        return 3
    elif x <= bin_sizes[3]:
        return 4
    elif x <= bin_sizes[4]:
        return 5
    elif x <= bin_sizes[5]:
        return 6
    elif x <= bin_sizes[6]:
        return 7
    else:
        return 8
    
data_long["binned_DEGs"] = data_long["DEGs"].apply(bin_DEG_numbers)    

In [ ]:
all_normalized_distances_long = all_normalized_distances_mean.reset_index().rename(columns={'index': 'cytokine'})
all_normalized_distances_long = all_normalized_distances_long.melt(id_vars=['cytokine'], var_name='cell_type', value_name='Distance')

# clip at 0 and 1 (should basically already be the case, just make sure that minor numerical issues don't cause problems)
all_normalized_distances_long["Distance"] = all_normalized_distances_long["Distance"].clip(lower=0, upper=1)

# merge with the DEGs
data_long = data_long.merge(all_normalized_distances_long, on=["cytokine", "cell_type"])

#### Plotting of absolute DEG numbers

In [ ]:
fig, ax = plt.subplots(figsize=cm_to_inches((5, 20)))

# Scatter plot with square markers; `s` controls the marker size, which we scale here by 'DEGs' values
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['cell_type'] == cell_type]
    ax.scatter(
        subset['cell_type'], subset['cytokine'],
        s=subset['binned_DEGs'] * 2,  # Scale sizes as needed
        marker='s', c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        label=cell_type,
        alpha=subset['Distance']  # Scale alpha by 'Distance' values
    )

x_offset_level1 = -6  # Horizontal placement offset for family labels
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()
plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

# x_offset_level2 = -5  # Horizontal placement offset for family labels
# vals_per_label_level2 = cytokine_info["level2_family"].value_counts()
# plot_overview_labels_along_y_axis(fig, ax, level2_family_labels, vals_per_label_level2, x_offset_level2,
#                                   len(subset['cytokine']),
#                                   fontsize=6, fontweight="normal", color="gray", linewidth=0.5)


# Create legend for DEG numbers
legend_elements = []
for bin_size, bin_label in zip(bin_sizes, bin_text):
    legend_elements.append(
        plt.scatter([], [], 
                   s=bin_DEG_numbers(bin_size) * 2,  # Base size matching the main plot
                   marker='s',
                   color='gray',
                   edgecolor='k',
                   label=f'{bin_label} DEGs')
    )

# Add the legend
leg = plt.legend(handles=legend_elements,
                bbox_to_anchor=(1.05, 1),
                loc='upper left',
                borderaxespad=0,
                fontsize=6)

# Adjust layout to make room for legend
plt.subplots_adjust(right=0.85)

# rotate x-axis labels
plt.xticks(rotation=90)
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)
# set label font size to 6
plt.yticks(fontsize=5.5)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_DEG_numbers_by_cell_type.{format}"), 
                bbox_inches="tight", 
                dpi=300)

#### Plotting of relative DEG numbers

In [ ]:
data_long["max"] = data_long.groupby("cell_type")["DEGs"].transform("max")

# clip at at most 1% so that the dot size doesn't become too small
data_long["DEGs_clip"] = data_long.apply(lambda x: min(max(x["DEGs"], x["max"]/101.0), x["max"]), axis=1)
data_long["DEG_rel"] = data_long["DEGs_clip"] / data_long["max"]

# show the response magnitude as quintiles
data_long["distance_quintile"] = (pd.qcut(data_long["Distance"], q=5, labels=False) + 1) / 5
labels = [
    "<0.01",
    "0.01-0.02",
    "0.02-0.05",
    "0.05-0.10",
    "0.10-0.25",
    "0.25-0.50",
    "0.50-1.00"
]

edges = [-np.inf]

for lab in labels:
    nums = re.findall(r"[\d.]+", lab)

    if lab.startswith("<"):
        upper = float(nums[0])
        edges.append(upper)
    else:
        low, high = nums[:2]
        edges.append(float(high))

bins = np.array(edges)

data_long["DEG_rel_binned"] = pd.cut(
    data_long["DEG_rel"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

max_size = 25
total_labels = len(labels)
size_mapping = {lab: ((i + 1) / total_labels)**2 * max_size for i, lab in enumerate(labels)}
data_long["size"] = data_long["DEG_rel_binned"].map(size_mapping)

In [ ]:
# Plotting
fig, ax = plt.subplots(figsize=cm_to_inches((3.8, 20)))

# Scatter plot with square markers; `s` controls the marker size, which we scale here by 'DEGs' values
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['cell_type'] == cell_type]
    ax.scatter(
        subset['cell_type'], subset['cytokine'],
        s=subset["size"],
        marker='H', c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        alpha=subset['distance_quintile'],
        linewidth=0.5
    )

size_marker_legend_elements = []
size_marker_legend_elements.append(
        plt.scatter([], [], 
                   s=0,
                   marker='H',
                   color='gray',
                   edgecolor='k',
                   label=f'DEG fraction')
    )
for lab in labels:
    size_marker_legend_elements.append(
        plt.scatter([], [], 
                   s=size_mapping[lab],
                   marker='H',
                   color='gray',
                   edgecolor='k',
                   label=f'{lab}')
    )

x_offset_level1 = -6  # Horizontal placement offset for family labels
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()

plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

fig.canvas.draw()
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)

#plt.title(r"log$_{10}$(relative DEG number)",fontsize=7)
plt.xticks(rotation=90, fontsize=5.5)
# set label font size to 6
plt.yticks(fontsize=5.5)
plt.tick_params(length=0, axis='both')
plt.legend(
    handles=size_marker_legend_elements,
    bbox_to_anchor=(1.05, 1),
    loc='upper left',
    borderaxespad=0,
    fontsize=5.5,
    ncol=2
)

# plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_relative_DEG_numbers_by_cell_type.{format}"), bbox_inches="tight", dpi=300)

#### Calculate and plot supplementary barplots (tsi, total DEGs, strong impact)

In [ ]:
def calc_tsi(df, cytokine, column):
    """Calculates the tsi, subtracting half the response magnitude threshold value for each cell type."""
    cytokine_df = df[df["cytokine"] == cytokine]
    threshold_values = [magnitude_threshold_by_cell_type[c]/2.0 for c in cytokine_df['cell_type']]
    values = cytokine_df[column].values - threshold_values
    values = [max(v, 0.05) for v in values]
    max_value = np.max(values)
    if any(strong_impact_by_cell_type.loc[cytokine].values):
        tsi = sum((1-values/max_value)/(len(values)-1))
    else:
        tsi = 0
    return tsi

tsi_by_cytokine = {}
for cytokine in CYTOKINE_ORDER:
    tsi =  calc_tsi(data_long[data_long["cell_type"].isin(ALL_CELL_TYPES)], cytokine, "Distance")
    tsi_by_cytokine[cytokine] = tsi
    
tsi_by_cytokine = pd.Series(tsi_by_cytokine)    

In [ ]:
# need to invert the order due to the way that the heatmap is plotted
tsi_by_cytokine = tsi_by_cytokine[CYTOKINE_ORDER[::-1]]
tsi_by_cytokine_0 = tsi_by_cytokine[tsi_by_cytokine == 0]
zero_positions = [tsi_by_cytokine.index.get_loc(idx) for idx in tsi_by_cytokine_0.index]

# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 20)))
plt.barh(tsi_by_cytokine.index, tsi_by_cytokine.values, color="black")
plt.scatter([0.2] * len(zero_positions), zero_positions, color="gray", alpha=1, marker='x', s=15)
plt.ylim(-1, len(CYTOKINE_ORDER))
# get the current figure and draw it (to get apply_greek_labels_heatmap to work)
fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.xticks(np.arange(0, 1.1, 1), [0, 1], rotation=0, fontsize=6)
plt.title("TSI", fontsize=7)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_tsi_by_cytokine.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
tsi_by_cytokine_filter = tsi_by_cytokine[tsi_by_cytokine > 0].sort_values(ascending=False)
tsi_by_cytokine_filter.index = [introduce_greek_letters(i) for i in tsi_by_cytokine_filter.index]
plt.figure(figsize=cm_to_inches((14, 3)))
plt.bar(tsi_by_cytokine_filter.index, tsi_by_cytokine_filter.values, color="black")
plt.xticks(rotation=90)
plt.ylabel("TSI")
plt.xlim(-1, len(tsi_by_cytokine_filter))
plt.ylim(0, 1)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_tsi_by_cytokine_flat.{format}"), dpi=300)

In [ ]:
strong_impact_by_cytokine = strong_impact_by_cell_type.sum(axis=1).iloc[::-1]

# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 20)))
plt.barh(strong_impact_by_cytokine.index, strong_impact_by_cytokine.values, color="black")
plt.ylim(-1, len(CYTOKINE_ORDER))
# get the current figure and draw it (to get apply_greek_labels_heatmap to work)
fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.xticks([0, len(ALL_CELL_TYPES)], [0, len(ALL_CELL_TYPES)], rotation=0, fontsize=6)
plt.xlim(0, len(ALL_CELL_TYPES)+1)
plt.title("strong\nimpact", fontsize=7)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_strong_impact_by_cytokine.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
# get the max DEGs per cell type
max_DEGs_by_cell_type = data_long.groupby("cell_type")["DEGs"].max()
max_DEGs_by_cell_type = max_DEGs_by_cell_type[ALL_CELL_TYPES]
# plot the max DEGs per cell type
#plt.figure(figsize=cm_to_inches((5.36, 1.3)))
plt.figure(figsize=cm_to_inches((4.75, 1.3)))
plt.bar(max_DEGs_by_cell_type.index, max_DEGs_by_cell_type.values, color="black", width=0.7)
plt.xticks([])
plt.ylabel("max.\nDEGs")
# cut whitespace at left and right
plt.xlim(-0.75, len(ALL_CELL_TYPES)-0.25)
#plt.yticks([0, 7000], ["0", r"$7\cdot 10^{3}$"])
plt.yticks([0, 7000], ["0", r"7000"])
plt.ylim(0, 7000)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_max_DEGs_by_cell_type.{format}"), bbox_inches="tight", dpi=300)


## 2.3 - Additional response magnitude plots

In [ ]:
strong_impact_by_cell_type_long = strong_impact_by_cell_type.melt(var_name="cell_type", value_name="strong_impact", ignore_index=False).reset_index().rename({"index": "cytokine"}, axis=1)
data_long = pd.merge(data_long, strong_impact_by_cell_type_long, on=["cell_type", "cytokine"])

In [ ]:
# Plotting
fig, ax = plt.subplots(figsize=cm_to_inches((4.7, 20)))

# Scatter plot with square markers; `s` controls the marker size, which we scale here by 'DEGs' values
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    subset = data_long[data_long['cell_type'] == cell_type]
    ax.scatter(
        subset['cell_type'], subset['cytokine'],
        s = 10 * subset['strong_impact'].astype(int),
        marker='s', c="black", edgecolor="black", #c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        linewidth=0,
    )

x_offset_level1 = -6  # Horizontal placement offset for family labels
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()
plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

fig.canvas.draw()
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)

# plt.title(r"log$_{10}$(relative DEG number)",fontsize=7)
plt.xticks(rotation=90, fontsize=6)
# set label font size to 6
plt.yticks(fontsize=5.5)

#plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.3_strong_impact_vs_expresso.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
# Plotting
fig, ax = plt.subplots(figsize=cm_to_inches((4.7, 20)))

# Scatter plot with square markers; `s` controls the marker size, which we scale here by 'DEGs' values
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['cell_type'] == cell_type]
    ax.scatter(
        subset['cell_type'], subset['cytokine'],
        s = 10 * subset['Distance'],
        marker='s', c="black", edgecolor="black", #c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        linewidth=0,
    )

x_offset_level1 = -6  # Horizontal placement offset for family labels
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()

plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

fig.canvas.draw()
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)

# plt.title(r"log$_{10}$(relative DEG number)",fontsize=7)
plt.xticks(rotation=90, fontsize=6)
# set label font size to 6
plt.yticks(fontsize=5.5)

#plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.3_response_magnitude_by_size.{format}"), bbox_inches="tight", dpi=300)

## 2.4 - Donor variability analysis

In [ ]:
def get_upper_triangle(df, diagonal=False):
    """
    Extract the upper triangle of a DataFrame.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The input DataFrame (typically a correlation or distance matrix)
    diagonal : bool, default=False
        Whether to include the diagonal elements
        
    Returns:
    --------
    pandas.Series
        A series containing the upper triangle values with MultiIndex
    """
    # Get the upper triangle mask
    if diagonal:
        mask = np.triu(np.ones_like(df, dtype=bool))
    else:
        mask = np.triu(np.ones_like(df, dtype=bool), k=1)
    
    # Apply mask and stack to get a series
    upper_triangle = df.where(mask)
    
    return upper_triangle

def delete_diagonal(df):
    """
    Remove the diagonal elements from a DataFrame by setting them to NaN.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The input DataFrame (typically a correlation or distance matrix)
        
    Returns:
    --------
    pandas.DataFrame
        A copy of the DataFrame with diagonal elements set to NaN
    """
    df_copy = df.copy()
    np.fill_diagonal(df_copy.values, np.nan)
    return df_copy

In [ ]:
current_plot_folder = os.path.join(plot_folder, "2.4_donor_variability")
os.makedirs(current_plot_folder, exist_ok=True)

In [ ]:
# shorten donor names and do some bookkeeping
cols = list(sum_gene_count_df_norm_by_donor.columns.levels[2])
new_cols = [c.replace("Donor", "D") if isinstance(c, str) else c for c in cols]
sum_gene_count_df_norm_by_donor.columns = sum_gene_count_df_norm_by_donor.columns.set_levels(new_cols, level=2)

cols = list(sum_gene_count_df_raw_by_donor.columns.levels[2])
new_cols = [c.replace("Donor", "D") if isinstance(c, str) else c for c in cols]
sum_gene_count_df_raw_by_donor.columns = sum_gene_count_df_raw_by_donor.columns.set_levels(new_cols, level=2)

idx = all_counts_by_condition.index
new_idx = idx.set_levels([idx.levels[0], idx.levels[1], [c.replace("Donor", "D") if isinstance(c, str) else c for c in idx.levels[2]]], level=[0, 1, 2])
all_counts_by_condition.index = new_idx

In [ ]:
# filter per donor to get only expressed genes
mask = sum_gene_count_df_raw_by_donor < 10
sum_gene_count_df_norm_by_donor_filter = sum_gene_count_df_norm_by_donor.copy()
sum_gene_count_df_norm_by_donor_filter[mask] = np.nan

sum_gene_count_df_norm_by_donor_pbs = sum_gene_count_df_norm_by_donor_filter.loc[:, (slice(None), "PBS", slice(None))].droplevel(1, axis=1)
sum_gene_count_df_norm_by_donor_pbs = sum_gene_count_df_norm_by_donor_pbs.dropna(axis=1, how='all')

### 2.4.0 - Basic metadata

In [ ]:
DONOR_NAMES = [f"Donor{i+1}" for i in range(12)]
donor_metadata = pd.read_excel("../annotations/donor_metadata.xlsx")
donor_metadata["Donor number"] = donor_metadata["Donor number"].str.replace(" ", "")
donor_metadata.index = donor_metadata["Donor number"]
donor_metadata["age"] = 2024 - donor_metadata["Birth year"]
donor_metadata = donor_metadata.sort_values(by="Sex")
DONOR_NAMES = donor_metadata.index.tolist()
DONOR_NAMES = [d.replace("Donor", "D") for d in DONOR_NAMES]

In [ ]:
donor_metadata["Donor number"] = donor_metadata["Donor number"].str.replace("Donor", "D")
donor_metadata['Weight (kg)'] = donor_metadata['Weight (lbs)']*0.453592
donor_metadata['Height (m)'] = donor_metadata['Height (in)']*2.54/100
donor_metadata["BMI"] = donor_metadata["Weight (kg)"] / donor_metadata['Height (m)']**2

In [ ]:
donor_metadata[donor_metadata["Sex"] == "Female"]["age"].sort_values()

In [ ]:
# --- parameters ---
stack_gap = 0.15   # vertical offset step for stacked donors
tolerance = 10    # how close (in years) counts as "same spot"

plt.figure(figsize=cm_to_inches((3, 4)))

for i, sex in enumerate(["Male", "Female"]):
    subset = donor_metadata[donor_metadata["Sex"] == sex].sort_values("age")
    ages = subset["age"].values
    donors = subset["Donor number"].values

    used = np.zeros(len(ages), dtype=bool)

    for idx in range(len(ages)):
        if used[idx]:
            continue

        # cluster: all donors within tolerance of this age
        mask = np.abs(ages - ages[idx]) < tolerance
        cluster_indices = np.where(mask & ~used)[0]
        used[cluster_indices] = True

        n = len(cluster_indices)
        if n == 1:
            offsets = [0]
        else:
            # symmetric offsets around 0, e.g. for n=4 → [-0.525, -0.175, 0.175, 0.525]
            offsets = np.linspace(-(n-1)/2, (n-1)/2, n) * stack_gap

        for j, ci in enumerate(cluster_indices):
            plt.text(
                ages[ci], i + offsets[j], donors[ci],
                ha="center", va="center", fontsize=6, alpha=1
            )

# --- formatting ---
plt.ylabel("")
plt.xlabel("")
plt.gca().xaxis.set_label_position('top')
plt.gca().xaxis.tick_top()
plt.xlabel("age (years)")

plt.yticks(
    range(len(donor_metadata["Sex"].unique())),
    ["Male", "Female"]
)
plt.xlim(20, 80)
plt.ylim(-0.5, len(donor_metadata["Sex"].unique()) - 0.5)

plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "2.4_donor_age_dist.svg"))
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((3, 4)))
stack_gap = 0.15
tolerance = 5

for i, sex in enumerate(["Male", "Female"]):
    subset = donor_metadata[donor_metadata["Sex"] == sex].sort_values("age")
    ages = subset["BMI"].values
    donors = subset["Donor number"].values

    used = np.zeros(len(ages), dtype=bool)

    for idx in range(len(ages)):
        if used[idx]:
            continue

        mask = np.abs(ages - ages[idx]) < tolerance
        cluster_indices = np.where(mask & ~used)[0]
        used[cluster_indices] = True

        n = len(cluster_indices)
        if n == 1:
            offsets = [0]
        else:
            offsets = np.linspace(-(n-1)/2, (n-1)/2, n) * stack_gap

        for j, ci in enumerate(cluster_indices):
            plt.text(
                ages[ci], i + offsets[j], donors[ci],
                ha="center", va="center", fontsize=6, alpha=1
            )

# --- formatting ---
plt.ylabel("")
plt.xlabel("")
plt.gca().xaxis.set_label_position('top')
plt.gca().xaxis.tick_top()
plt.xlabel("body mass index")

plt.yticks(
    range(len(donor_metadata["Sex"].unique())),
    ["Male", "Female"]
)
plt.xlim(15, 45)
plt.ylim(-0.5, len(donor_metadata["Sex"].unique()) - 0.5)

plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "2.4_donor_weight_dist.svg"))
plt.show()

In [ ]:
donor_groups = {
    "D1": "older\nfemale",
    "D3": "older\nfemale", 
    "D4": "older\nfemale",
    "D7": "younger\nfemale",
    "D8": "younger\nfemale",
    "D12": "younger\nfemale", 
    "D2": "younger\nmale",
    "D5": "younger\nmale",
    "D6": "younger\nmale", 
    "D9": "younger\nmale",
    "D10": "younger\nmale",
    "D11": "younger\nmale"
}

palette = plt.get_cmap("tab20").colors  # 20 distinct colors
DONOR_COLORS = {donor: palette[i] for i, donor in enumerate(DONOR_NAMES)}

In [ ]:
fig = plt.figure(figsize=cm_to_inches((3.7,4.3)))
plt.barh(
    donor_metadata["Donor number"].iloc[::-1],
    donor_metadata['age'].iloc[::-1],
    edgecolor="black",
    color="skyblue",
    height=0.7)
plt.xlim(25, 80)
plt.xlabel("age (years)")
plot_overview_labels_along_y_axis(
    fig,
    plt.gca(),
    labels=['older\nfemale', 'younger\nfemale', 'younger\nmale'],
    vals_per_label=pd.Series(donor_groups).value_counts(),
    x_offset=-1.2,
    no_y_vals=len(DONOR_NAMES),
    fontsize=6.5
)
plt.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, 'donor_ages_bar.svg'))

fig = plt.figure(figsize=cm_to_inches((3.7,4.3)))
plt.barh(
    donor_metadata["Donor number"].iloc[::-1],
    donor_metadata['BMI'].iloc[::-1],
    edgecolor="black",
    color="skyblue",
    height=0.7)
plt.xlim(20, 40)
plt.xlabel("BMI")
plot_overview_labels_along_y_axis(
    fig,
    plt.gca(),
    labels=['older\nfemale', 'younger\nfemale', 'younger\nmale'],
    vals_per_label=pd.Series(donor_groups).value_counts(),
    x_offset=13,
    no_y_vals=len(DONOR_NAMES),
    fontsize=6.5    
)
plt.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, 'donor_bmi_bar.svg'))


In [ ]:
# Replace "Donor" with "D" in index level 2
all_counts_by_condition.index = all_counts_by_condition.index.set_levels(
    [level.str.replace("Donor", "D") if i == 2 else level 
     for i, level in enumerate(all_counts_by_condition.index.levels)],
    level=[0, 1, 2]
)

all_counts_by_condition_summarized = all_counts_by_condition.groupby(level=[0, 2]).sum()
all_counts_by_condition_summarized = all_counts_by_condition_summarized.swaplevel(0, 1)

# Prepare data for stacked barplot
plot_data = all_counts_by_condition_summarized.unstack(level=0, fill_value=0)
plot_data = plot_data.reindex(DONOR_NAMES, axis=1)  # Order donors
plot_data = plot_data.reindex(ALL_CELL_TYPES, axis=0)  # Order cell types
plot_data = plot_data / plot_data.sum()

# Create stacked barplot of cell type counts by donor
fig, ax = plt.subplots(figsize=cm_to_inches((8, 4.3)))
# Create stacked barplot
bottom = np.zeros(len(DONOR_NAMES))
for cell_type in ALL_CELL_TYPES:
    if cell_type in plot_data.index:
        values = plot_data.loc[cell_type, DONOR_NAMES[::-1]].values
        ax.barh(DONOR_NAMES[::-1], values, left=bottom, 
               color=CELL_TYPE_COLORS[cell_type], label=cell_type, height=0.8)
        bottom += values

plot_overview_labels_along_y_axis(
    fig,
    plt.gca(),
    labels=['older\nfemale', 'younger\nfemale', 'younger\nmale'],
    vals_per_label=pd.Series(donor_groups).value_counts(),
    x_offset=-0.4,
    no_y_vals=len(DONOR_NAMES),
    fontsize=6.5
)
plt.tick_params(axis='y', length=0)
ax.set_xlabel('fraction of cells')
plt.tight_layout()
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=6, ncol=1)
plt.savefig(os.path.join(current_plot_folder, "2.4_cell_type_proportions.svg"))

In [ ]:
cell_type_order = [
    "B cell",
    "CD4 T cell",
    "CD8 T cell",
    "Treg",
    "MAIT",
    "NKT",
    "NK CD56hi",
    "NK CD56low",
    "pDC",
    "cDC",
    "CD14 Mono",
    "CD16 Mono",
    "HSPC",
    "ILC",
    "granulocyte",
    "plasmablast"
]


# If all_counts_by_condition is a Series with MultiIndex (cell_type, ...):
g = all_counts_by_condition.groupby(level=0)

vals = g.median()
mad = g.apply(lambda s: np.mean(np.abs(s - s.median())))

vals = vals.reindex(cell_type_order)
mad  = mad.reindex(cell_type_order)

vals = vals.iloc[::-1]
mad  = mad.iloc[::-1]

plt.figure(figsize=cm_to_inches((4, 6)))
plt.barh(vals.index, vals.values, xerr=mad.values, color="skyblue", edgecolor="black", height=0.7, ecolor="black", capsize=0)

plt.xscale("log")
plt.axvline(x=10, color="red", linestyle="--", linewidth=1, label='minimum median cell count')
plt.xlabel("median cell count per condition\n(cell type-cytokine-donor)") 
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "2.4_median_cell_counts.svg"))

In [ ]:
# mean counts per condition across cell types
all_counts_by_condition.groupby(level=[1, 2]).sum().mean()

In [ ]:
all_counts_by_condition_norm = all_counts_by_condition.unstack().fillna(0)
all_counts_by_condition_norm = 100 * all_counts_by_condition_norm / all_counts_by_condition_norm.sum()
all_counts_by_condition_norm = all_counts_by_condition_norm.stack()

In [ ]:
zero_counts = {}
means_by_cell_type = {}
stds_by_cell_type = {}
for cell_type in cell_type_order:
    df = all_counts_by_condition[cell_type][CYTOKINE_ORDER]
    mean = df.groupby(level=1).mean()
    std = df.groupby(level=1).std()
    zero_counts[cell_type] = (df == 0).sum()
    means_by_cell_type[cell_type] = mean
    stds_by_cell_type[cell_type] = std
means_by_cell_type = pd.DataFrame(means_by_cell_type).loc[DONOR_NAMES]
stds_by_cell_type = pd.DataFrame(stds_by_cell_type).loc[DONOR_NAMES]

In [ ]:
from matplotlib.ticker import ScalarFormatter

fig, axes = plt.subplots(8, 2, figsize=cm_to_inches((9, 12)), sharex=True)
axes = axes.flatten()
for i, col in enumerate(cell_type_order):
    means = means_by_cell_type[col]
    stds = stds_by_cell_type[col]
    ax = axes[i]
    ax.bar(means.index, means.values, yerr=stds.values, capsize=2)
    ax.set_title(col, fontsize=6)
    if i % 4 == 0:
        ax.set_ylabel("mean cell count")
    if i < 12:
        ax.tick_params(axis='x', length=0)
    ax.yaxis.set_major_formatter(ScalarFormatter(useMathText=True))
    ax.yaxis.get_major_formatter().set_scientific(True)
    ax.yaxis.get_major_formatter().set_powerlimits((0, 0))

plt.subplots_adjust(hspace=0.75)
plt.savefig(os.path.join(current_plot_folder, "2.4_mean_cell_counts_by_cytokine.svg"), bbox_inches="tight")

In [ ]:
sum_across_donors = all_counts_by_condition.groupby(level=1).sum()
sum_across_donors = 6 * sum_across_donors / sum_across_donors["PBS"]
sum_across_donors = sum_across_donors.sort_values().drop("PBS")

In [ ]:
# Distribution of cell counts across wells isn't uniform; plot heatmap of counts per well
metadata = pd.read_csv('../../data/10M_Parse_PBMC_cytokines_cell_metadata.csv')
counts_by_well = metadata.groupby(['bc1_well']).size()
info = metadata['cytokine'] + "_" + metadata['bc1_well']

info = info.unique()
info = pd.Series(info).to_frame(name='condition')
info['well'] = info['condition'].str.split('_').str[1]
info['cytokine'] = info['condition'].str.split('_').str[0]

info = info.sort_values(
    by='well',
    key=lambda s: s.map(lambda x: (re.match(r'([A-Z]+)(\d+)', x).group(1),
                                   int(re.match(r'([A-Z]+)(\d+)', x).group(2))))
)
info['count'] = info['well'].map(counts_by_well)
info['column'] = info['well'].str.extract(r'([A-Z]+)')[0]
info['row'] = info['well'].str.extract(r'(\d+)')[0]
info_pivot = info.pivot(index='row', columns='column', values='count')

columns = ["A", "B", "C", "D", "E", "F", "G", "H"]
rows = [str(i) for i in range(1, 13)]

sns.heatmap(
    info_pivot.reindex(index=rows, columns=columns)/info_pivot.stack().mean(),
    cmap='viridis',
    cbar_kws={"label": "cell count"},
    annot=True,
    fmt='.2f'
)
plt.show()

In [ ]:
info['dist_from_center'] = info.apply(lambda x: abs((int(x['row']) - 6.5)), axis=1)
info['relative_count'] = info['count'] / info['count'].mean()

In [ ]:
plt.figure(figsize=cm_to_inches((4, 3)))
plt.scatter(
    info['dist_from_center'],
    info['relative_count'],
    s = 5,
)
a,b = np.polyfit(info['dist_from_center'], info['relative_count'], 1)
x = np.linspace(0, 6, 100)
plt.plot(x, a*x + b, color='black')
plt.xlabel("distance from center row")
plt.ylabel("relative count")
plt.show()

info['adjustment_factor'] = 1 / (a * info['dist_from_center'] + b)
info['adjusted_count'] = info['count'] * info['adjustment_factor']
info['relative_adjusted_count'] = info['adjusted_count'] / info[info['cytokine'] == "PBS"]['adjusted_count'].mean()
plt.figure(figsize=cm_to_inches((4, 3)))
plt.scatter(
    info['dist_from_center'],
    info['relative_adjusted_count'],
    s = 5,
)
plt.show()
count_adjustment_by_cytokine = info.groupby('cytokine')['adjustment_factor'].mean()

In [ ]:
plt.figure(figsize=cm_to_inches((4, 3)))
plt.hist(np.log2(info['relative_adjusted_count'].values), bins=20)
plt.ylabel("count")
plt.xlabel("log2(relative adjusted count)")
plt.xlim(-1, 1)
plt.show()

In [ ]:
all_counts_by_condition.index.set_names(["cell_type", "cytokine", "donor"], inplace=True)

In [ ]:
from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

reference = "PBS"
correction = "fdr_bh"  # multiple testing correction method
ref_wells = 6  # six wells per donor for PBS
cyto_wells = 1 # one well per donor for cytokine
pseudocounts = 5

results = []
for cell_type, sub in all_counts_by_condition.groupby(level="cell_type"):
    if cell_type not in ALL_CELL_TYPES:
        continue
    print("Processing cell type:", cell_type)
    
    # pivot donors × cytokines
    df = sub.unstack("cytokine")
    ref = df[reference] * count_adjustment_by_cytokine[reference]

    for cytokine in df.columns:
        if cytokine == reference:
            continue
        
        x = ref
        y = df[cytokine] * count_adjustment_by_cytokine[cytokine]
        common = x.dropna().index.intersection(y.dropna().index)
        
        if len(common) < 2:
            print(f"Skipping {cell_type} - {cytokine} due to insufficient donors")
            continue
        
        x = (x.loc[common] / ref_wells) + pseudocounts # there are six wells per donor for PBS
        y = (y.loc[common] / cyto_wells) + pseudocounts # there is only one well per donor for cytokine

        # compute log2 fold-change (mean per donor)
        log2fc = np.log2(y/x).mean()
        try:
            stat, pval = wilcoxon(y, x, zero_method="wilcox", alternative="two-sided")
        except ValueError:
            pval = np.nan

        results.append({
            "cell_type": cell_type,
            "cytokine": cytokine,
            "n_donors": len(common),
            "pval": pval,
            "log2FC": log2fc
        })

results_df = pd.DataFrame(results)

# multiple testing correction per cell type
results_df["pval_adj"] = np.nan
for cell_type, idx in results_df.groupby("cell_type").groups.items():
    mask = results_df.index.isin(idx)
    pvals = results_df.loc[mask, "pval"].values
    _, padj, _, _ = multipletests(pvals, method=correction)
    results_df.loc[mask, "pval_adj"] = padj

results_df = results_df.sort_values(["cell_type", "pval_adj"])

In [ ]:
fig, axes = plt.subplots(2, 6, figsize=cm_to_inches((22, 6)), sharey=True, sharex=True)
axes = axes.flatten()

for i, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes[i]
    subset = results_df[results_df["cell_type"] == cell_type].copy()
    subset['logfc_abs'] = subset['log2FC'].abs()
    relevant_padj = subset["pval_adj"] < 0.05
    relevant_fc = (subset["log2FC"].abs() > 1)
    relevant = subset[relevant_padj | relevant_fc].index
    if len(relevant) < 3:
        top3 = subset.sort_values('logfc_abs', ascending=False).head(3)
        top3 = top3[top3['logfc_abs']>0.25].index
        relevant = relevant.union(top3)
        
    ax.scatter(subset["log2FC"], -np.log10(subset["pval_adj"]), color=CELL_TYPE_COLORS[cell_type], s=10, edgecolors="none")
    for index in relevant:
        row = subset.loc[index]
        ax.text(row["log2FC"], -np.log10(row["pval_adj"]), introduce_greek_letters(row["cytokine"]), fontsize=6, ha="center", va="bottom")
    
    ax.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.axvline(-1, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.axvline(1, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax.set_title(cell_type, fontsize=6)
    if i % 6 == 0:
        ax.set_ylabel("-log10(adj. pval)")
    if i >= 6:
        ax.set_xlabel("log2FC(cell count)")
    ax.set_xlim(-3, 3)
    ax.set_ylim(0, 2.5)
    ax.tick_params(axis='both', which='major', labelsize=6)
    
plt.subplots_adjust(hspace=0.4, wspace=0.1)
plt.savefig(os.path.join(current_plot_folder, "2.4_differential_abundance_volcano_plots.svg"), bbox_inches="tight")

In [ ]:
# SCCODA analysis - takes rather long, so uncomment to run only if needed
# from sccoda.util import comp_ana as mod
# from sccoda.util import cell_composition_data as dat
# from sccoda.util import data_visualization as viz
# import sccoda.datasets as scd

# all_counts_by_condition = all_counts_by_condition.reset_index()
# all_counts_by_condition.columns = ["cell_type", "cytokine", "donor", "count"]
# all_counts_by_condition = all_counts_by_condition.pivot(index=['cytokine', 'donor'], columns='cell_type', values='count').reset_index()
# relevant_cols = ['cytokine', 'donor'] + ALL_CELL_TYPES
# all_counts_by_condition = all_counts_by_condition[relevant_cols]
# data_all = dat.from_pandas(all_counts_by_condition, covariate_columns=["cytokine", "donor"])

# for reference_cell_type in ["CD14 Mono", "CD16 Mono"]:
#     model = mod.CompositionalAnalysis(data_all, formula="cytokine + donor", reference_cell_type=reference_cell_type)
#     sim_results = model.sample_hmc()
#     sim_results.save(os.path.join(output_tables_folder, f"sccoda_sim_results_{reference_cell_type.replace(' ', '').lower()}.pkl"))

In [ ]:
sim_results = {}
with open (os.path.join(output_tables_folder, "sccoda_sim_results_cd4.pkl"), "rb") as f:
    sim_results["CD4 T cell"] = pickle.load(f)
with open (os.path.join(output_tables_folder, "sccoda_sim_results_cd14mono.pkl"), "rb") as f:
    sim_results["CD14 Mono"] = pickle.load(f)

In [ ]:
for reference_cell_type, results in sim_results.items():

    donor_effects = results.effect_df.copy()
    donor_effects = donor_effects.reset_index()
    donor_effects = donor_effects[donor_effects["Covariate"].str.startswith("donor")].copy()
    donor_effects["donor"] = donor_effects["Covariate"].str.replace("donor[T.", "").str.replace("]", "")

    cytokine_effects = results.effect_df.copy()
    cytokine_effects = cytokine_effects.reset_index()
    cytokine_effects = cytokine_effects[cytokine_effects["Covariate"].str.startswith("cytokine")].copy()
    cytokine_effects["cytokine"] = cytokine_effects["Covariate"].str.replace("cytokine[T.", "").str.replace("]", "")
    cytokine_effects['capped_inclusion'] = cytokine_effects['Inclusion probability'].apply(lambda x: min(x, 0.9999))

    fig, axes = plt.subplots(2, 6, figsize=cm_to_inches((22, 6)), sharey=True, sharex=True)
    axes = axes.flatten()


    for i, cell_type in enumerate(ALL_CELL_TYPES):
        ax = axes[i]
        subset = cytokine_effects[cytokine_effects["Cell Type"] == cell_type].copy()
        subset = subset.reset_index(drop=True)
        relevant_padj = subset["Inclusion probability"] > 0.95
        relevant_fc = (subset["log2-fold change"].abs() > 0.25)
        relevant = subset[relevant_padj & relevant_fc].index
        if len(relevant) < 3:
            top3 = subset.sort_values('Inclusion probability', key=lambda x: x.abs(), ascending=False).head(3)
            top3 = top3[top3['log2-fold change'].abs()>0.1].index
            relevant = relevant.union(top3)
        
        ax.scatter(subset["log2-fold change"], -np.log10(1-subset["capped_inclusion"]),
                color=CELL_TYPE_COLORS[cell_type], s=10, edgecolors="none")
        
        for index in relevant:
            row = subset.loc[index]
            ax.text(
                row["log2-fold change"],
                -np.log10(1-row["capped_inclusion"]),
                introduce_greek_letters(row["cytokine"]), fontsize=6, ha="center", va="bottom")
        
        ax.axhline(-np.log10(0.05), color="black", linestyle="--", linewidth=1, alpha=0.5)
        ax.axvline(-0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
        ax.axvline(0.5, color="black", linestyle="--", linewidth=1, alpha=0.5)
        title = cell_type + "(ref)" if cell_type == reference_cell_type else cell_type
        ax.set_title(title, fontsize=6, fontdict={'fontweight': 'bold' if cell_type == reference_cell_type else 'normal'})
        if i % 6 == 0:
            ax.set_ylabel("-log10(1 - incl. prob.)")
        if i >= 6:
            ax.set_xlabel("log2FC(rel. cell fraction)")
        ax.set_xlim(-2, 2)
        ax.set_ylim(0, 5)
        ax.tick_params(axis='both', which='major', labelsize=6)
        
    plt.subplots_adjust(hspace=0.4, wspace=0.1)
    plt.savefig(os.path.join(current_plot_folder, f"2.4_sccoda_volcano_plots_{reference_cell_type.replace(' ', '').lower()}.svg"), bbox_inches="tight")

### 2.4.1 - Donor baseline characterization

In [ ]:
from pybiomart import Server

server = Server(host="http://www.ensembl.org")
dataset = server["ENSEMBL_MART_ENSEMBL"]["hsapiens_gene_ensembl"]

genes = dataset.query(
    attributes=["chromosome_name", "external_gene_name"],
    filters={"chromosome_name": ["X", "Y"]}
)

sex_gene_list = set(genes["Gene name"].dropna().unique().tolist())

In [ ]:
# compare PBS ratios to median
sum_gene_count_df_norm_pbs_median = sum_gene_count_df_norm_by_donor_pbs.T.groupby(level=0).median().T

donor_pbs_ratio_to_median = {}
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    median_df = sum_gene_count_df_norm_pbs_median[cell_type].copy()
    
    # buncha filtering
    curr_filter = inclusive_filters_by_cell_type[cell_type] # genes passing basic expression filter
    curr_filter = [gene for gene in curr_filter if not gene in sex_gene_list] # remove sex-specific genes
    curr_filter = [gene for gene in curr_filter if not "ENSG" in gene] # remove genes that might not be meaningful (and possibly sex-specific, since unannotated)
    curr_filter = [gene for gene in curr_filter if gene in protein_coding_genes] # filter to protein-coding genes for ease of interpretation
    
    for donor in DONOR_NAMES:
        df = sum_gene_count_df_norm_by_donor_pbs.loc[curr_filter, (cell_type, donor)]
        df_filter, median_df_filter = get_common_index(df, median_df)
        ratio = df_filter / median_df_filter
        donor_pbs_ratio_to_median[f"{cell_type}_{donor}"] = ratio

# Convert to DataFrame with two-level column index
donor_pbs_ratio_to_median_df = pd.DataFrame(donor_pbs_ratio_to_median)

# Split the column names to create MultiIndex
cell_type_donor_pairs = [col.split('_', 1) for col in donor_pbs_ratio_to_median_df.columns]
multi_index = pd.MultiIndex.from_tuples(cell_type_donor_pairs, names=['cell_type', 'donor'])
donor_pbs_ratio_to_median_df.columns = multi_index

In [ ]:
# plot some of the ratios
for cell_type in ["CD4 T cell", "CD14 Mono"]:
    df = np.log2(donor_pbs_ratio_to_median_df[cell_type][DONOR_NAMES])

    for donors in [("D1", "D3"), ("D3", "D4"), ("D3", "D8"), ("D3", "D10"), ("D5", "D12")]:
        donor1 = donors[0]
        donor2 = donors[1]
        series1, series2 = get_common_index(df[donor1], df[donor2])
        plt.figure(figsize=cm_to_inches((2.15, 1.85)))
        plt.scatter(series1, series2, alpha=1, s=2, rasterized=True, edgecolors="none", color=CELL_TYPE_COLORS[cell_type])

        r = np.corrcoef(series1, series2)[0,1]
        plt.text(-4, 3, f"r={r:.2f}", fontsize=6)
        plt.xlabel(f'baseline log2FC {donor1}')
        plt.ylabel(f'baseline log2FC {donor2}')
        min_x, max_x = series1.min(), series1.max()
        min_y, max_y = series2.min(), series2.max()
        plt.xlim(min_x - 1, max_x + 1)
        plt.ylim(min_y - 1, max_y + 1)
        plt.tick_params(axis='x', length=0)
        plt.tick_params(axis='y', length=0)
        plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=4))
        plt.gca().yaxis.set_major_locator(plt.MaxNLocator(integer=True, nbins=4))

        # Can figure out unexpected strong outliers here        
        # Label the outliers
        # for gene in outliers:
        #     plt.annotate(gene, (series1[gene], series2[gene]), 
        #                 xytext=(1, 1), textcoords='offset points', fontsize=6)
        
        plt.savefig(os.path.join(current_plot_folder, f"pbs_ratio_to_median_scatter_{cell_type}_{donor1}_{donor2}.svg"), dpi=400)

In [ ]:
# plot inflammation-related genes of interest
for cell_type in ["B cell", "CD4 T cell", "NK CD56low", "CD14 Mono"]:
    genes = ["IFIT1", "IFIT2", "IFIT3", "RSAD2", "OAS1", "MX1", "TNFSF10", "IL1RN"]
    vals = np.log2(donor_pbs_ratio_to_median_df[cell_type])

    donor_set_1 = ["D1", "D3", "D4", "D10"]
    donor_set_2 = [donor for donor in DONOR_NAMES if not donor in donor_set_1]
    donor_sets = [donor_set_1, donor_set_2]

    plt.figure(figsize=cm_to_inches((4, 3.2)))
    colors = ["royalblue", "forestgreen"]
    labels = [", ".join(donor_set_1), ", ".join(donor_set_2)]
    for i, comm in enumerate(donor_sets):
        vals_donors = vals.loc[genes, comm]
        for j, gene in enumerate(genes):
            x_pos = j + (i - 0.5) * 0.35  # slight shift for different communities
            gene_vals = vals_donors.loc[gene].dropna()
            plt.scatter([x_pos] * len(gene_vals), gene_vals, 
                    color=colors[i], alpha=1, s=10, 
                    label=labels[i] if j == 0 else "", edgecolors='black', linewidth=0.5)
    for donor in vals.columns:
        vals_donors = vals.loc[genes, donor]
        if donor in donor_set_1:
            i = 0
        else:
            i = 1
        plt.plot(genes, vals_donors, 
                color=colors[i], alpha=0.5, linewidth=0.5)

    plt.tick_params(axis='x', length=0)
    plt.xticks(range(len(genes)), genes, rotation=90, fontsize=6)
    plt.ylabel(f"log2(basline log2FC)", fontsize=6)
    #plt.legend(fontsize=6, loc="upper left")
    plt.tight_layout()
    plt.title(cell_type)
    plt.savefig(os.path.join(current_plot_folder, f"gene_expression_dots_donor_baseline_information_{cell_type}.svg"), dpi=300)

In [ ]:
# Is this consistent across stimulations and for a gene set taken from CytoSig?
mean_vals_across_stim = sum_gene_count_df_norm_by_donor_filter.groupby(level=[0, 2], axis=1).median()
cell_type = "CD4 T cell"

genes = [
    'IFIT3',
    'IFI44',
    'IFIT2',
    'IFI6',
    'SAMD9',
    'IFITM1',
    'IFI35',
    'STAT1',
    'SP110'
 ]

donor_set_1 = ["D1", "D3", "D4", "D10"]
donor_set_2 = [donor for donor in DONOR_NAMES if not donor in donor_set_1]
donor_sets = [donor_set_1, donor_set_2]

mean_vals_across_stim[cell_type] = mean_vals_across_stim[cell_type].div(mean_vals_across_stim[cell_type].max(axis=1).fillna(1), axis=0)

plt.figure(figsize=cm_to_inches((8, 4)))
for donor in DONOR_NAMES:
    donor_color = "red" if donor in donor_set_1 else "blue"
    plt.plot(
        genes,
        np.log2(mean_vals_across_stim[cell_type].loc[genes, donor]),
        color = donor_color,
    )
plt.ylabel("log2(normalized expression)")
plt.xlabel("IFN genes")
plt.savefig(os.path.join(current_plot_folder, f"2.4_donor_variability_{cell_type.replace(' ', '').lower()}.png"),
            bbox_inches="tight", dpi=300)

In [ ]:
# plot inflammation-related genes of interest
for cell_type in ["CD4 T cell"]:
    genes = ["IFIT1", "IFIT2", "IFIT3", "RSAD2", "OAS1", "MX1", "TNFSF10", "IL1RN"]
    vals = np.log2(donor_pbs_ratio_to_median_df[cell_type])

    plt.figure(figsize=cm_to_inches((7,7)))
    labels = [", ".join(donor_set_1), ", ".join(donor_set_2)]
    for i, donor in enumerate(DONOR_NAMES):
        vals_donors = vals.loc[genes, donor]
        for j, gene in enumerate(genes):
            x_pos = j
            gene_vals = vals_donors.loc[gene]
            plt.scatter([x_pos], [gene_vals], 
                    color=DONOR_COLORS[donor], alpha=1, s=10, 
                    label=donor if j == 0 else "", edgecolors='black', linewidth=0.5)

    plt.tick_params(axis='x', length=0)
    plt.xticks(range(len(genes)), genes, rotation=90, fontsize=6)
    plt.ylabel(f"log2(basline log2FC)", fontsize=6)
    plt.legend(fontsize=6, loc=[1, 0])
    plt.tight_layout()
    plt.title(cell_type)
    plt.savefig(os.path.join(current_plot_folder, f"gene_expression_dots_donor_baseline_information_{cell_type}_individual_donors.svg"), dpi=300)

In [ ]:
donor_set_1 = ["D1", "D3", "D4", "D10"]
donor_set_2 = [donor for donor in DONOR_NAMES if not donor in donor_set_1]

# also plot the normalized expression values as a sanity check
for cell_type in ["B cell", "CD4 T cell", "NK CD56low", "CD14 Mono"]:
    genes = ["IFIT1", "IFIT2", "IFIT3", "RSAD2", "OAS1", "MX1", "TNFSF10", "IL1RN"]
    vals = np.log2(sum_gene_count_df_norm_by_donor[cell_type]["PBS"])
    donor_sets = [donor_set_1, donor_set_2]

    plt.figure(figsize=cm_to_inches((4.5, 3.2)))
    colors = ["royalblue", "forestgreen"]
    labels = [", ".join(donor_set_1), ", ".join(donor_set_2)]
    for i, comm in enumerate(donor_sets):
        vals_donors = vals.loc[genes, comm]
        for j, gene in enumerate(genes):
            x_pos = j + (i - 0.5) * 0.35  # slight shift for different communities
            gene_vals = vals_donors.loc[gene].dropna()
            plt.scatter([x_pos] * len(gene_vals), gene_vals, 
                    color=colors[i], alpha=1, s=10, 
                    label=labels[i] if j == 0 else "", edgecolors='black', linewidth=0.5)

    plt.tick_params(axis='x', length=0)
    plt.xticks(range(len(genes)), genes, rotation=90, fontsize=6)
    plt.ylabel(f"log2(donor PBS tpm)", fontsize=6)
    #plt.legend(fontsize=6, loc="upper left")
    plt.tight_layout()
    plt.savefig(os.path.join(current_plot_folder, f"gene_expression_dots_donor_baseline_information_raw_{cell_type}.svg"), dpi=300)

In [ ]:
# plot a table of top and bottom genes by donor
top_10_by_donor = {}
bottom_10_by_donor = {}
for donor in DONOR_NAMES:
    vals = donor_pbs_ratio_to_median_df['CD4 T cell'][donor].dropna()
    vals = vals[~vals.index.str.contains('orf')]
    top_10_by_donor[donor] = np.log2(vals.sort_values(ascending=False).head(10))
    bottom_10_by_donor[donor] = np.log2(vals.sort_values(ascending=True).head(10))

# Create a visualization of top and bottom genes by donor
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=cm_to_inches((14.5, 6)), sharex=True)

# Prepare data for plotting
n_genes = 10  # Top/bottom 10 genes

# Top genes subplot
top_data = []
for donor in DONOR_NAMES:
    top_genes = top_10_by_donor[donor]
    for i, (gene, value) in enumerate(top_genes.items()):
        top_data.append({'donor': donor, 'gene': gene, 'value': value, 'rank': i})

top_df = pd.DataFrame(top_data)

# Create heatmap data for top genes
top_pivot = top_df.pivot_table(index='rank', columns='donor', values='value', fill_value=0)
top_gene_names = {}
for donor in DONOR_NAMES:
    top_genes = top_10_by_donor[donor]
    for i, gene in enumerate(top_genes.index):
        if i not in top_gene_names:
            top_gene_names[i] = []
        top_gene_names[i].append(gene)

# Bottom genes subplot
bottom_data = []
for donor in DONOR_NAMES:
    bottom_genes = bottom_10_by_donor[donor]
    for i, (gene, value) in enumerate(bottom_genes.items()):
        bottom_data.append({'donor': donor, 'gene': gene, 'value': value, 'rank': i})

bottom_df = pd.DataFrame(bottom_data)
bottom_pivot = bottom_df.pivot_table(index='rank', columns='donor', values='value', fill_value=0)

top_pivot = top_pivot.reindex(columns=DONOR_NAMES)
bottom_pivot = bottom_pivot.reindex(columns=DONOR_NAMES)

# Plot top genes
im1 = ax1.imshow(top_pivot.values, cmap='Reds', aspect='auto')
ax1.set_yticks(range(n_genes))
ax1.set_yticklabels([f"#{i+1}" for i in range(n_genes)])
ax1.set_title('Top 10 up- and downregulated genes by donor (CD4 T cell)', fontsize=7.5)
ax1.set_ylabel('Gene rank', fontsize=7)
ax1.tick_params(length=0)

# Add gene names as text annotations
for i in range(n_genes):
    for j, donor in enumerate(DONOR_NAMES):
        gene = top_10_by_donor[donor].index[i]
        value = top_10_by_donor[donor].iloc[i]
        ax1.text(j, i, f'{gene}', ha='center', va='center', 
                fontsize=5, color='white' if value > top_pivot.values.max()*0.7 else 'black')

# Plot bottom genes
im2 = ax2.imshow(bottom_pivot.values, cmap='Blues_r', aspect='auto')
ax2.set_yticks(range(n_genes))
ax2.set_yticklabels([f"#{i+1}" for i in range(n_genes)])
ax2.set_xticks(range(len(DONOR_NAMES)))
ax2.set_xticklabels(DONOR_NAMES, rotation=0)
ax2.set_ylabel('Gene rank', fontsize=7)
ax2.tick_params(length=0)

# Add gene names as text annotations
for i in range(n_genes):
    for j, donor in enumerate(DONOR_NAMES):
        gene = bottom_10_by_donor[donor].index[i]
        value = bottom_10_by_donor[donor].iloc[i]
        ax2.text(j, i, f'{gene}', ha='center', va='center', 
                fontsize=5, color='white' if abs(value) > abs(bottom_pivot.values.min())*0.7 else 'black')

# Add colorbars
cbar1 = plt.colorbar(im1, ax=ax1, shrink=1)
cbar1.set_label('log2(baseline FC)', fontsize=6)
cbar2 = plt.colorbar(im2, ax=ax2, shrink=1)
cbar2.set_label('log2(baseline FC)', fontsize=6)

plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, f"top_bottom_genes_by_donor_CD4T.svg"), dpi=300)

In [ ]:
corrs = {}
for cell_type in ALL_CELL_TYPES:
    df = np.log2(donor_pbs_ratio_to_median_df[cell_type][DONOR_NAMES])
    corr = df.corr()
    corrs[cell_type] = corr
corrs = pd.concat(corrs, axis=1).T
corrs = corrs.groupby(level=1).mean().loc[DONOR_NAMES][DONOR_NAMES]

fig = plt.figure(figsize=cm_to_inches((6,5)))
sns.heatmap(corrs, vmin=-1, vmax=1, cmap='coolwarm',
            cbar_kws={"label": f'Pearson r baseline log2FC', "shrink": 0.735}, square=True)

plot_overview_labels_along_y_axis(
    fig,
    plt.gca(),
    labels=['younger\nmale', 'younger\nfemale', 'older\nfemale'],
    vals_per_label=pd.Series(donor_groups).value_counts(),
    x_offset=-2.5,
    no_y_vals=len(DONOR_NAMES)
)
plt.tick_params(axis='x', which='both', bottom=False, top=False, length=0)
plt.tick_params(axis='y', which='both', left=False, right=False, length=0)
plt.xlabel("")
plt.ylabel("")
#plt.title("B cell, CD4 T cell, NK CD16low, CD14 Mono", fontsize=6.5)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "pbs_ratio_to_median_corr.svg"))    

In [ ]:
# calculate mean corr within groups
corrs_group_1 = get_upper_triangle(corrs.loc[donor_set_1][donor_set_1])
print(f"mean corr within group 1: {corrs_group_1.stack().mean():.3f}")
print(f"std corr within group 1: {corrs_group_1.stack().std():.3f}")

In [ ]:
plt.figure(figsize=cm_to_inches((3.8,3.2)))

# plot all other donors
for donor in DONOR_NAMES:
    if donor == "D2":
        continue
    series_donor = np.log2(donor_pbs_ratio_to_median_df['CD14 Mono'][donor])
    series_edger = edger_df_log2FoldChange['CD14 Mono']['IL-32-beta']
    series_donor, series_edger = get_common_index(series_donor, series_edger)    
    plt.scatter(series_donor.iloc[::5], series_edger.iloc[::5], alpha=0.5, s=1, color="black", edgecolors="none", rasterized=True)

# plot donor D2 in color
series_donor = np.log2(donor_pbs_ratio_to_median_df['CD14 Mono']['D2'])
series_edger = edger_df_log2FoldChange['CD14 Mono']['IL-32-beta']
series_donor, series_edger = get_common_index(series_donor, series_edger)    
plt.scatter(series_donor, series_edger, alpha=0.5, s=1, color=DONOR_COLORS["D2"], edgecolors="none", rasterized=True)

# labeling
plt.xlabel(r'baseline log2FC')
plt.xlim(-10, 10)
plt.ylim(-10, 10)
plt.xticks([-10, -5, 0, 5, 10])
plt.ylabel(introduce_greek_letters('consensus log2FC(IL-32-beta)'))
r = np.corrcoef(series_donor, series_edger)[0,1]
plt.text(-9,7,f"r={r:.2f}", color=DONOR_COLORS["D2"])
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "D2_CD14_Mono_scatter.svg"), dpi=400)

In [ ]:
# inflammation donors experienced an interferon-like response at baseline
cell_type = "CD4 T cell"
cytokine = "IFN-beta"
donors = ["D1", "D3", "D4", "D10", "D11"]
plt.figure(figsize=cm_to_inches((5, 5)))
for donor in donors:
    series_donor = np.log2(donor_pbs_ratio_to_median_df[cell_type][donor])
    series_edger = edger_df_log2FoldChange[cell_type]['IFN-beta']
    series_donor, series_edger = get_common_index(series_donor, series_edger)
    plt.scatter(series_donor, series_edger, s=1, alpha=0.5, color=DONOR_COLORS[donor],
                edgecolors="none", rasterized=True, label=donor)
plt.xlabel(r'log2(donor PBS/median PBS)')
plt.xlim(-4, 5)
plt.ylim(-4, 5)
plt.ylabel(introduce_greek_letters(f'edgeR log2FC({cytokine})'))
plt.title(cell_type, fontsize=6.5)
plt.tight_layout()
plt.legend(loc=[1,0], fontsize=6, markerscale=6)
plt.savefig(os.path.join(current_plot_folder, f"{cell_type}_{cytokine}_scatter.svg"), dpi=400)

In [ ]:
all_corrs = {}
all_dot_products = {}
for donor in DONOR_NAMES:
    df_donor = np.log2(donor_pbs_ratio_to_median_df.loc[:, (slice(None), donor)]).droplevel(1, axis=1)
    df_edger = edger_df_log2FoldChange.loc[:, (slice(None), 'IL-32-beta')].droplevel(1, axis=1)
    corr = df_edger.corrwith(df_donor, method="pearson")[ALL_CELL_TYPES]
    all_corrs[donor] = corr
    
    # Calculate dot product for each cell type
    dot_products = {}
    for cell_type in ALL_CELL_TYPES:
        if cell_type in df_donor.columns and cell_type in df_edger.columns:
            # Get common index between the two series
            donor_series = df_donor[cell_type].dropna()
            edger_series = df_edger[cell_type].dropna()
            common_idx = donor_series.index.intersection(edger_series.index)
            
            if len(common_idx) > 0:
                dot_product = np.dot(donor_series.loc[common_idx], edger_series.loc[common_idx])
                dot_products[cell_type] = dot_product
            else:
                dot_products[cell_type] = np.nan
        else:
            dot_products[cell_type] = np.nan
    
    all_dot_products[donor] = pd.Series(dot_products)

all_corrs = pd.DataFrame(all_corrs)
all_dot_products = np.log2(abs(pd.DataFrame(all_dot_products)+1))
all_dot_products = (all_dot_products - all_dot_products.max().max()) - all_dot_products.min().min()

plt.figure(figsize=cm_to_inches((3.8, 4.7)))
for donor_name in DONOR_NAMES:
    sizes = all_dot_products.loc[ALL_CELL_TYPES_EXCLUDE[::-1], donor_name]
    plt.scatter(
        all_corrs.loc[ALL_CELL_TYPES_EXCLUDE[::-1], donor_name],
        np.arange(len(ALL_CELL_TYPES_EXCLUDE)),
        alpha=1,
        s=3,
        color="black" if donor_name !="D2" else DONOR_COLORS["D2"],
        label="D2" if donor_name == "D2" else None,
        edgecolors="none",
        rasterized=True
        )
    
plt.yticks(np.arange(len(ALL_CELL_TYPES_EXCLUDE)), ALL_CELL_TYPES_EXCLUDE[::-1], fontsize=6)
plt.tick_params(axis='y', length=0)
plt.xlabel(introduce_greek_letters('r(baseline log2FC, consensus log2FC)'))
plt.xticks([-0.5,0,0.5,1])
plt.title(introduce_greek_letters("IL-32-beta"), fontsize=7)
# plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "donor_il32_corr_by_cell_type.svg"), dpi=400)

### 2.4.2 analyze response magnitudes

#### standard response magnitude

This has the potentially problematic effect of normalizing away maximum reponse differences per donor.

In [ ]:
all_normalized_euclidean_distances_by_donor = {}
all_raw_euclidean_distances_by_donor = {}
for donor in DONOR_NAMES:
    print(f"Processing donor {donor}")
    all_distances_norm = {}
    all_distances_raw = {}
    df = sum_gene_count_df_norm_by_donor_filter.loc[:, (slice(None), slice(None), donor)].copy().droplevel(2, axis=1)
    df_log = np.log2(df + 1)

    for cell_type in ALL_CELL_TYPES_EXCLUDE:
        distances = {}
        
        # get the inclusive filter and average normalized log2 data
        all_inclusive_filter = inclusive_filters_by_cell_type[cell_type]
        average_norm_filter = df_log.loc[all_inclusive_filter, :]
        
        for cytokine in CYTOKINE_ORDER:
            if cytokine in average_norm_filter[cell_type].columns:
                distance = compute_response_euclidean_distance(
                    average_norm_filter, cell_type, cytokine,
                    dropna=True, min_genes=1000,
                    print_min_gene_warning=True, length_norm=True)
                distances[cytokine] = distance
        distances_series = pd.Series(distances)
        all_distances_raw[cell_type] = distances_series
        all_distances_norm[cell_type] = winsorize_and_normalize_distances(distances_series)
        
    all_distances_norm = pd.DataFrame(all_distances_norm)
    all_distances_raw = pd.DataFrame(all_distances_raw)
    all_normalized_euclidean_distances_by_donor[donor] = all_distances_norm
    all_raw_euclidean_distances_by_donor[donor] = all_distances_raw

all_raw_euclidean_distances_by_donor = pd.concat(all_raw_euclidean_distances_by_donor, axis=1)
all_normalized_euclidean_distances_by_donor = pd.concat(all_normalized_euclidean_distances_by_donor, axis=1)/100.0

### donor-specific FC

In [ ]:
all_donor_ratios_to_PBS = {}
for donor in all_counts_by_condition.index.levels[2]:
    subset_norm_sum = sum_gene_count_df_norm_by_donor_filter.loc[:, (slice(None), slice(None), donor)].droplevel(2, axis=1)
    ratio_to_PBS = pd.DataFrame(index=subset_norm_sum.index, columns=subset_norm_sum.columns)
    for cell_type in subset_norm_sum.columns.levels[0]:
        if cell_type in ALL_CELL_TYPES:
            pbs_values = subset_norm_sum.loc[:, (cell_type, 'PBS')]
            for cytokine in subset_norm_sum.loc[:, cell_type].columns:
                ratio_to_PBS.loc[:, (cell_type, cytokine)] = subset_norm_sum.loc[:, (cell_type, cytokine)] / pbs_values
    
    ratio_to_PBS = ratio_to_PBS.drop('PBS', level=1, axis=1)
    subset_counts_by_condition = all_counts_by_condition.loc[(slice(None), slice(None), donor)]
    subset_counts_by_condition_filter = subset_counts_by_condition[subset_counts_by_condition > 20] # this is the cell count
    valid_columns = [col for col in ratio_to_PBS.columns if col in subset_counts_by_condition_filter.index]
    dropped_columns = [col for col in ratio_to_PBS.columns if col not in subset_counts_by_condition_filter.index]
    
    ratio_to_PBS = ratio_to_PBS[valid_columns]
    all_donor_ratios_to_PBS[donor] = ratio_to_PBS.astype(float)
    
# Merge all ratios to PBS into a single dataframe
all_donor_ratios_to_PBS = pd.concat(all_donor_ratios_to_PBS.values(), keys=all_donor_ratios_to_PBS.keys(), names=['donor'], axis=1)

#### raw response strength

This is an alternate way of calculating a type of response that might be less noisy for individual donor calculations.

In [ ]:
# This needs to be done to avoid extremely weird silent errors in the pandas backend ...
GENE_SET_BY_CELL_TYPE = {}
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    gene_set = inclusive_filters_by_cell_type[cell_type]
    GENE_SET_BY_CELL_TYPE[cell_type] = list(gene_set)

def get_donor_vals_ratios(cytokine, cell_type):
    """Returns linear ratios of cytokine expression to PBS expression for a given cell type and cytokine"""
    gene_set = GENE_SET_BY_CELL_TYPE[cell_type]
    try:
        sub = all_donor_ratios_to_PBS.loc[gene_set, (slice(None), cell_type, cytokine)] # this sort of slicing is sometimes weirdly non-deterministic?
        # sub: (genes x donors)
        if sub.empty:
            print(sub)
            print("Empty DataFrame for cytokine: ", cytokine, " and cell type: ", cell_type)
        sub = sub.droplevel([1, 2], axis=1)  # keep only donor columns
        relevant_donors = [donor for donor in DONOR_NAMES if donor in sub.columns]
        sub = sub.reindex(columns=relevant_donors)
        sub = sub.astype(float)
    except Exception as e:
        # Missing combination -> return empty DF with the same donor columns
        sub = pd.DataFrame(index=gene_set, columns=DONOR_NAMES, dtype=float)
    return sub

In [ ]:
def get_abs_sum(cytokine, cell_type):
    vals = get_donor_vals_ratios(cytokine, cell_type)   # raw FC genes × donors
    t = np.abs(np.log2(vals))
    t = np.exp2((t - 0.5).clip(lower=0)) - 1
    return np.sqrt(t.sum(axis=0, min_count=1))

all_abs_sums_by_donor_dict = {}
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    for cytokine in CYTOKINE_ORDER:
        abs_sum = get_abs_sum(cytokine, cell_type)
        all_abs_sums_by_donor_dict[(cell_type, cytokine)] = abs_sum

all_abs_sums_by_donor = pd.DataFrame(all_abs_sums_by_donor_dict).loc[DONOR_NAMES]

In [ ]:
# Fix the ordering to cytokine x [cell_type, donor]
all_raw_euclidean_distances_by_donor = all_raw_euclidean_distances_by_donor.swaplevel(0,1, axis=1)
all_normalized_euclidean_distances_by_donor = all_normalized_euclidean_distances_by_donor.swaplevel(0,1, axis=1)

# fix the ordering to cytokine x [cell_type, donor]
df_swapped = all_abs_sums_by_donor.stack(level=1)
df_swapped = df_swapped.unstack(level=0)
all_abs_sums_by_donor = df_swapped

In [ ]:
all_normalized_euclidean_distances_by_donor.head(3)

In [ ]:
all_abs_sums_by_donor.head(3)

#### raw response strength distribution

In [ ]:
fig, axes = plt.subplots(
    1, len(ALL_CELL_TYPES_EXCLUDE),
    sharey=True,
    figsize=cm_to_inches((18, 18))
)

axes = np.atleast_1d(axes)

for i, (ax, cell_type) in enumerate(zip(axes, ALL_CELL_TYPES_EXCLUDE)):
    data = all_abs_sums_by_donor[cell_type].copy()
    data = data - data.min().min()
    data = data / data.max().max()
    data = data.loc[CYTOKINE_ORDER, DONOR_NAMES]
    sns.boxplot(
        data=data.T,
        ax=ax,
        orient="h",
        flierprops=dict(marker='o', markersize=2,
                        markerfacecolor='black', markeredgecolor='none',
                        markeredgewidth=0.5)
    )

    ax.set_title(cell_type, fontsize=6.5)
    ax.set_xlim(0, 1)

    # y-axis: show ticks/labels only on the leftmost subplot
    ax.tick_params(
        axis='y', rotation=0, labelsize=5.5,
        left=(i == 0), labelleft=(i == 0),
        right=False, labelright=False
    )
    # ax.tick_params(axis='x', rotation=90, labelsize=5.5)
    
    # Apply greek labels only once (shared y-axis)
    if i == 0:
        apply_greek_labels(ax)
        vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()
        plot_overview_labels_along_y_axis(
            fig, ax,
            SINGLE_FAMILY_ORDER[::-1], vals_per_label_level1,
            -1.2,
            90,
            fontsize=6, fontweight="bold", color="black", linewidth=1,
            rotation=0)
    ax.set_xticks([0, 0.8], labels=[0, 0.8])

plt.tight_layout()
plt.subplots_adjust(wspace=0.1)

# Add global x-axis label
fig.text(0.55, -0.01, 'Raw Response Strength', ha='center', va='center', fontsize=8)
for format in ["png", "svg"]:
    fig.savefig(os.path.join(current_plot_folder, f"2.4_boxplot_raw_response_strength_distribution.{format}"), dpi=300, bbox_inches='tight')

#### variability of relative response strengths
variability by cell type is here defined as the standard deviation of the response magnitude relative to the maximum response magnitude

In [ ]:
cell_type = "CD4 T cell"
donor_pairs = [("D5", "D7"), ("D5", "D3")]
plt.figure(figsize=cm_to_inches((3.4,3)))
symbols = ['o', 's']
colors = ['black', "lightgray"]
for i, donor_pair in enumerate(donor_pairs):
    donor1 = donor_pair[0]
    donor2 = donor_pair[1]
    
    vals1 = all_normalized_euclidean_distances_by_donor.loc[:, (cell_type, donor1)]
    vals2 = all_normalized_euclidean_distances_by_donor.loc[:, (cell_type, donor2)]
    
    plt.scatter(vals1, vals2, s=5, edgecolors="none", color=colors[i], label=f"{donor1}-{donor2}", marker=symbols[i])
    
    # for debugging only
    # Add tiny text labels for each point
    # for j, cytokine in enumerate(vals1.index):
    #     plt.text(vals1.iloc[j], vals2.iloc[j], cytokine, fontsize=2, ha='center', va='center', alpha=0.7)
    
plt.xlabel("response magnitude")
plt.ylabel("response magnitude")
plt.xticks([0, 0.5, 1])
plt.yticks([0, 0.5, 1])
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "response_mag_donor_donor.svg"))

In [ ]:
all_variability_by_cell_type = {}

for cell_type in ALL_CELL_TYPES_EXCLUDE:
    strong_impact = strong_impact_by_cell_type[cell_type].copy()
    distances_cell_type = all_normalized_euclidean_distances_by_donor[cell_type]
    distances_cell_type = distances_cell_type.div(distances_cell_type.max(axis=1).clip(lower=0.5), axis=0)
    response_strength_variability = distances_cell_type.std(axis=1)
    strong_impact = strong_impact.loc[strong_impact]
    response_strength_variability = response_strength_variability[strong_impact.index]
    all_variability_by_cell_type[cell_type] = response_strength_variability

all_variability_by_cell_type = pd.DataFrame(all_variability_by_cell_type)
all_variability_by_cell_type = all_variability_by_cell_type.reindex(CYTOKINE_ORDER, fill_value=0)
all_variability_by_cell_type = all_variability_by_cell_type.iloc[::-1]

In [ ]:
# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 20)))
mean = all_variability_by_cell_type.mean(axis=1).values
std = all_variability_by_cell_type.std(axis=1).values
min_val = mean[mean > 0].min()
max_val = mean.max()

plt.barh(all_variability_by_cell_type.index, mean, color="black")
plt.errorbar(mean, range(len(mean)), xerr=std, fmt='none', color='lightgray', alpha=1, capsize=0, capthick=0.5)

plt.ylim(-1, len(CYTOKINE_ORDER))

fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.title("donor\nresponse\nstrength\nvariability", fontsize=7)
plt.xlim(min_val-0.1*(max_val-min_val), max_val+0.1*(max_val-min_val))
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.4_response_strength_variability.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
il32_variability = {}
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    vals = all_abs_sums_by_donor.loc["IL-32-beta", cell_type]
    if vals.isna().sum() > 1:
        print(f"NaNs found in {cell_type}")
        continue
    vals = vals/vals.max()
    il32_variability[cell_type] = vals
    
il32_variability = pd.DataFrame(il32_variability)
plt.figure(figsize=cm_to_inches((3.8,1)))
plt.bar(il32_variability.index, il32_variability.mean(axis=1), color='black', width=0.7)
plt.errorbar(range(len(il32_variability.index)), il32_variability.mean(axis=1), yerr=il32_variability.std(axis=1),
             fmt='none', color='lightgray', alpha=1, capsize=0, capthick=0.5)
plt.tick_params(axis='x', length=0)
plt.tick_params(axis='y', length=0)
plt.xticks(fontsize=6, rotation=90)
plt.yticks(fontsize=6)
plt.xlim(-0.5, len(DONOR_NAMES)-0.5)

plt.ylabel("raw response\nstrength", fontsize=6)
plt.savefig(os.path.join(current_plot_folder, "IL-32-beta-raw-response-variability.svg"))

#### Pearson r of response magnitudes between donors (per cell type)

In [ ]:
records = []
for cell_type in ALL_CELL_TYPES:
    corr = all_normalized_euclidean_distances_by_donor[cell_type].corr()
    donors = corr.index
    for i, d1 in enumerate(donors):
        for j, d2 in enumerate(donors):
            if i == j: # skip diagonal
                continue
            records.append({
                "cell_type": cell_type,
                "donor1": d1,
                "donor2": d2,
                "correlation": corr.loc[d1, d2]
            })

long_df = pd.DataFrame(records)
long_df = long_df.drop_duplicates(subset=["cell_type", "donor1", "donor2"])

In [ ]:
fig, axes = plt.subplots(len(DONOR_NAMES), len(DONOR_NAMES), 
                         figsize=cm_to_inches((5, 10)),
                         sharex=True, sharey=True)

for i, d1 in enumerate(DONOR_NAMES):
    for j, d2 in enumerate(DONOR_NAMES):
        ax = axes[i, j]

        if j > i:   # upper triangle blank
            ax.axis("off")
            continue
        elif i == j:   # diagonal blank
            ax.axis("off")
            continue

        # get canonical (sorted) donor pair
        dmin, dmax = sorted([d1, d2])
        subset = long_df[(long_df["donor1"] == dmin) & (long_df["donor2"] == dmax)]
        if len(subset) == 0:
            subset = long_df[(long_df["donor1"] == dmax) & (long_df["donor2"] == dmin)]

        # mean line
        mean = subset["correlation"].mean()
        ax.plot([-0.3, 0.3], [mean, mean], color="black", linestyle="-", linewidth=1.5)

        # scatter with jitter
        for ct, row in subset.groupby("cell_type"):
            jitter = np.random.uniform(-0.2, 0.2, size=len(row))
            ax.scatter(
                jitter, 
                row["correlation"],
                color=CELL_TYPE_COLORS.get(ct, "grey"),
                s=10,
                alpha=0.8,
                edgecolors="none"
            )

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_xlim(-0.5, 0.5)
        ax.set_ylim(0, 1)
        if j == 0:
            ax.set_ylabel(d1)
        if i == len(DONOR_NAMES)-1:
            ax.set_xlabel(d2, rotation=0)

fig.text(-0.05, 0.5, 'Pearson r of response magnitudes [0, 1]', va='center', rotation=90, fontsize=7)

plt.tight_layout()
plt.subplots_adjust(wspace=0.075, hspace=0.05)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(current_plot_folder, f"2.4_correlation_of_response_magnitudes.{format}"), dpi=300)

In [ ]:
donor_group_1 = ["D1", "D3", "D4", "D10"]
donor_group_2 = [donor for donor in DONOR_NAMES if not donor in donor_group_1]

fig, axes = plt.subplots(2, 2, 
    figsize=cm_to_inches((2.35, 3.0)),
    sharex=False, sharey=False)

group_name_1 = ["D1 D3\nD4 D10"]
group_name_2 = ["D2 D5 D6\nD7 D8 D9\nD11 D12"]
for i, groups in enumerate(itertools.product([donor_group_1, donor_group_2], [donor_group_1, donor_group_2])):
    vals = long_df[long_df['donor1'].isin(groups[0])]
    vals = vals[vals['donor2'].isin(groups[1])]
    mean = vals['correlation'].mean()
    
    row = i % 2
    col = i // 2
    ax = axes[row, col]
    ax.plot([-0.3, 0.3], [mean, mean], color="black", linestyle="-", linewidth=1.5)

    # scatter with jitter
    mean_by_cell_type = vals.groupby("cell_type")['correlation'].mean()
    for ct, val in mean_by_cell_type.items():
        jitter = np.random.uniform(-0.2, 0.2)
        ax.scatter(
            jitter, 
            val,
            color=CELL_TYPE_COLORS.get(ct, "grey"),
            s=10,
            alpha=0.8,
            edgecolors="none"
        )

    ax.set_xticks([])
    ax.set_xlim(-0.5, 0.5)
    ax.set_ylim(0, 1)
    
    if i > 1:  # Right plots
        ax.yaxis.tick_right()
        ax.yaxis.set_label_position("right")
        ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1.0'], fontsize=6)
    else:
        ax.yaxis.set_ticks_position("none")        # no ticks at all
        ax.set_yticks([])                          # enforce no ticks
        ax.set_yticklabels([])                     # enforce no labels
    
    # Set group names based on which groups are being compared
    if i % 2 == 1:
        if groups[0] == donor_group_1:
            x_label = group_name_1[0]
        else:
            x_label = group_name_2[0]
        ax.set_xlabel(x_label)
    
    if i < 2:
        if groups[1] == donor_group_1:
            y_label = group_name_1[0]
        else:
            y_label = group_name_2[0]
        ax.set_ylabel(y_label)
fig.suptitle("Pearson r of response magnitudes", fontsize=7)
plt.savefig(os.path.join(current_plot_folder, "Response_mag_by_group.svg"))

In [ ]:
# switch with donor_group_2 for the other value
vals = long_df[long_df['donor1'].isin(donor_group_1)]
vals = vals[vals['donor2'].isin(donor_group_1)]
mean = vals['correlation'].mean()
std = vals['correlation'].std()
print(f"mean correlation between groups: {mean:.3f} ± {std:.3f}")

In [ ]:
discrepancies = {}
for cell_type in ["B cell", "CD4 T cell", "NK CD56low", "CD14 Mono"]:
    strong_impact = strong_impact_by_cell_type[cell_type].index # this is all of them
    # don't really need to subset for this one?
    
    group_1_distances = all_normalized_euclidean_distances_by_donor[cell_type].loc[strong_impact, donor_group_1].mean(axis=1)
    group_2_distances = all_normalized_euclidean_distances_by_donor[cell_type].loc[strong_impact, donor_group_2].mean(axis=1)
    
    discrepancies[cell_type] = group_1_distances - group_2_distances
discrepancies = pd.DataFrame(discrepancies)

In [ ]:
# Prepare data for plotting
mean_vals_series = discrepancies.mean(axis=1).sort_values()
mean_vals_series = mean_vals_series[abs(mean_vals_series) > 0.1] # only plot interesting ones
mean_vals_data = mean_vals_series.to_numpy()

std_vals_series = discrepancies.std(axis=1)
std_vals_series = std_vals_series[mean_vals_series.index]
std_vals_data = std_vals_series.to_numpy()

# Calculate y1 and y2 for the shaded area
y1_data = mean_vals_data - std_vals_data
y2_data = mean_vals_data + std_vals_data

In [ ]:
from unicode_utils import apply_greek_labels
plt.figure(figsize=cm_to_inches((8,3.5)))

x_plot_indices = np.arange(len(mean_vals_series))

# Plot mean line using NumPy arrays
plt.plot(x_plot_indices, mean_vals_data, '-', markersize=2, color="black")

# Plot standard deviation as shaded area
plt.fill_between(x_plot_indices, 
                 y1_data,
                 y2_data,
                 alpha=0.3,
                 color='gray',
                 edgecolors="none")

for cell_type in discrepancies.columns:
    series = discrepancies[cell_type].loc[mean_vals_series.index]
    plt.scatter(x_plot_indices, series.to_numpy(), s=6, 
                color=CELL_TYPE_COLORS[cell_type], alpha=1, label=f"{cell_type}", zorder=2, edgecolor="none")

plt.plot([-1, len(x_plot_indices)], [0, 0], color='black', linewidth=0.5)

plt.xticks(x_plot_indices, mean_vals_series.index, rotation=90, ha='center', fontsize=5.5)
plt.ylabel('response magnitude\n(group 1-group 2)')
plt.margins(x=0.01) # Remove whitespace on left and right

plt.xlim(-0.5,len(x_plot_indices) - 0.5)
plt.ylim(-0.6, 0.6)  # Set y-axis limits
plt.gca().set_yticks([-0.6, -0.4, -0.2, 0, 0.2, 0.4, 0.6])

apply_greek_labels(plt.gca())
plt.legend(loc="lower right", ncol=2, fontsize=6)
plt.tick_params(axis='x', length=0)
plt.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, "group1_group2_discrepancy_long.svg"))

In [ ]:
plt.figure(figsize=cm_to_inches((1.0, 3.7)))
cell_type_means = long_df.groupby("cell_type")['correlation'].mean()[ALL_CELL_TYPES_EXCLUDE].iloc[::-1]
errorbar = long_df.groupby("cell_type")['correlation'].std()[ALL_CELL_TYPES_EXCLUDE].iloc[::-1]
colors = [CELL_TYPE_COLORS.get(ct, "grey") for ct in cell_type_means.index]
plt.barh(cell_type_means.index, cell_type_means.values, color=colors)
plt.errorbar(cell_type_means.values, cell_type_means.index, xerr=errorbar.values, fmt='none', color='black',
             alpha=1, capsize=1.5, capthick=0.5, linewidth=1)
plt.xticks(fontsize=6)

plt.ylim(-0.75, len(cell_type_means)-0.25)
plt.xlim(0, 1)
plt.xlabel("Pearson r")
plt.ylabel("")
for format in ["png", "svg"]:
    plt.savefig(os.path.join(current_plot_folder, f"2.4_correlation_of_response_magnitudes_by_cell_type.{format}"), dpi=300)

In [ ]:
plt.figure(figsize=cm_to_inches((1.0, 3.7)))
donor_means = long_df.groupby("donor2")['correlation'].mean()[DONOR_NAMES].iloc[::-1]
errorbar = long_df.groupby("donor2")['correlation'].std()[DONOR_NAMES].iloc[::-1]
plt.barh(donor_means.index, donor_means.values, color="gray")
plt.errorbar(donor_means.values, donor_means.index, xerr=errorbar.values, fmt='none', color='black',
             alpha=1, capsize=1.5, capthick=0.5, linewidth=1)
plt.xticks(fontsize=6)

plt.ylim(-0.75, len(donor_means)-0.25)
plt.xlim(0, 1)
plt.xlabel("Pearson r")
plt.ylabel("")
for format in ["png", "svg"]:
    plt.savefig(os.path.join(current_plot_folder, f"2.4_correlation_of_response_magnitudes_by_donor.{format}"), dpi=300)

## 2.4.3 - Donor-specific fold-change analysis

In [ ]:
# correlation between donor-specific fc and edger fc
donor_corrs_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    donor_corrs_by_cytokine = {}
    for cytokine in CYTOKINE_ORDER:
        # get both
        fc_edger = edger_df_log2FoldChange[cell_type][cytokine].copy()
        fc_donors = np.log2(get_donor_vals_ratios(cytokine, cell_type))

        # correlate
        correlations = fc_donors.corrwith(fc_edger, method='pearson')
        donor_corrs_by_cytokine[cytokine] = correlations
    donor_corrs_by_cytokine = pd.DataFrame(donor_corrs_by_cytokine).T
    donor_corrs_by_cell_type[cell_type] = donor_corrs_by_cytokine

In [ ]:
mean_corrs_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    # calculate mean corr for strong impact cytokines only (not entirely necessary but probably more meaningful)
    strong_impact = strong_impact_by_cell_type[cell_type]
    strong_impact = strong_impact[strong_impact].index
    corrs = donor_corrs_by_cell_type[cell_type].loc[strong_impact]
    mean_corrs_by_cell_type[cell_type] = corrs.mean(axis=1)
    
mean_corrs_by_cell_type = pd.DataFrame(mean_corrs_by_cell_type)
mean_corrs_by_cell_type = mean_corrs_by_cell_type.dropna(axis=0, how="all")

In [ ]:
mean = mean_corrs_by_cell_type.mean(axis=1)
std = mean_corrs_by_cell_type.std(axis=1)

mean = mean.reindex(CYTOKINE_ORDER[::-1]).fillna(0)
std = std.reindex(CYTOKINE_ORDER[::-1]).fillna(0)

# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 20)))
min_val = mean[mean > 0].min()
max_val = mean.max()

plt.barh(all_variability_by_cell_type.index, mean, color="black")
plt.errorbar(mean, range(len(mean)), xerr=std, fmt='none', color='lightgray', alpha=1, capsize=0, capthick=0.5)

plt.ylim(-1, len(CYTOKINE_ORDER))

fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.title("mean corr", fontsize=7)
plt.xlim(min_val-0.1*(max_val-min_val), max_val+0.1*(max_val-min_val))
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.4_mean_corr_variability.{format}"), bbox_inches="tight", dpi=300)

#### Find outlier donors for particular cytokine - cell type combinations

In [ ]:
width_factor_donor_outliers = 0.35

donor_corr_outliers_by_cell_type = {}
threshold_by_cell_type = {}
top8_median = {}
for cell_type in ALL_CELL_TYPES:
    outliers_by_cytokine = {}
    threshold_by_cytokine = {}
    for cytokine in donor_corrs_by_cell_type[cell_type].index:
        median = donor_corrs_by_cell_type[cell_type].loc[cytokine].sort_values(ascending=False).head(8).median()
        top8_median[f"{cell_type}_{cytokine}"] = median
        
        non_outlier_range = width_factor_donor_outliers * max(median, 0.6)
        threshold_by_cytokine[cytokine] = median - non_outlier_range
        
        dev_from_median = donor_corrs_by_cell_type[cell_type].loc[cytokine] - median
        outliers = dev_from_median < -non_outlier_range
        outliers_by_cytokine[cytokine] = outliers
        
    threshold_by_cell_type[cell_type] = pd.Series(threshold_by_cytokine)
    outliers_by_cytokine = pd.DataFrame(outliers_by_cytokine)
    donor_corr_outliers_by_cell_type[cell_type] = outliers_by_cytokine
threshold_by_cell_type = pd.DataFrame(threshold_by_cell_type)
    
# Build a DataFrame cytokine × cell_type with counts
corr_outliers_by_cell_type_and_cytokine = []
corr_outliers_by_cell_type_and_donor = {}
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    corr_outliers_by_cell_type_and_donor[cell_type] = donor_corr_outliers_by_cell_type[cell_type].sum(axis=1)
    curr_outliers = donor_corr_outliers_by_cell_type[cell_type].sum(axis=0)
    corr_outliers_by_cell_type_and_cytokine.append(curr_outliers.rename(cell_type))

corr_outliers_by_cell_type_and_cytokine = pd.concat(corr_outliers_by_cell_type_and_cytokine, axis=1).fillna(0)
corr_outliers_by_cell_type_and_donor = pd.DataFrame(corr_outliers_by_cell_type_and_donor).fillna(0)

# Ensure cytokine order matches CYTOKINE_ORDER
corr_outliers_by_cell_type_and_cytokine = corr_outliers_by_cell_type_and_cytokine.reindex(CYTOKINE_ORDER)    

In [ ]:
# visually verify for a specific cell type to make sure this works as expected
cell_type = "CD8 T cell"

fig, ax = plt.subplots(figsize=cm_to_inches((4, 22)))
sns.heatmap(donor_corrs_by_cell_type[cell_type], vmin=0, vmax=0.8, cmap="viridis", cbar=False)
ax.tick_params(axis='y', length=0, labelsize=5.5)
ax.tick_params(axis='x', length=0, labelsize=5.5)
ax.set_yticks(np.arange(0.5, len(donor_corrs_by_cell_type[cell_type].index)))
ax.set_yticklabels(donor_corrs_by_cell_type[cell_type].index, rotation=0)
apply_greek_labels_heatmap(ax)

outliers_curr = donor_corr_outliers_by_cell_type[cell_type].T.copy()
for cytokine, donor in outliers_curr.stack().index:
    if outliers_curr.loc[cytokine, donor]:
        cytokine_idx = donor_corrs_by_cell_type[cell_type].index.get_loc(cytokine)
        donor_idx = donor_corrs_by_cell_type[cell_type].columns.get_loc(donor)
        ax.add_patch(plt.Rectangle((donor_idx, cytokine_idx), 1, 1, fill=False, edgecolor='red', lw=0.8))

plt.title(cell_type, fontsize=7)
plt.show()

In [ ]:
top8_median = pd.Series(top8_median)
top8_median.index = top8_median.index.str.split("_", expand=True)
top8_median = top8_median.unstack()

In [ ]:
# visualize the top8 median correlations for only the cytokines with at least one strong impact
# this basically shows that most donors agree
any_strong = strong_impact_by_cell_type.any(axis=1)
any_strong = any_strong[any_strong].index
top8_median=top8_median[any_strong]
plt.figure(figsize=cm_to_inches((18,2)))
for cell_type in ALL_CELL_TYPES:
    strong_impact = strong_impact_by_cell_type[cell_type].copy()
    top8_strong = top8_median.loc[cell_type]
    top8_strong = top8_strong[top8_strong.index.isin(strong_impact[strong_impact].index)]
    top8_strong = top8_strong.reindex(any_strong)
    plt.scatter(top8_strong.index, top8_strong.values, s=5, color=CELL_TYPE_COLORS[cell_type], edgecolors="none")
plt.tick_params(axis='x', length=0)
plt.xlim(-0.5, len(any_strong)-0.5)
plt.xticks(rotation=90, fontsize=5.5)
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0], fontsize=6)
apply_greek_labels_heatmap(plt.gca())
plt.ylim(0, 1)
plt.show()

In [ ]:
size_scale = 0.5  # same scale used everywhere for s=count*size_scale

fig, (ax_main, ax_leg) = plt.subplots(
    1, 2,
    figsize=cm_to_inches((6, 4)),
    gridspec_kw={'width_ratios': [5, 2]},
    constrained_layout=True
)

# --- main scatter (donor × cell type) ---
for i, donor in enumerate(DONOR_NAMES):
    for j, cell_type in enumerate(ALL_CELL_TYPES_EXCLUDE[::-1]):
        count = corr_outliers_by_cell_type_and_donor.loc[donor, cell_type]
        if count > 0:
            ax_main.scatter(
                i, j,
                s=count * size_scale,
                alpha=1,
                c=CELL_TYPE_COLORS[cell_type],
                edgecolors="none"
            )

ax_main.set_yticks(range(len(ALL_CELL_TYPES_EXCLUDE)))
ax_main.set_yticklabels(ALL_CELL_TYPES_EXCLUDE[::-1], rotation=0)
ax_main.set_xticks(range(len(DONOR_NAMES)))
ax_main.set_xticklabels(DONOR_NAMES, rotation=90)
ax_main.tick_params(length=0)

# --- size legend subplot (shares the same size_scale) ---
example_counts = [1, 5, 10, 25, 50, 100]

# draw largest at top
for k, count in enumerate(reversed(example_counts)):
    y = k
    ax_leg.scatter(
        0, y,
        s=count * size_scale,
        alpha=1,
        c='gray',
        edgecolors='none'
    )
    ax_leg.text(0.6, y, str(count), va='center', ha='left', fontsize=6)

ax_leg.set_title('outlier count', fontsize=7, pad=6)
ax_leg.set_xlim(-0.5, 2.5)
ax_leg.set_ylim(-0.5, len(example_counts) - 0.5)
ax_leg.axis('off')

# --- save once (both subplots in the same figure) ---
for fmt in ['png', 'svg']:
    fig.savefig(
        os.path.join(current_plot_folder, f"corr_outliers_by_donor_x_cell_type_with_legend.{fmt}"),
        dpi=300
    )

plt.figure(figsize=cm_to_inches((4.5, 1.5)))
plt.bar(DONOR_NAMES, corr_outliers_by_cell_type_and_donor.sum(axis=1)[DONOR_NAMES], color="black", width=0.8)
plt.xlim(-0.5, len(DONOR_NAMES)-0.5)
plt.ylabel("total")
plt.tight_layout()
for format in ['png', 'svg']:
    plt.savefig(os.path.join(current_plot_folder, f"corr_outliers_by_donor_x_cell_type_barplot.{format}"), dpi=300)

In [ ]:
plt.figure(figsize=cm_to_inches((18, 5)))

offset = 0
yticks = []
yticklabels = []

for index, cell_type in enumerate(ALL_CELL_TYPES_EXCLUDE[::-1]):
    curr_outliers = corr_outliers_by_cell_type_and_cytokine[cell_type]
    curr_outliers = curr_outliers.reindex(CYTOKINE_ORDER).fillna(0)

    # plot bars with vertical offset
    plt.bar(
        curr_outliers.index,
        curr_outliers.values,
        bottom=offset,
        color=CELL_TYPE_COLORS[cell_type],
        width=0.8,
        label=cell_type
    )
    plt.axhline(y=offset,linewidth=0.5,color="black")
    plt.text(-2-index%2, offset+max(0,max(curr_outliers.values)/2-1), f"{max(curr_outliers.values)}", fontsize=5.5)

    # track midpoint for yticks
    yticks.append(offset + curr_outliers.max() / 2)
    yticklabels.append(cell_type)

    # increment offset for next cell type
    offset += curr_outliers.max() + 0.5   # +1 for spacing

plt.xticks(rotation=90, fontsize=5.5)
plt.yticks([])# yticks, yticklabels)
plt.ylabel("outlier donors", labelpad=15)
plt.xlim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(plt.gca())

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(current_plot_folder, f"stacked_barplot_donor_outliers.{format}"))
plt.show()

In [ ]:
corr_outliers_by_cell_type_and_cytokine[~strong_impact_by_cell_type] = np.nan
mean = corr_outliers_by_cell_type_and_cytokine.drop("HSPC", axis=1).mean(axis=1)
std = corr_outliers_by_cell_type_and_cytokine.drop("HSPC", axis=1).std(axis=1)

plt.figure(figsize=cm_to_inches((18.38, 3)))

plt.bar(mean.index, mean, color="black")
plt.errorbar(mean.index, mean, yerr=std, color="grey", fmt='none')

plt.xticks(rotation=90, fontsize=5.5)
plt.ylabel("outlier donors", labelpad=15)
plt.xlim(-1, len(CYTOKINE_ORDER))
plt.ylim(0, 6)
apply_greek_labels_heatmap(plt.gca())
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(current_plot_folder, f"stacked_barplot_donor_outliers_mean.{format}"))
plt.show()

In [ ]:
mean = corr_outliers_by_cell_type_and_cytokine.mean(axis=1)
std = corr_outliers_by_cell_type_and_cytokine.std(axis=1)

mean = mean.reindex(CYTOKINE_ORDER)
mean = mean[mean > 0]
std = std.reindex(mean.index)

# --- Parameters ---
n_panels = 5
n = len(mean)
chunk = int(np.ceil(n / n_panels))

# Split into chunks
mean_chunks = [mean.iloc[i*chunk:(i+1)*chunk] for i in range(n_panels)]
std_chunks  = [std.iloc[i*chunk:(i+1)*chunk]  for i in range(n_panels)]

# Reverse all but the last panel for readability; keep the last as-is
for i in range(n_panels):
    mean_chunks[i] = mean_chunks[i].iloc[::-1]
    std_chunks[i]  = std_chunks[i].iloc[::-1]

panels = list(zip(mean_chunks, std_chunks))

# --- Plot ---
fig, axes = plt.subplots(1, n_panels, figsize=cm_to_inches((9, 3)), sharex=True)
axes = axes.flatten()

xmax = max(mean.values) + 0.1 * (max(mean.values) - min(mean.values))
ymax = max(len(m) for m, _ in panels)  # ensure equal y scaling

for i, (m, s) in enumerate(panels):
    ax = axes[i]

    # Calculate top padding (for top alignment)
    pad = ymax - len(m)
    y_pos = np.arange(pad, pad + len(m))

    # Draw bars and errors
    ax.barh(y_pos, m.values, color="black", height=0.6)
    ax.errorbar(m.values, y_pos, xerr=s.values, color="grey", fmt='none')

    # Add yticks and labels (blank for padding)
    ax.set_yticks(np.arange(ymax))
    tick_labels = [""] * pad + list(m.index)
    ax.set_yticklabels(tick_labels, fontsize=5.5)

    if i == 2:
        ax.set_xlabel("number of outlier donors averaged across cell types", labelpad=5)
    
    ax.set_ylim(-0.5, ymax - 0.5)  # same height, no inversion
    
    apply_greek_labels_heatmap(ax)
    ax.set_xlim(0, 6)
    ax.tick_params(axis='both', length=0, rotation=0)

plt.subplots_adjust(wspace=1.5)
for fmt in ["png", "svg"]:
    plt.savefig(os.path.join(current_plot_folder, f"barplot_donor_outliers_split{n_panels}.{fmt}"),
                bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=cm_to_inches((6, 1.75)))
plt.bar(DONOR_NAMES, corr_outliers_by_cell_type_and_donor.sum(axis=1)[DONOR_NAMES], color="black", width=0.8)
plt.xlim(-0.5, len(DONOR_NAMES)-0.5)
plt.ylim(0, 350)
plt.ylabel("total outliers\nby donor")
plt.tick_params(axis='x', length=0, rotation=90)
plt.tight_layout()
for format in ['png', 'svg']:
    plt.savefig(os.path.join(current_plot_folder, f"corr_outliers_by_donor_x_cell_type_barplot.{format}"), dpi=300)

In [ ]:
plt.figure(figsize=cm_to_inches((6, 1.75)))
bottom = np.zeros(len(DONOR_NAMES))
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    values = corr_outliers_by_cell_type_and_donor[cell_type][DONOR_NAMES].values
    plt.bar(DONOR_NAMES, values, bottom=bottom, color=CELL_TYPE_COLORS[cell_type], width=0.8, label=cell_type)
    bottom += values

plt.xlim(-0.5, len(DONOR_NAMES)-0.5)
plt.ylim(0, 350)
plt.ylabel("total outliers\nby donor")
plt.tick_params(axis='x', length=0, rotation=90)
plt.tight_layout()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=6)
for format in ['png', 'svg']:
    plt.savefig(os.path.join(current_plot_folder, f"corr_outliers_by_donor_x_cell_type_stacked_barplot.{format}"), dpi=300)

### Get correlation plots between different donors

In [ ]:
def get_donor_fc_vals_corr(cytokine, cell_type, log2=True):
    """Get the correlation matrix of donor raw FC values for a given cytokine and cell type."""
    vals = get_donor_vals_ratios(cytokine, cell_type)
    if log2:
        vals = np.log2(vals)
    relevant_donor_names = [donor for donor in DONOR_NAMES if donor in vals.columns]
    vals_corr = vals.corr().loc[relevant_donor_names, relevant_donor_names]
    return vals_corr

def get_donor_fc_scatterplot(
        cytokine1,
        cytokine2,
        cell_type,
        donor1,
        donor2,
        log2=True,
        display_counts=False,
        display_r=True,
        savepath=None,
        display_plot=True,
        fixed_limits=None,
        small_plot=False
    ):
    """Get a scatterplot comparing raw FC values between two donors and two cytokines for a given cell type."""
    vals1 = get_donor_vals_ratios(cytokine1, cell_type)
    vals2 = get_donor_vals_ratios(cytokine2, cell_type)
    x = vals1[donor1].copy().dropna()
    y = vals2[donor2].copy().dropna()
    if log2:
        x = np.log2(x)
        y = np.log2(y)
    x,y = get_common_index(x,y)
    r = np.corrcoef(x, y)[0, 1]
    rho = stats.spearmanr(x, y)[0]
    
    if not fixed_limits:
        min_x = min(x.min(), y.min()) - 0.5
        max_x = max(x.max(), y.max()) + 0.5
    else:
        min_x = fixed_limits[0]
        max_x = fixed_limits[1]

    if small_plot:
        size = (2.6, 2.6)
    else:
        size = (3.5, 3.5)
    plt.figure(figsize=cm_to_inches(size))
    plt.scatter(x, y, s=1, edgecolors="none", rasterized=True, alpha=1)
    if not small_plot:
        plt.plot([min_x, max_x], [min_x, max_x], color="black", linestyle="--")

    if display_counts:
        count_x_cond = all_counts_by_condition.loc[(cell_type, cytokine1, donor1)]
        count_x_pbs = all_counts_by_condition.loc[(cell_type, "PBS", donor1)]
        count_y_cond = all_counts_by_condition.loc[(cell_type, cytokine2, donor2)]
        count_y_pbs = all_counts_by_condition.loc[(cell_type, "PBS", donor2)]
        # Add text annotations for counts
        plt.text(0.05, 0.95, f"{donor1}\n{cytokine1} {count_x_cond:.0f}\n(PBS/6) {count_x_pbs/6:.0f}", 
                transform=plt.gca().transAxes, fontsize=6, verticalalignment='top')
        plt.text(0.95, 0.05, f"{donor2}\n{cytokine2} {count_y_cond:.0f}\n(PBS/6) {count_y_pbs/6:.0f}", 
                transform=plt.gca().transAxes, fontsize=6, verticalalignment='bottom', 
                horizontalalignment='right') 

    if display_r:
        plt.text(0.05, 0.95, f"r = {r:.2f}", transform=plt.gca().transAxes, fontsize=6, verticalalignment='top')
        #plt.text(0.05, 0.80, f"ρ = {rho:.2f}", transform=plt.gca().transAxes, fontsize=6, verticalalignment='top')

    plt.xlim(min_x, max_x)
    plt.ylim(min_x, max_x)
    plt.xticks([min_x, 0, max_x])
    plt.yticks([min_x, 0, max_x])
    
    plt.tick_params(axis='x', length=0)
    plt.tick_params(axis='y', length=0)

    if cytokine1 == cytokine2:
        if not small_plot:
            plt.title(introduce_greek_letters(f"{cell_type}: {cytokine1}"), fontsize=7)
        plt.xlabel(f"log2FC ({donor1})")
        plt.ylabel(f"log2FC ({donor2})")
    else:
        if not small_plot:
            plt.title(f"{cell_type}", fontsize=7)
        plt.xlabel(introduce_greek_letters(f"log2FC ({cytokine1}, {donor1})"))
        plt.ylabel(introduce_greek_letters(f"log2FC ({cytokine2}, {donor2})"))

    plt.tight_layout()
    if savepath is not None:
        for format in [".png", ".svg"]:
            plt.savefig(savepath+format, dpi=300)
    if display_plot:
        plt.show()
    plt.close()
    

def get_donor_fc_scatterplot_to_edgeR(
        cytokine_donor,
        cytokine_edger,
        cell_type,
        donor,
        log2=True,
        display_r=True,
        savepath=None,
        display_plot=True,
        fixed_limits=None,
        small_plot=False,
        ax = None,
    ): 
    """Get a scatterplot comparing raw FC values for a given donor and cytokine to edgeR log2FC values for a given cytokine and cell type."""
    x = get_donor_vals_ratios(cytokine_donor, cell_type)[donor].copy().dropna()
    y = edger_df_log2FoldChange[cell_type][cytokine_edger].copy().dropna()
    if log2:
        x = np.log2(x)
    else:
        y = 2**y
        
    x,y = get_common_index(x,y)
    r = np.corrcoef(x, y)[0, 1]
    rho = stats.spearmanr(x, y)[0]
    
    if not fixed_limits:
        min_x = min(x.min(), y.min()) - 0.5
        max_x = max(x.max(), y.max()) + 0.5
    else:
        min_x = fixed_limits[0]
        max_x = fixed_limits[1]

    if small_plot:
        size = (2.6, 2.6)
    else:
        size = (3.5, 3.5)
        
    if ax is None:
        output_flag = True
        fig, ax = plt.subplots(figsize=cm_to_inches(size))
    else:
        output_flag = False
    ax.scatter(x, y, s=1, edgecolors="none", rasterized=True, alpha=1)
    if not small_plot:
        ax.plot([min_x, max_x], [min_x, max_x], color="black", linestyle="--")

    if display_r:
        ax.text(0.05, 0.95, f"r = {r:.2f}", transform=ax.transAxes, fontsize=6, verticalalignment='top')

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_x, max_x)
    ax.set_xticks([min_x, 0, max_x])
    ax.set_yticks([min_x, 0, max_x])
    
    ax.tick_params(axis='x', length=0)
    ax.tick_params(axis='y', length=0)

    if cytokine_donor == cytokine_edger:
        if not small_plot:
            ax.set_title(introduce_greek_letters(f"{cell_type}: {cytokine_donor}"), fontsize=7)
        ax.set_xlabel(f"log2FC ({donor})")
        ax.set_ylabel(f"consensus log2FC")
    else:
        if not small_plot:
            ax.set_title(f"{cell_type}", fontsize=7)
        ax.set_xlabel(introduce_greek_letters(f"log2FC ({cytokine_donor}, {donor})"))
        ax.set_ylabel(introduce_greek_letters(f"consensus log2FC ({cytokine_edger})"))

    if output_flag:
        plt.tight_layout()
        if savepath is not None:
            for format in [".png", ".svg"]:
                plt.savefig(savepath+format, dpi=300)
        if display_plot:
            plt.show()
        plt.close()
    
 
    
cell_type = "CD4 T cell"
cytokine1 = "IL-2"
cytokine2 = "IL-2"
donor1 = "D3"
donor2 = "D1"
get_donor_fc_scatterplot(cytokine1, cytokine2, cell_type, donor1, donor2, small_plot=True, fixed_limits=[-4, 4])

# cell_type = "CD4 T cell"
# cytokine_donor = "IL-32-beta"
# cytokine_edger = "IL-32-beta"
# donor = "D3"
# get_donor_fc_scatterplot_to_edgeR(cytokine_donor, cytokine_edger, cell_type, donor, small_plot=True, fixed_limits=[-4, 4])

In [ ]:
def get_donor_corr_heatmap(cytokine, cell_type, log2=True,
                     del_colorbar=False, annot_heatmap=True,
                     communities=None,
                     savepath=None, display_plot=True, hide_ylabel=False):
    """Get a heatmap of donor correlations for a given cytokine and cell type, with barplots of response magnitude, raw response strength, and edger correlation outliers."""
    # get values for the heatmap
    vals = get_donor_vals_ratios(cytokine, cell_type)
    if log2:
        vals = np.log2(vals)
    if communities is not None:
        comms_flat = [i for x in communities for i in x]
        relevant_donor_names = comms_flat
    else:
        relevant_donor_names = [donor for donor in DONOR_NAMES if donor in vals.columns]
    vals_corr = vals.corr().loc[relevant_donor_names, relevant_donor_names]
    
    # get values for the top barplot
    response_magnitudes = all_normalized_euclidean_distances_by_donor.loc[cytokine, (cell_type, relevant_donor_names)].values
    
    # get values for the second barplot
    abs_sums = all_abs_sums_by_donor.loc[cytokine, (cell_type, relevant_donor_names)].values
    abs_sums = abs_sums / abs_sums.max()
    
    # values for bottom barplot are calculated below
    
    # number of data points
    n = len(relevant_donor_names)

    fig = plt.figure(figsize=cm_to_inches((6, 7)), constrained_layout=False)
    gs  = fig.add_gridspec(4, 2, height_ratios=[1, 1, 4, 1], width_ratios=[1, 0.05], hspace=0.1, wspace=0)

    ax_bar  = fig.add_subplot(gs[0, 0])
    ax_bar2 = fig.add_subplot(gs[1, 0])
    ax_heat = fig.add_subplot(gs[2, 0])
    cax     = fig.add_subplot(gs[2, 1])
    ax_bar3 = fig.add_subplot(gs[3, 0])

    # --- Heatmap ---
    hm = sns.heatmap(
        vals_corr,
        ax=ax_heat,
        annot=annot_heatmap,
        cmap="coolwarm",
        fmt=".2f",
        square=True,
        cbar=True,
        cbar_ax=cax,
        vmin=-1,
        vmax=1,
        cbar_kws={"label": "Pearson r of donorlog2FC", },
        annot_kws={"size": 5.5}
    )
    
    # Draw black rectangles around communities
    if communities is not None:
        for community in communities:
            # Find the indices of this community in the current plot
            community_indices = [relevant_donor_names.index(donor) for donor in community if donor in relevant_donor_names]
            min_idx = min(community_indices)
            max_idx = max(community_indices)
            # Draw rectangle around the community block
            rect = plt.Rectangle((min_idx, min_idx), max_idx - min_idx + 1, max_idx - min_idx + 1, 
                                fill=False, edgecolor='black', linewidth=2, clip_on=False)
            ax_heat.add_patch(rect)

    #ax_heat.set_xticklabels(ax_heat.get_xticklabels(), rotation=90, ha='center')
    ax_heat.set_xticklabels([])
    ax_heat.tick_params(axis='both', which='both', length=0)
    ax_heat.tick_params(axis='x', pad=2)
    ax_heat.tick_params(axis='y', pad=2)
    ax_heat.set_ylabel("")
    ax_heat.set_xlabel("")
    
    if communities is None:
        plot_overview_labels_along_y_axis(fig, ax_heat,
                                        ["Male", "Female"],
                                        {"Female": 6, "Male": 6},
                                        x_offset=-2.5,
                                        no_y_vals=12,
                                        fontsize=6, fontweight="bold", color="black", linewidth=1,
                                        rotation=90)

    # --- Bars ---
    bar_centers = np.arange(n) + 0.5
    ax_bar.bar(
        bar_centers,
        response_magnitudes,
        width=0.75,
        color="darkgray",
        align="center",
        edgecolor="black",
        linewidth=1
    )
    ax_bar.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    ax_bar.tick_params(axis='y', which='both', bottom=False, top=False, labelbottom=False, length=0)
    if hide_ylabel:
        ax_bar.set_ylabel("")
    else:
        ax_bar.set_ylabel('Relative\nResponse\nMagnitude')
    ax_bar.set_ylim(0, 1)
    ax_bar.set_yticks([0, 0.7])
    ax_bar.set_xlim(0, n)
    
    bar_centers = np.arange(n) + 0.5
    ax_bar2.bar(
        bar_centers,
        abs_sums,
        width=0.75,
        color="lightsalmon",
        align="center",
        edgecolor="black",
        linewidth=1
    )
    ax_bar2.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)
    ax_bar2.tick_params(axis='y', which='both', bottom=False, top=False, labelbottom=False, length=0)
    if hide_ylabel:
        ax_bar2.set_ylabel("")
    else:   
        ax_bar2.set_ylabel('Norm. Raw\nResponse\nStrength')
    ax_bar2.set_ylim(0, 1)
    ax_bar2.set_yticks([0, 0.7])
    ax_bar2.set_xlim(0, n)

    # clean spines
    for ax in (ax_bar, ax_bar2, ax_heat):
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    # bottom barplot
    vals = donor_corrs_by_cell_type[cell_type].loc[cytokine]
    vals = vals[relevant_donor_names]
    median = vals.sort_values(ascending=False).head(8).median()
    non_outlier_range = width_factor_donor_outliers * max(median, 0.6)
    ax_bar3.axhspan(median - non_outlier_range, median + non_outlier_range, alpha=0.3, facecolor='gray', edgecolor="none", linewidth=0)
    x0, x1 = -0.5, len(vals)-0.5
    ax_bar3.hlines([median - non_outlier_range, median + non_outlier_range], x0, x1, colors='black', linewidth=0.5)
    
    ax_bar3.bar(vals.index, vals, edgecolor="black", color="skyblue",width=0.7, zorder=3)
    min_y = 0 #max(min(vals.min(), 0.5)-0.1,0)
    max_y = 1
    ax_bar3.set_xlim(-0.5, n-0.5)
    ax_bar3.set_ylim(min_y, max_y)
    ax_bar3.set_xticks(range(len(vals)))
    ax_bar3.set_xticklabels(vals.index, rotation=90)
    
    ax_bar3.tick_params(axis='y', which='both', bottom=False, top=False, labelbottom=False, length=0)
    
    if hide_ylabel:
        ax_bar3.set_ylabel("")
    else:
        ax_bar3.set_ylabel('Pearson r\nto\noverall\nlog2FC')

    # --- Manual Alignment AFTER Drawing ---
    # First, force a draw of the canvas to finalize the initial layout
    fig.canvas.draw()

    # Get the final positions of the axes in figure coordinates (values from 0 to 1)
    pos_heat = ax_heat.get_position()
    pos_bar = ax_bar.get_position()
    pos_bar2 = ax_bar2.get_position()
    pos_bar3 = ax_bar3.get_position()

    # Set the bar plot's position to match the heatmap's horizontal extent.
    # The format is [left, bottom, width, height].
    ax_bar.set_position([pos_heat.x0, pos_bar.y0, pos_heat.width, pos_bar.height])
    ax_bar2.set_position([pos_heat.x0, pos_bar2.y0, pos_heat.width, pos_bar2.height])
    ax_bar3.set_position([pos_heat.x0, pos_bar3.y0, pos_heat.width, pos_bar3.height])
    
    # Set overall figure title
    fig.suptitle(f'{introduce_greek_letters(cytokine)} on {cell_type}', fontsize=7, y=0.95)
    if del_colorbar:
        cax.set_visible(False)
    
    if savepath is not None:
        for format in ["png", "svg"]:
            plt.savefig(os.path.join(savepath, f"corr_heatmap_{cytokine}_{cell_type}.{format}"), dpi=300, bbox_inches="tight")

    if display_plot:
        plt.show()
    plt.close()

get_donor_corr_heatmap("IL-2", "NK CD56low", annot_heatmap=False, del_colorbar=True, hide_ylabel=True)

In [ ]:
cytokine = "IL-10"
cell_type = "CD4 T cell"
donor_combs = [
    ("D1", "D3"),
    ("D1", "D7"),
    ("D7", "D9")
]
for donor_comb in donor_combs:
    get_donor_fc_scatterplot(
        cytokine,
        cytokine,
        cell_type,
        donor_comb[0],
        donor_comb[1],
        small_plot=True,
        fixed_limits=[-5,5],
        savepath=os.path.join(current_plot_folder, f"scatter_log2fc_{donor_comb[0]}_{donor_comb[1]}")
    )

In [ ]:
cell_type = "CD4 T cell"

limits_and_labels = {
    "IFN-beta": {"limits": [-5,5], "labels": [-4,0,4]},
    "IL-10": {"limits": [-4,4], "labels": [-3,0,3]},
}

for cytokine in ["IFN-beta", "IL-10"]:
    fig, axes = plt.subplots(1,12,figsize=cm_to_inches((18,2.4)))
    for i, donor in enumerate(DONOR_NAMES):
        ax = axes[i]
        get_donor_fc_scatterplot_to_edgeR(
            cytokine,
            cytokine,
            cell_type,
            donor,
            small_plot=True,
            fixed_limits=limits_and_labels[cytokine]['limits'],
            ax = ax
            )
        ax.set_xticks(limits_and_labels[cytokine]['labels'])
        ax.set_yticks(limits_and_labels[cytokine]['labels'])
        ax.label_outer()
        ax.set_xlabel(donor)
    plt.figtext(0.5, -0.1, f"log2FC donor", ha='center', va='bottom', fontsize=6)    
    plt.figtext(0.5, 0.95, introduce_greek_letters(cytokine), ha='center', va='bottom', fontsize=8)
    plt.tight_layout()
    savepath=os.path.join(current_plot_folder, f"scatter_log2fc_{cytokine}_to_overall")
    plt.savefig(savepath+".svg", dpi=400)  

In [ ]:
cell_type_cytokine_pairs = [
    ("CD4 T cell", "IL-10"),
    ("CD4 T cell", "IFN-beta"),
    ("CD4 T cell", "IL-15"),
    ("NK CD56low", "IL-15"),
    ("CD8 T cell", "VEGF"),
    ("CD14 Mono", "IL-32-beta"),
    ("CD4 T cell", "IL-32-beta"),
    ("B cell", "CD40L")
]

for index, (cell_type, cytokine) in enumerate(cell_type_cytokine_pairs):    
    get_donor_corr_heatmap(
        cytokine,
        cell_type,
        annot_heatmap=False,
        del_colorbar=index!=0,
        hide_ylabel=index!=0,
        savepath=current_plot_folder,
        communities=[donor_group_2, donor_group_1]
        )

In [ ]:
# %%capture output
all_donor_correlation_dfs_across_cell_types = {}
for cell_type in ALL_CELL_TYPES:
    print(cell_type)

    all_correlations = {}
    for cytokine in CYTOKINE_ORDER:
        vals_corr = get_donor_fc_vals_corr(cytokine, cell_type)
        # Store only the upper triangle of the correlation matrix
        n = len(vals_corr)
        upper_triangle_corrs = []
        for i in range(n):
            for j in range(i+1, n):
                upper_triangle_corrs.append(vals_corr.iloc[i, j])
        all_correlations[cytokine] = upper_triangle_corrs

    mean_correlations = {}
    std_correlations = {}
    mean_response_magnitude = {}
    std_response_magnitude = {}
    for cytokine in CYTOKINE_ORDER:
        mean_correlations[cytokine] = np.mean(all_correlations[cytokine])
        std_correlations[cytokine] = np.std(all_correlations[cytokine])
        mean_response_magnitude[cytokine] = np.mean(all_normalized_euclidean_distances_by_donor.loc[cytokine, (cell_type, slice(None))])
        std_response_magnitude[cytokine] = np.std(all_normalized_euclidean_distances_by_donor.loc[cytokine, (cell_type, slice(None))])
    mean_correlations = pd.Series(mean_correlations)
    std_correlations = pd.Series(std_correlations)
    mean_response_magnitude = pd.Series(mean_response_magnitude)
    std_response_magnitude = pd.Series(std_response_magnitude)    
    all_donor_correlation_dfs_across_cell_types[cell_type] = pd.DataFrame({
        "mean_correlations": mean_correlations,
        "std_correlations": std_correlations,
        "mean_response_magnitude": mean_response_magnitude,
        "std_response_magnitude": std_response_magnitude
    })

    # # ----------------------------------------------------
    # plt.figure(figsize=cm_to_inches((4.5, 3.5)))
    # plt.scatter(mean_correlations, mean_response_magnitude, s=5, color='tab:blue')
    # r = np.corrcoef(mean_correlations, mean_response_magnitude)[0, 1]

    # # find outliers
    # slope, intercept, r_value, p_value, std_err = stats.linregress(mean_correlations, mean_response_magnitude)
    # fitted_line = slope * mean_correlations + intercept

    # # Calculate residuals
    # residuals = mean_response_magnitude - fitted_line

    # # Find the strongest outlier (largest absolute residual)
    # strongest_outlier_idx = np.abs(residuals).idxmax()
    # strongest_outlier_name = strongest_outlier_idx

    # # Highlight and label the strongest outlier
    # outlier_x = mean_correlations[strongest_outlier_idx]
    # outlier_y = mean_response_magnitude[strongest_outlier_idx]
    # plt.scatter(outlier_x, outlier_y, color='tab:blue', s=20, zorder=5)
    # plt.annotate(strongest_outlier_name, (outlier_x, outlier_y), 
    #             xytext=(2, 2), textcoords='offset points', 
    #             fontsize=6, ha='left', va='bottom')

    # plt.xlim(0, 1)
    # plt.ylim(0, 1.05)
    # plt.text(0.1, 0.85, f'r = {r:.2f}', fontsize=7)
    # plt.xlabel('Mean Pearson r of donorlog2FC')
    # plt.ylabel('Mean response magnitude')
    # plt.title(cell_type)
    # plt.tight_layout()
    # for format in ["png", "svg"]:
    #     plt.savefig(os.path.join(current_plot_folder, f"2.4_mean_correlations_vs_mean_response_magnitude_{cell_type}.{format}"), dpi=300)
        
    # ----------------------------------------------------
    # plt.figure(figsize=cm_to_inches((4.5, 3.5)))
    # plt.scatter(std_correlations, std_response_magnitude, s=5, color='tab:blue')
    # r = np.corrcoef(std_correlations, std_response_magnitude)[0, 1]

    # # find outliers
    # slope, intercept, r_value, p_value, std_err = stats.linregress(std_correlations, std_response_magnitude)
    # fitted_line = slope * std_correlations + intercept

    # # Calculate residuals
    # residuals = std_response_magnitude - fitted_line

    # # Find the strongest outlier (largest absolute residual)
    # strongest_outlier_idx = np.abs(residuals).idxmax()
    # strongest_outlier_name = strongest_outlier_idx

    # # Plot the fitted line
    # plt.plot(std_correlations, fitted_line, 'r--', alpha=0.7, linewidth=1)

    # # Highlight and label the strongest outlier
    # outlier_x = std_correlations[strongest_outlier_idx]
    # outlier_y = std_response_magnitude[strongest_outlier_idx]
    # plt.scatter(outlier_x, outlier_y, color='tab:blue', s=20, zorder=5)
    # plt.annotate(strongest_outlier_name, (outlier_x, outlier_y), 
    #              xytext=(2, 2), textcoords='offset points', 
    #              fontsize=6, ha='left', va='bottom')

    # plt.xlim(0, 1.1*max(std_correlations))
    # plt.ylim(0, 1.1*max(std_response_magnitude))
    # plt.text(0.1*max(std_correlations), 0.85*max(std_response_magnitude), f'r = {r:.2f}', fontsize=7)
    # plt.xlabel('Mean std of Pearson r of donor-specificlog2FC')
    # plt.ylabel('Std response magnitude')
    # plt.title(cell_type)
    # plt.tight_layout()
    # for format in ["png", "svg"]:
    #     plt.savefig(os.path.join(current_plot_folder, f"2.4_std_correlations_vs_std_response_magnitude_{cell_type}.{format}"), dpi=300)

    # ----------------------------------------------------
    # # Create a list to store data for boxplot
    # boxplot_data = []
    # cytokine_labels = []

    # # Extract data from all_correlations dictionary
    # for cytokine, correlations in all_correlations.items():
    #     strong_impact = strong_impact_by_cell_type.loc[cytokine, cell_type]
    #     if not strong_impact:
    #         continue
    #     boxplot_data.append(correlations)
    #     cytokine_labels.append(cytokine)

    # # Create boxplot
    # plt.figure(figsize=(12, 6))
    # plt.boxplot(boxplot_data, labels=cytokine_labels)
    # plt.xticks(rotation=90, ha='center')
    # plt.ylabel('Correlation Values')
    # plt.xlabel('Cytokines')
    # plt.title('Distribution of Correlations by Cytokine')
    # plt.tight_layout()
    # plt.show()   

In [ ]:
# the stronger the response, the more correlation we see (makes sense)
plt.figure(figsize=cm_to_inches((4, 3.8)))
all_x_vals = []
all_y_vals = []
all_r_vals = []
for index, cell_type in enumerate(ALL_CELL_TYPES):
    x_vals = all_donor_correlation_dfs_across_cell_types[cell_type]["mean_correlations"]
    y_vals = all_donor_correlation_dfs_across_cell_types[cell_type]["mean_response_magnitude"]

    x_vals = x_vals.dropna()
    y_vals = y_vals.dropna()
    x_vals, y_vals = get_common_index(x_vals, y_vals)
    all_x_vals.extend(x_vals)
    all_y_vals.extend(y_vals)

    plt.scatter(x_vals, y_vals, s=5, color=CELL_TYPE_COLORS[cell_type], alpha=0.75, edgecolors="none")

    r = np.corrcoef(x_vals, y_vals)[0, 1]
    all_r_vals.append(r)
    
plt.xlim(0, 0.8)
plt.ylim(0, 1.05)
plt.title(f'mean r = {np.mean(all_r_vals):.2f}', fontsize=7)
plt.xlabel('Mean Pearson r of donorlog2FC')
plt.ylabel('Mean response magnitude')
#plt.legend(fontsize=7, loc="lower right")
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(current_plot_folder, f"2.4_mean_correlations_vs_mean_response_magnitude.{format}"), dpi=300)

## Find interesting sub-structure in donor_responses

In [ ]:
plot_folder_heatmaps = os.path.join(current_plot_folder, "heatmaps_interesting_substructure")
os.makedirs(plot_folder_heatmaps, exist_ok=True)

def get_leiden_communities(cytokine, cell_type, resolution=0.5):
    distance_matrix = get_donor_fc_vals_corr(cytokine, cell_type).astype(float)
    # Get the distance matrix.
    df = distance_matrix.copy()
    # Ensure that the diagonal is zero (i.e., no self-loops)
    np.fill_diagonal(df.values, 0)
    all_communities, _ = leiden_clustering(distance_matrix, resolution, False, False)
    return all_communities

# this is the margin that we want between within vs across community correlations
margin_threshold_for_substructure = 0.3

# Check if all_community_means exists in locals
if not os.path.exists(os.path.join(intermediate_output_folder, "donor_analysis_all_community_means.pkl")):
    all_community_means = {}
    all_community_stds = {}
    all_community_members = {}
    all_r_vals_to_other_cytokines_by_comm = {}
    all_r_vals_to_other_cytokines_by_donor = {}
    for cytokine_idx, cytokine in enumerate(CYTOKINE_ORDER):
        print(f"Process cytokine {cytokine_idx+1}/{len(CYTOKINE_ORDER)}: {cytokine}")
        strong_impact = strong_impact_by_cell_type.loc[cytokine, ALL_CELL_TYPES]
        strong_impact = strong_impact[strong_impact].index
        for cell_type in strong_impact:
            print(cell_type)
            
            if f"{cytokine}_{cell_type}" in all_community_means: # what was I dong here? avoiding running this twice?
                continue

            resolution_vals = [0.3, 0.5, 0.7]
            success = False
            for resolution_val in resolution_vals:
                print("Attempting: ", resolution_val)
                try:
                    comms = get_leiden_communities(cytokine, cell_type, resolution_val)
                except BaseException as e:
                    print(f"Error processing cytokine {cytokine} for cell type {cell_type}: {e}")
                    continue
                
                # only continue for potentially interesting communities
                comms_pass = [comm for comm in comms if len(comm) > 1]
                if len(comms_pass) < 2:
                    print("No second comunity of size 2 or more.")
                    continue
                
                comms_size = [comm for comm in comms if len(comm) > 3]
                if len(comms_size) == 0:
                    print("No primary community of size 4 or more.")
                    continue
    
                # ------------------------------------------------------------------
                # look into within vs across community ratio means
                vals_corr = delete_diagonal(get_donor_fc_vals_corr(cytokine, cell_type))
                corr_comms = {}
                for comm_comb in itertools.product(comms_pass, comms_pass):
                    name1 = "_".join(comm_comb[0])
                    name2 = "_".join(comm_comb[1])
                    corr = vals_corr.loc[comm_comb[0], comm_comb[1]]
                    mean = np.nanmean(corr.values.flatten())
                    corr_comms[(name1, name2)] = mean

                # Convert to DataFrame
                comm_names = ["_".join(comm) for comm in comms_pass]
                corr_comms_df = pd.DataFrame(index=comm_names, columns=comm_names)
                for (name1, name2), value in corr_comms.items():
                    corr_comms_df.loc[name1, name2] = value
                    
                # get the margin
                margin_to_self = corr_comms_df - np.diag(corr_comms_df)
                current_margin_threshold = max(0.2, margin_threshold_for_substructure * corr_comms_df.max().max())

                interesting_communities = (margin_to_self < -margin_threshold_for_substructure).any(axis=0)
                interesting_communities = interesting_communities[interesting_communities].index

                # continue if the communities don't look interesting
                if len(interesting_communities) < 2:
                    print("Insufficient correlation substructure between communities.")
                    continue
                
                success = True
                final_res_val = resolution_val
                break
            if not success:
                continue
            print("Succeeded with: ", final_res_val)
            
            # ------------------------------------------------------------------
            # get ratio vals --------------------------------------------------
            raw_ratio_vals = np.log2(get_donor_vals_ratios(cytokine, cell_type))
            mean_by_community = {}
            std_by_community = {}
            community_members = {}
            for i, comm in enumerate(comms_pass):
                name = "_".join(comm)
                community_members[name] = comm
                comm_vals = raw_ratio_vals[comm]
                mean_by_community[name] = comm_vals.mean(axis=1, skipna=True) # get mean and std of ratio vals
                std_by_community[name] = comm_vals.std(axis=1, skipna=True)
            
            # see which other cytokines are most similar to the cytokine of interest (by community)
            r_vals_to_other_cytokines = {}
            for comm, cytokine_curr in itertools.product(mean_by_community.keys(), CYTOKINE_ORDER):
                mean_by_community_donor = mean_by_community[comm].dropna()
                edger_lfc = edger_df_log2FoldChange[(cell_type, cytokine_curr)].dropna()
                mean_by_community_donor, edger_lfc = get_common_index(mean_by_community_donor, edger_lfc)
                r = np.corrcoef(mean_by_community_donor, edger_lfc)[0, 1]
                r_vals_to_other_cytokines[(comm, cytokine_curr)] = r
            r_vals_to_other_cytokines = pd.Series(r_vals_to_other_cytokines)
            all_community_means[f"{cytokine}_{cell_type}"] = mean_by_community
            all_community_stds[f"{cytokine}_{cell_type}"] = std_by_community
            all_community_members[f"{cytokine}_{cell_type}"] = community_members
            all_r_vals_to_other_cytokines_by_comm[f"{cytokine}_{cell_type}"] = r_vals_to_other_cytokines
            
            # see which other cytokines are most similar to the cytokine of interest (by donor)
            r_vals_to_other_cytokines = {}
            for comm in mean_by_community.keys():
                for donor in DONOR_NAMES:
                    for cytokine_curr in CYTOKINE_ORDER:
                        if not donor in raw_ratio_vals:
                            continue
                        fc_vals = raw_ratio_vals[donor].dropna()
                        edger_lfc = edger_df_log2FoldChange[(cell_type, cytokine_curr)].dropna()
                        fc_vals, edger_lfc = get_common_index(fc_vals, edger_lfc)
                        r = np.corrcoef(fc_vals, edger_lfc)[0, 1]
                        r_vals_to_other_cytokines[(comm, cytokine_curr)] = r
            all_r_vals_to_other_cytokines_by_donor[f"{cytokine}_{cell_type}"] = r_vals_to_other_cytokines
            
            get_donor_corr_heatmap(cytokine, cell_type, log2=True,
                                del_colorbar=True, annot_heatmap=False, communities=comms,
                                savepath=plot_folder_heatmaps, display_plot=False)
            
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_means.pkl"), "wb") as f:
        pickle.dump(all_community_means, f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_stds.pkl"), "wb") as f:
        pickle.dump(all_community_stds, f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_members.pkl"), "wb") as f:
        pickle.dump(all_community_members, f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_r_vals_to_other_cytokines_by_comm.pkl"), "wb") as f:
        pickle.dump(all_r_vals_to_other_cytokines_by_comm, f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_r_vals_to_other_cytokines_by_donor.pkl"), "wb") as f:
        pickle.dump(all_r_vals_to_other_cytokines_by_donor, f)
else:
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_means.pkl"), "rb") as f:
        all_community_means = pickle.load(f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_stds.pkl"), "rb") as f:
        all_community_stds = pickle.load(f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_community_members.pkl"), "rb") as f:
        all_community_members = pickle.load(f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_r_vals_to_other_cytokines_by_comm.pkl"), "rb") as f:
        all_r_vals_to_other_cytokines_by_comm = pickle.load(f)
    with open(os.path.join(intermediate_output_folder, "donor_analysis_all_r_vals_to_other_cytokines_by_donor.pkl"), "rb") as f:
        all_r_vals_to_other_cytokines_by_donor = pickle.load(f)

In [ ]:
internal_by_combo = {}
cross_by_combo = {}
minority_comm_by_combo = {}
majority_comm_by_combo = {}
for cytokine, cell_type in [key.split("_") for key in all_community_members.keys()]:
    # if not cytokine == "IL-32-beta" or cell_type != "CD4 T cell":
    #     continue
    # print(cytokine, cell_type)
    # print(all_community_members[f"{cytokine}_{cell_type}"])

    comms_pass = [comm for comm in all_community_members[f"{cytokine}_{cell_type}"].values() if len(comm) > 1]

    # look into within vs across community ratio means
    vals_corr = delete_diagonal(get_donor_fc_vals_corr(cytokine, cell_type))
    corr_comms = {}
    corr_comms_std = {}
    for comm_comb in itertools.product(comms_pass, comms_pass):
        name1 = "_".join(comm_comb[0])
        name2 = "_".join(comm_comb[1])
        corr = vals_corr.loc[comm_comb[0], comm_comb[1]]
        mean = np.nanmean(corr.values.flatten())
        corr_comms[(name1, name2)] = mean
        std = np.nanstd(corr.values.flatten())
        corr_comms_std[(name1, name2)] = std

    # Convert to DataFrame
    comm_names = ["_".join(comm) for comm in comms_pass]
    corr_comms_df = pd.DataFrame(index=comm_names, columns=comm_names)
    corr_comms_df_std = pd.DataFrame(index=comm_names, columns=comm_names)
    for (name1, name2), value in corr_comms.items():
        corr_comms_df.loc[name1, name2] = value
    for (name1, name2), value in corr_comms_std.items():
        corr_comms_df_std.loc[name1, name2] = value
    
    # get the margin
    margin_to_self = corr_comms_df - np.diag(corr_comms_df)
    current_margin_threshold = max(0.2, margin_threshold_for_substructure * corr_comms_df.max().max())
    
    interesting_communities = (margin_to_self < -margin_threshold_for_substructure).any(axis=0)
    interesting_communities = interesting_communities[interesting_communities].index
    
    # Find both indices (row and column) of the maximum margin
    max_margin_row, max_margin_col = margin_to_self.stack().idxmin()
    #print("Max margin between communities:", max_margin_row, max_margin_col)
    
    internal_corr_1 = corr_comms_df.loc[max_margin_row, max_margin_row]
    internal_corr_2 = corr_comms_df.loc[max_margin_col, max_margin_col]
    internal_min = min(internal_corr_1, internal_corr_2)
    cross_corr = corr_comms_df.loc[max_margin_row, max_margin_col]

    internal_by_combo[(cytokine, cell_type)] = internal_min
    cross_by_combo[(cytokine, cell_type)] = cross_corr
    
    len_1 = max_margin_row.count("_") + 1
    len_2 = max_margin_col.count("_") + 1
    smaller_comm = max_margin_row if len_1 < len_2 else max_margin_col
    larger_comm = max_margin_col if len_1 < len_2 else max_margin_row
    minority_comm_by_combo[(cytokine, cell_type)] = smaller_comm
    majority_comm_by_combo[(cytokine, cell_type)] = larger_comm

In [ ]:
counts_minority_by_donor = pd.Series(0, index=DONOR_NAMES)
for combo in minority_comm_by_combo.keys():
    donors = minority_comm_by_combo[combo].split("_")
    counts_minority_by_donor[donors] += 1
counts_minority_by_donor = counts_minority_by_donor.iloc[::-1]

plt.figure(figsize=cm_to_inches((3.5, 5)))
counts_minority_by_donor.plot(kind="barh", color="salmon", edgecolor="black", width=0.7)
plt.xlabel("Count in smaller community\nfor substantial substructure")
plt.tight_layout()
plt.savefig(os.path.join(current_plot_folder, f"2.4_minority_community_counts_by_donor.svg"), dpi=300)

counts_minority_by_donor_and_cytokine = pd.DataFrame(0, index=DONOR_NAMES, columns=CYTOKINE_ORDER)
for combo in minority_comm_by_combo.keys():
    donors = minority_comm_by_combo[combo].split("_")
    counts_minority_by_donor_and_cytokine.loc[donors, combo[0]] += 1
counts_minority_by_donor_and_cytokine = counts_minority_by_donor_and_cytokine.loc[:, counts_minority_by_donor_and_cytokine.sum(axis=0) > 0]
counts_minority_by_donor_and_cytokine = counts_minority_by_donor_and_cytokine.iloc[::-1]

plt.figure(figsize=cm_to_inches((3.5, 5)))
bottom = np.zeros(len(DONOR_NAMES))
colormap = plt.get_cmap("tab10").colors
for i, cytokine in enumerate(counts_minority_by_donor_and_cytokine.columns):
    values = counts_minority_by_donor_and_cytokine[cytokine].values
    plt.barh(counts_minority_by_donor_and_cytokine.index, values, left=bottom,
             label=introduce_greek_letters(cytokine), color=colormap[i])
    bottom += values
plt.xlabel("Count in smaller community\nfor substantial substructure")
plt.tight_layout()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=6, ncol=2)
plt.savefig(os.path.join(current_plot_folder, f"2.4_minority_community_counts_by_donor_stacked.svg"), dpi=300)

In [ ]:
cross_by_combo = pd.Series(cross_by_combo)
internal_by_combo = pd.Series(internal_by_combo)
overall_by_combo = pd.DataFrame({
    "cross": cross_by_combo,
    "internal": internal_by_combo
})
overall_by_combo['diff'] = overall_by_combo['internal'] - overall_by_combo['cross']
overall_by_combo['label'] = overall_by_combo.index.map(lambda x: f"{x[0]}_{x[1]}")

In [ ]:
# --- Prepare adjusted y positions with gaps between cytokine groups ---
cytokines = [l.split('_')[0] for l in overall_by_combo['label']]
cell_types = [l.split('_')[1] for l in overall_by_combo['label']]

gap = 0.6  # controls spacing between cytokine groups
y_positions = []
extra_space = 0
for i, cyto in enumerate(cytokines):
    if i > 0 and cyto != cytokines[i - 1]:
        extra_space += gap
    y_positions.append(i + extra_space)

plt.figure(figsize=cm_to_inches((6, 8)))
# Plot bars at adjusted y positions
plt.barh(y_positions, overall_by_combo['internal'], color="lightseagreen", edgecolor="black", label="min within groups")
plt.barh(y_positions, overall_by_combo['cross'], color="coral", edgecolor="black", label="between groups")

# --- Custom y-axis labeling ---
plt.yticks(y_positions, cell_types)

ax = plt.gca()

# Compute group centers for cytokine labels
unique_cytokines = []
group_centers = []
for i, cyto in enumerate(cytokines):
    if i == 0 or cyto != cytokines[i - 1]:
        unique_cytokines.append(cyto)
        start = i
        end = i
        while end + 1 < len(cytokines) and cytokines[end + 1] == cyto:
            end += 1
        start_y = y_positions[start]
        end_y = y_positions[end]
        group_centers.append((start_y + end_y) / 2)

# Draw cytokine names (rotated, centered)
for i, (cyto, center) in enumerate(zip(unique_cytokines, group_centers)):
    ax.text(-0.35-0.1*(i%2), center, introduce_greek_letters(cyto),
            ha='right', va='center', rotation=90, fontsize=6,
            transform=ax.get_yaxis_transform())
plt.legend(fontsize=7, loc="lower right")
plt.xlabel("Pearson r")
plt.savefig(os.path.join(current_plot_folder, f"2.4_internal_vs_cross_community_correlation.svg"), dpi=300, bbox_inches="tight")

In [ ]:
# a couple more examples of scatterplots
cytokine = "IL-1-alpha"
cell_type = "CD4 T cell"
donor_combs = [
    ("D5", "D6"),
    ("D5", "D4"),
    ("D3", "D4")
]
for donor_comb in donor_combs:
    get_donor_fc_scatterplot(
        cytokine,
        cytokine,
        cell_type,
        donor_comb[0],
        donor_comb[1],
        small_plot=True,
        fixed_limits=[-4,4],
        savepath=os.path.join(plot_folder_heatmaps, f"scatter_log2fc_{donor_comb[0]}_{donor_comb[1]}")
    )

In [ ]:
interesting_substructure = pd.DataFrame(index=CYTOKINE_ORDER, columns=ALL_CELL_TYPES, data=0)
for key in all_community_means.keys():
    cytokine, cell_type = key.split("_")
    interesting_substructure.loc[cytokine, cell_type] += 1
interesting_substructure = interesting_substructure.loc[:, interesting_substructure.sum(axis=0) > 0]
interesting_substructure = interesting_substructure[interesting_substructure.sum(axis=1) > 0]

plt.figure(figsize=cm_to_inches((4.25, 3.65)))
for i, cytokine in enumerate(interesting_substructure.index):
    for j, cell_type in enumerate(interesting_substructure.columns[::-1]):
        count = interesting_substructure.loc[cytokine, cell_type]
        if count > 0:
            plt.scatter(i, j, s=count*10, alpha=1, c=CELL_TYPE_COLORS[cell_type], edgecolors=None)

plt.yticks(range(len(interesting_substructure.columns)), interesting_substructure.columns[::-1], rotation=0)
plt.xticks(range(len(interesting_substructure.index)), interesting_substructure.index, rotation=90)
apply_greek_labels(plt.gca())
plt.xlim(-1, len(interesting_substructure.index))
plt.ylim(-1, len(interesting_substructure.columns))
plt.tick_params(length=0)
plt.title("substantial response sub-structure", fontsize=6.5)
plt.tight_layout()
for format in ['png', 'svg']:
    plt.savefig(os.path.join(current_plot_folder, f"interesting_substructure_counts.{format}"), dpi=300)

In [ ]:
def format_community_name(name):
    donors = name.split("_")
    text = ""
    
    # Determine block dimensions based on number of donors
    num_donors = len(donors)
    if num_donors <= 4:
        cols = 2
    elif num_donors <= 9:
        cols = 3
    else:
        cols = 4
    
    for i, donor in enumerate(donors):
        text += donor + " "
        # Add newline when reaching end of row (but not on last item)
        if (i + 1) % cols == 0 and i != num_donors - 1:
            text += "\n"
    
    return text

for key in all_community_means.keys():
    cytokine, cell_type = key.split("_")
    minor_community = minority_comm_by_combo[(cytokine, cell_type)]
    major_community = majority_comm_by_combo[(cytokine, cell_type)]
    df = all_r_vals_to_other_cytokines_by_comm[f'{cytokine}_{cell_type}'].unstack()
    df = df.loc[[major_community, minor_community], :]
    all_cytokines_to_display = [cytokine]
    for i in range(len(df.index)):
        subset = abs(df).iloc[i].sort_values(ascending=False)
        subset = subset[~subset.index.str.contains(cytokine)]
        subset = subset[subset>0.3]
        cytokines_to_display = [c for c in subset.head(3).index if c not in all_cytokines_to_display]
        all_cytokines_to_display.extend(cytokines_to_display)
    plt.figure(figsize=cm_to_inches((3.5,2.0)))
    sns.heatmap(df[all_cytokines_to_display], cbar=True, cmap='coolwarm', vmin=-1, vmax=1,
                cbar_kws={"label": "Pearson r to consensus log2C"}, annot=True, fmt=".2f", annot_kws={"size": 5, "rotation": 90}, square=False)
    comm_names = df.index
    plt.yticks(np.arange(len(comm_names))+0.5, [format_community_name(name) for name in comm_names], rotation=0, fontsize=6)
    apply_greek_labels_heatmap(plt.gca())
    plt.tick_params(length=0)
    plt.title(f"{introduce_greek_letters(cytokine)} on {cell_type}", fontsize=7)
    for format in ['png', 'svg']:
        plt.savefig(os.path.join(plot_folder_heatmaps, f"corr_to_other_cytokines_{cytokine}_{cell_type}.{format}"), dpi=300)
    plt.close()

In [ ]:
# compare response to IL-32-beta in CD4 T cells to overall response to IL-10 by donor subgroup
cell_types = ["CD4 T cell"]
for cell_type in cell_types:
    cytokine_base = "IL-32-beta"
    cytokine_comp = "IL-10"
    colors = {'D7_D8_D12_D5_D6_D9_D11': "forestgreen", "D1_D3_D4_D10": "royalblue"} # D2 not present because it doesn't cluster here
    plt.figure(figsize=cm_to_inches((3.4,3.2)))
    current_means = all_community_means[f'{cytokine_base}_{cell_type}'].copy()
    for i, comm in enumerate(current_means.keys()):
        current_mean = current_means[comm].dropna()
        edger_lfc = edger_df_log2FoldChange[(cell_type, cytokine_comp)].dropna()
        current_mean, edger_lfc = get_common_index(current_mean, edger_lfc)
        r = np.corrcoef(current_mean, edger_lfc)[0, 1]
        plt.scatter(current_mean, edger_lfc, edgecolors="none", alpha=0.5, s=1, color=colors[comm], rasterized=True,
                    label=f"r={r:.2f}")
    plt.xlabel(introduce_greek_letters(f"mean log2FC ({cytokine_base})"), fontsize=6)
    plt.ylabel(f"consensus log2FC ({cytokine_comp})", fontsize=6)
    plt.xticks([-2, 0, 2])
    plt.tight_layout()
    plt.legend(loc='upper left')
    plt.savefig(os.path.join(plot_folder_heatmaps, f"2.4_scatter_across_cyto_{cell_type}_{cytokine_base}_{cytokine_comp}.svg"), dpi=400)

# 3 - Receptor expression analysis

In [ ]:
plot_folder = os.path.join(base_plot_folder, "3_literature_responses")
os.makedirs(plot_folder, exist_ok=True)

## 3.1 - Receptor expression

In [ ]:
def get_receptor_expression(average_control_expression, cytokine):
    receptor_genes_all = str(cytokine_info_parse.loc[cytokine, "receptor gene"])
    if receptor_genes_all == "nan":
        print(f"No receptor genes found for {cytokine}")
    
    all_receptor_expression_vals = []
    receptor_variants = receptor_genes_all.split(";")
    for receptor_variant in receptor_variants:
        receptor_genes = receptor_variant.split(",")
        receptor_genes = [gene.strip().replace(" ", "") for gene in receptor_genes]
        filtered_receptor_genes = [gene for gene in receptor_genes if gene in average_control_expression.index]
        
        if len(filtered_receptor_genes) > 0:
            receptor_expression = average_control_expression.loc[filtered_receptor_genes, :].min(axis=0) # all subunits must be expressed to trigger a response
            all_receptor_expression_vals.append(receptor_expression)
    if len(all_receptor_expression_vals) > 0:
        receptor_expression = pd.concat(all_receptor_expression_vals, axis=1).max(axis=1) # any of the alternative receptors can trigger a response
    else:
        receptor_expression = None
    return receptor_expression

receptor_expression_by_cytokine = {}
for cytokine in CYTOKINE_ORDER:
    receptor_expression = get_receptor_expression(control_norm_gene_count_df, cytokine)
    if receptor_expression is not None:
        receptor_expression_by_cytokine[cytokine] = receptor_expression
receptor_expression_by_cytokine = pd.DataFrame(receptor_expression_by_cytokine).T

In [ ]:
# Receptor expression by donor, if needed
# receptor_expression_by_cytokine_and_donor = {}
# for cytokine in CYTOKINE_ORDER:
#     receptor_by_donor = {}
#     for donor in DONOR_NAMES:
#         receptor_expression = get_receptor_expression(sum_gene_count_df_norm_by_donor_pbs.loc[:, (slice(None), donor)].droplevel(1, axis=1), cytokine)
#         if receptor_expression is not None:
#             receptor_by_donor[donor] = receptor_expression
#     receptor_expression_by_cytokine_and_donor[cytokine] = pd.DataFrame(receptor_by_donor)
#     receptor_expression_by_cytokine_and_donor[cytokine] = receptor_expression_by_cytokine_and_donor[cytokine].reindex(index=ALL_CELL_TYPES_EXCLUDE, columns=DONOR_NAMES)
    
# # Convert the nested dictionary structure to a proper DataFrame
# receptor_expression_by_cytokine_and_donor_flat = {}
# for cytokine, df in receptor_expression_by_cytokine_and_donor.items():
#     for donor in df.columns:
#         for cell_type in df.index:
#             receptor_expression_by_cytokine_and_donor_flat[(cytokine, donor, cell_type)] = df.loc[cell_type, donor]

# # Create MultiIndex DataFrame from the flattened dictionary
# multi_index = pd.MultiIndex.from_tuples(list(receptor_expression_by_cytokine_and_donor_flat.keys()), 
#                                        names=['cytokine', 'donor', 'cell_type'])
# receptor_expression_series = pd.Series(list(receptor_expression_by_cytokine_and_donor_flat.values()), 
#                                      index=multi_index)
# receptor_expression_by_cytokine_and_donor = receptor_expression_series.unstack(['cytokine', 'donor'])    

In [ ]:
receptor_expression_by_cytokine_raw = receptor_expression_by_cytokine.copy()

# generate the relative receptor expression
receptor_expression_by_cytokine = receptor_expression_by_cytokine[ALL_CELL_TYPES]
receptor_expression_by_cytokine_relative = receptor_expression_by_cytokine.div(receptor_expression_by_cytokine.max(axis=1), axis=0)

# we should have found receptors for all cytokines except for IL-32-beta
for cytokine in CYTOKINE_ORDER:
    if not cytokine in receptor_expression_by_cytokine_relative.index:
        print(f"No receptor expression found for {cytokine}")

# get the subset of distances relevant for later plotting
all_normalized_distances_mean_subset = all_normalized_distances_mean.loc[receptor_expression_by_cytokine_relative.index, receptor_expression_by_cytokine_relative.columns]

In [ ]:
receptor_expression_by_cytokine.to_csv(os.path.join(output_tables_folder, "3.1_receptor_expression_by_cytokine.csv"))
receptor_expression_by_cytokine_relative.to_csv(os.path.join(output_tables_folder, "3.1_receptor_expression_by_cytokine_relative.csv"))

In [ ]:
df = receptor_expression_by_cytokine.copy()
df = df.loc[[c for c in CYTOKINE_ORDER if c in df.index], :].reset_index()
df['index'] = df['index'].apply(introduce_greek_letters)
df = df.rename(columns={'index': 'Cytokine'})
write_to_SI_table(df, "receptor_expression")

#### Scatter plot between the receptor expression and response magnitude/DEG number

In [ ]:
# generate melted dataframes for the scatter plot
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative.melt(ignore_index=False).rename(columns={"value": "relative_expression"})
all_normalized_distances_mean_flat = all_normalized_distances_mean_subset.melt(ignore_index=False).rename(columns={"value": "distance"}).rename(columns={"variable": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.reset_index().merge(
    all_normalized_distances_mean_flat.reset_index(),
    on=["index", "cell_type"]).set_index("index")

# add the raw receptor expression
raw_expression = receptor_expression_by_cytokine_raw.melt(ignore_index=False)
raw_expression = raw_expression.rename(columns={"value": "raw_expression", "variable": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.merge(
    raw_expression, 
    left_on=["index", "cell_type"],
    right_on=[raw_expression.index, "cell_type"]
)

# add the relative number of DEGs
deg_count = data_long[["cell_type", "cytokine", "DEG_rel"]].set_index("cytokine").rename(columns={"cell_type": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.merge(
    deg_count, 
    left_on=["index", "cell_type"],
    right_on=[deg_count.index, "cell_type"]
)

# order values by ALL_CELL_TYPES
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.sort_values(by="cell_type", key=lambda x: pd.Series(x).map(dict(enumerate(ALL_CELL_TYPES))))

In [ ]:
# ---------------- THRESHOLDS -------------------
receptor_expression_low_threshold = 5
receptor_expression_present_threshold = 8
receptor_expression_high_threshold = 16
receptor_expression_very_high_threshold = 32
# ---------------- THRESHOLDS -------------------


excluded_cytokines = []
fig, axes = plt.subplots(nrows=3, ncols=4, figsize=cm_to_inches((12, 10)), constrained_layout=True)
axes = axes.flatten()

# Collect legend labels
legend_labels = []

outliers_rec_by_cell_type_low_high = []
outliers_rec_by_cell_type_high_low = []
outliers_rec_by_cell_type_high_high = []
primary_cell_types = []
for i, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes[i]
    df_subset = receptor_expression_by_cytokine_relative_flat[receptor_expression_by_cytokine_relative_flat["cell_type"] == cell_type].copy()
    df_subset['raw_expression_log2'] = np.log2(df_subset['raw_expression'])
    df_subset = df_subset[~df_subset["index"].isin(excluded_cytokines)].copy()
    
    r = np.corrcoef(df_subset["raw_expression_log2"],
                    df_subset["distance"])[0, 1]
    scatter = ax.scatter(df_subset["raw_expression_log2"],
                         df_subset["distance"], s=3,
                         color=CELL_TYPE_COLORS[cell_type])
    
    # strong response despite low receptor expression (secondary effects?)
    df_subset_low_high = df_subset[(df_subset["raw_expression"] < receptor_expression_low_threshold) & (df_subset["distance"] > 1.5 * magnitude_threshold_by_cell_type[cell_type])]
    
    # weak response despite high receptor expression
    df_subset_high_low = df_subset[(df_subset["raw_expression"] > receptor_expression_very_high_threshold) & (df_subset["distance"] < 1 * magnitude_threshold_by_cell_type[cell_type])]
    
    # clear primary targets (use first definition)
    df_subset_high_high = df_subset[(df_subset["raw_expression"] > receptor_expression_high_threshold) & (df_subset["distance"] > 1.5 * magnitude_threshold_by_cell_type[cell_type])]
    # df_subset_primary = df_subset[(df_subset["raw_expression"] > receptor_expression_high_threshold) & (df_subset["distance"] > 1.0 * magnitude_threshold_by_cell_type[cell_type])]
    
    outliers_rec_by_cell_type_low_high.append(df_subset_low_high)
    outliers_rec_by_cell_type_high_low.append(df_subset_high_low)
    outliers_rec_by_cell_type_high_high.append(df_subset_high_high)
    # primary_cell_types.append(df_subset_primary)
    
    # for plotting only (useful ones to plot labels for)
    df_subset_very_low = df_subset[(df_subset["raw_expression"] < receptor_expression_low_threshold) & (df_subset["distance"] > 1.3 * magnitude_threshold_by_cell_type[cell_type])]
    label = df_subset_very_low.sort_values(by="distance", ascending=False).head(3)["index"]
    
    # Add text labels for all cytokines in 'label'
    for idx, cytokine in enumerate(label):
        ax.annotate(
            introduce_greek_letters(cytokine),
            xy=(df_subset_very_low[df_subset_very_low["index"] == cytokine]["raw_expression_log2"].values[0],
                df_subset_very_low[df_subset_very_low["index"] == cytokine]["distance"].values[0]),
            xytext=(0, 0),  # offset for annotation text position
            textcoords='offset points',
            fontsize=5.5,
            color='black'
        )
    
    # Collect the label for the legend
    legend_labels.append((scatter, f"{cell_type} (r = {r:.2f})"))
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlim(-0.3, 9)
    
    ax.axhline(y=magnitude_threshold_by_cell_type[cell_type], linestyle='--', color='gray', alpha=0.75)
    ax.axvline(x=np.log2(receptor_expression_present_threshold), linestyle='--', color='gray', alpha=0.75)
    
    if i % 4 == 0:  # First column
        ax.set_ylabel("response magnitude")
    if i >= 8:  # Last row
        ax.set_xlabel(r"$\log_{2}$(receptor cpm)")
    ax.label_outer()  # Only show outer labels

outliers_rec_by_cell_type_low_high = pd.concat(outliers_rec_by_cell_type_low_high)
outliers_rec_by_cell_type_high_low = pd.concat(outliers_rec_by_cell_type_high_low)
outliers_rec_by_cell_type_high_high = pd.concat(outliers_rec_by_cell_type_high_high)
#primary_cell_types = pd.concat(primary_cell_types)

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Plot the legend once
handles, labels = zip(*legend_labels)
fig.legend(handles, labels, loc="lower right", fontsize=6, ncol=2)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_scatter_receptor_vs_distance.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
from matplotlib.colors import ListedColormap

cytokines_by_cell_type_classification = pd.DataFrame(
    index=[c for c in CYTOKINE_ORDER if c != "IL-32-beta"],
    columns=ALL_CELL_TYPES,
    data='weak receptor - weak response',
)
for _, row in outliers_rec_by_cell_type_low_high.iterrows():
    cytokines_by_cell_type_classification.loc[row['index'], row['cell_type']] = 'weak receptor - strong response'
for _, row in outliers_rec_by_cell_type_high_low.iterrows():
    cytokines_by_cell_type_classification.loc[row['index'], row['cell_type']] = 'strong receptor - weak response'

not_already_annotated = cytokines_by_cell_type_classification == 'weak receptor - weak response'
cytokines_by_cell_type_classification[strong_impact_by_cell_type & not_already_annotated] = 'expressed receptor - strong impact'
    
class_to_int = {
    'weak receptor - weak response': 0,
    'weak receptor - strong response': 1,
    'strong receptor - weak response': 2,
    'expressed receptor - strong impact': 3,
}
int_matrix = cytokines_by_cell_type_classification.replace(class_to_int)
int_matrix = int_matrix[~(int_matrix == 0).all(axis=1)] # get rid of boring entries
colors = [
    'lightgray',     # 0
    'salmon',        # 1
    'gold',          # 2
    'lightseagreen', # 3
]
cmap = ListedColormap(colors)

n = 3
row_indices = list(range(int_matrix.shape[0]))
chunk_size = (len(row_indices) + n - 1) // n 
row_idx_chunks = [row_indices[i:i+chunk_size] for i in range(0, len(row_indices), chunk_size)]

fig, axes = plt.subplots(1, n, figsize=cm_to_inches((14, 10)))
axes = np.atleast_1d(axes)  # in case n=1

for ax, row_idx in zip(axes, row_idx_chunks):
    submat = int_matrix.iloc[row_idx, :]
    sns.heatmap(
        submat,
        cmap=cmap,
        cbar=False,
        linewidths=0.5,
        linecolor='gray',
        square=True,
        ax=ax,
    )
    ax.tick_params(length=0)
    apply_greek_labels_heatmap(ax)

labels = list(class_to_int.keys())
handles = [
    mpatches.Patch(color=colors[class_to_int[label]], label=label)
    for label in labels
]
fig.legend(
    handles,
    labels,
    loc='lower center',       # or 'right', 'upper center', etc.
    ncol=2,                   # number of columns
    frameon=False,
    bbox_to_anchor=(0.5, -0.02)  # shift downward to clear plots
)

plt.subplots_adjust(wspace=0.5)
plt.savefig(os.path.join(plot_folder, "3_cytokine_response_classification_heatmap.svg"), dpi=300)

In [ ]:
# remove HSPCs here (not as meaningful because of higher noise; also not nearly as convincing as a secreter of cytokines due to low cell counts)
# high receptor low response
outliers_rec_by_cell_type_high_low = outliers_rec_by_cell_type_high_low[~outliers_rec_by_cell_type_high_low["cell_type"].isin(["HSPC"])]
outliers_rec_by_cell_type_high_low = outliers_rec_by_cell_type_high_low.sort_values(by="raw_expression", ascending=False)

# high receptor high response
outliers_rec_by_cell_type_high_high = outliers_rec_by_cell_type_high_high[~outliers_rec_by_cell_type_high_high["cell_type"].isin(["HSPC"])]
outliers_rec_by_cell_type_high_high = outliers_rec_by_cell_type_high_high.sort_values(by="raw_expression", ascending=False)

# low receptor high response
outliers_rec_by_cell_type_low_high = outliers_rec_by_cell_type_low_high[~outliers_rec_by_cell_type_low_high["cell_type"].isin(["HSPC"])]
outliers_rec_by_cell_type_low_high = outliers_rec_by_cell_type_low_high.sort_values(by="raw_expression", ascending=False)

In [ ]:
# Reformat as index x cell_type with 0/1
outlier_matrix = pd.crosstab(
    outliers_rec_by_cell_type_low_high["index"],
    outliers_rec_by_cell_type_low_high["cell_type"]
)
outlier_matrix = (outlier_matrix > 0).astype(int)
# Reorder the outlier_matrix according to CYTOKINE_ORDER (rows) and CELL_TYPE_ORDER (columns)
# Only keep those in the intersection of outlier_matrix and the order lists

# Get intersection for rows and columns
row_order = [cyt for cyt in CYTOKINE_ORDER if cyt in outlier_matrix.index]
col_order = [ct for ct in ALL_CELL_TYPES if ct in outlier_matrix.columns]

outlier_matrix = outlier_matrix.loc[row_order, col_order]
plt.figure(figsize=cm_to_inches((8, 6)))
ax = sns.heatmap(outlier_matrix, cmap="rocket", cbar=True, linewidths=0.5, linecolor='gray', square=True)
plt.xlabel("")
plt.ylabel("")
plt.title("Potential secondary responses")
# Remove the ticks but keep the labels
ax.tick_params(axis='x', which='both', bottom=False, top=False)
ax.tick_params(axis='y', which='both', left=False, right=False)
ax = apply_greek_labels_heatmap(ax)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"6.1.2_likely_secondary_responses.{format}"), dpi=300)

In [ ]:
# Reformat as index x cell_type with 0/1
outlier_matrix = pd.crosstab(
    outliers_rec_by_cell_type_high_low["index"],
    outliers_rec_by_cell_type_high_low["cell_type"]
)
outlier_matrix = (outlier_matrix > 0).astype(int)
# Reorder the outlier_matrix according to CYTOKINE_ORDER (rows) and CELL_TYPE_ORDER (columns)
# Only keep those in the intersection of outlier_matrix and the order lists

# Get intersection for rows and columns
row_order = [cyt for cyt in CYTOKINE_ORDER if cyt in outlier_matrix.index]
col_order = [ct for ct in ALL_CELL_TYPES if ct in outlier_matrix.columns]

outlier_matrix = outlier_matrix.loc[row_order, col_order]
plt.figure(figsize=cm_to_inches((8, 10)))
ax = sns.heatmap(outlier_matrix, cmap="rocket", cbar=True, linewidths=0.5, linecolor='gray', square=True)
plt.xlabel("")
plt.ylabel("")
plt.title("Strong receptor-weak response")
# Remove the ticks but keep the labels
ax.tick_params(axis='x', which='both', bottom=False, top=False)
ax.tick_params(axis='y', which='both', left=False, right=False)
ax = apply_greek_labels_heatmap(ax)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"6.1.2_strong_receptor_weak_response.{format}"), dpi=300)

In [ ]:
strong_sum = strong_impact_by_cell_type.loc[:, ~strong_impact_by_cell_type.columns.isin(["HSPC"])].sum(axis=1)

potential_failures = outliers_rec_by_cell_type_high_low["index"].value_counts().to_frame()
potential_failures["strong_impact"] = strong_sum.loc[potential_failures.index]
potential_failures = potential_failures[potential_failures['count'] > 1]

plt.figure(figsize=cm_to_inches((10, 8)))
plt.scatter(potential_failures['strong_impact'], potential_failures['count'], alpha=0)  # invisible points as placeholders

# Create a dictionary to track points at the same coordinates
point_dict = {}
for idx, row in potential_failures.iterrows():
    key = (row['strong_impact'], row['count'])
    if key not in point_dict:
        point_dict[key] = []
    point_dict[key].append(idx)

# Add text labels at each point with vertical spacing for duplicates
for coords, labels in point_dict.items():
    x, y = coords
    if len(labels) == 1:
        # Single label - center it
        plt.text(x, y, introduce_greek_letters(labels[0]), fontsize=6, 
                 ha='center', va='center')
    else:
        # Multiple labels at same point - space them vertically
        offset = 0.5  # Vertical spacing between labels
        total_height = offset * (len(labels) - 1)
        start_y = y - total_height/2
        
        for i, label in enumerate(labels):
            plt.text(x, start_y + i*offset, introduce_greek_letters(label), 
                     fontsize=6, ha='center', va='center')

plt.xlim(-0.9, 8.9)
plt.ylim(0, 14)
plt.xlabel('# strong impact')
plt.ylabel('# strong receptor - weak response')
plt.grid(True, alpha=0.3)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"6.1.2-failed-responses.{format}"), dpi=300)

In [ ]:
# primary_cell_types.rename(columns={
#     "index": "cytokine",
#     "distance": "response_magnitude",
#     "raw_expression": "receptor_rpm",
#     "relative_expression": "relative_receptor_expression"
#     },
#     inplace=True)
# primary_cell_types['receptor_rpm'] = 2**primary_cell_types['receptor_rpm']
# primary_cell_types = primary_cell_types[["cytokine", "cell_type", "receptor_rpm", "relative_receptor_expression", "response_magnitude"]]
# primary_cell_types.to_excel(os.path.join(output_tables_folder, "primary_target_cell_types.xlsx"))

In [ ]:
# inspect correlation across cell types - not very enlightening
# receptor_magnitude_corr = {}
# for cytokine in CYTOKINE_ORDER:
#     if not cytokine in receptor_expression_by_cytokine.index:
#         continue
#     if not any(strong_impact_by_cell_type.loc[cytokine]):
#         continue
#     receptor = receptor_expression_by_cytokine.loc[cytokine, ALL_CELL_TYPES_EXCLUDE]
#     magnitude = all_normalized_distances_mean.loc[cytokine, ALL_CELL_TYPES_EXCLUDE]
#     if max(magnitude) - min(magnitude) < 0.2:
#         continue
#     corr = np.corrcoef(receptor, magnitude)[0, 1]
#     receptor_magnitude_corr[cytokine] = corr
# receptor_magnitude_corr = pd.Series(receptor_magnitude_corr).sort_values(ascending=False)

# cytokine = "BAFF"
# receptor = receptor_expression_by_cytokine.loc[cytokine, ALL_CELL_TYPES_EXCLUDE]
# magnitude = all_normalized_distances_mean.loc[cytokine, ALL_CELL_TYPES_EXCLUDE]
# plt.figure(figsize=cm_to_inches((6,5)))
# plt.scatter(receptor, magnitude, s=15)
# plt.ylim(0, 1.05)
# plt.xlim(0, 1.2*max(receptor))
# plt.xlabel("receptor expression")
# plt.ylabel("response magnitude")
# plt.title(f"{cytokine}, corr: {receptor_magnitude_corr.loc[cytokine]:.2f}")
# plt.tight_layout()
# plt.show()

### Receptor expression heatmaps

In [ ]:
plt.figure(figsize=cm_to_inches((5, 7)))
ax = plt.gca()

log_counts = np.log2(receptor_expression_by_cytokine.clip(lower=1))


offset = 0.5
for i, cell_type in enumerate(ALL_CELL_TYPES[::-1]):
    data = log_counts[cell_type]
    counts, bins = np.histogram(data, bins=50, range=(0, 10))
    
    # Normalize counts to have similar heights
    if counts.max() > 0:
        counts = counts / counts.max() * 0.4
    
    # Plot histogram with vertical offset
    plt.bar(bins[:-1], counts, width=bins[1]-bins[0], bottom=i*offset, 
            alpha=0.7, label=cell_type if i == 0 else "", color=CELL_TYPE_COLORS[cell_type])
    
    # Add cell type label
    plt.text(-0.2, i*offset + 0.2, cell_type, ha='right', va='center', fontsize=6)
    
    # Add a horizontal line at the bottom of each histogram instead of a box
    plt.plot([0, 10], [i*offset, i*offset], color='black', linestyle='-', linewidth=0.5)

plt.plot([np.log2(receptor_expression_present_threshold), np.log2(receptor_expression_present_threshold)], [0, len(ALL_CELL_TYPES)*offset], color='red', linestyle='--', linewidth=1, label='Expression threshold')
plt.xlabel(r'log$_{2}$(receptor expression + 1)')
plt.yticks([])
plt.xlim(0, 10)
plt.ylim(0, len(ALL_CELL_TYPES)*offset)

plt.tight_layout()
# # Save the figure
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_receptor_expression_dist.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
fig = plt.figure(figsize=cm_to_inches((6, 20)))
ax = sns.heatmap(np.log10(receptor_expression_by_cytokine_relative.clip(0.01, 1)), 
                cmap="coolwarm", 
                square=True, 
                cbar_kws={"shrink": 0.4, "label": r"$\log_{10}$(relative receptor expression)"})


ax.set_yticks(np.arange(len(receptor_expression_by_cytokine_relative.index)) + 0.5)
ax.set_yticklabels([cytokine for cytokine in receptor_expression_by_cytokine_relative.index], fontsize=5.5)
apply_greek_labels_heatmap(ax)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', rotation=90, labelsize=5.5, length=0)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_receptor_expression_by_cytokine_relative_log.{format}"), 
                bbox_inches="tight", 
                dpi=300)

In [ ]:
max_receptor_expression = receptor_expression_by_cytokine[ALL_CELL_TYPES].max(axis=1)

plt.figure(figsize=cm_to_inches((1, 22.85)))
max_receptor_expression = max_receptor_expression[::-1]
plt.barh(y=max_receptor_expression.index, width=np.log10(max_receptor_expression.values), color="black", height=0.7)
plt.ylim(-0.5, len(max_receptor_expression)-0.5)

# get the current figure and draw it (to get apply_greek_labels_heatmap to work for this plot)
fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())

plt.xlim(0, 3)
plt.xlabel(r"$\log_{10}$(max. cpm)", fontsize=6)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_maximum_receptor_expression_by_cytokine.{format}"), bbox_inches="tight", dpi=300)

## 3.2 - Individual cell type and expected literature responses

In [ ]:
plot_folder = os.path.join(base_plot_folder, "3.2_individual_literature_responses")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
edger_df_padj_upregulated = edger_df_padj.copy()
edger_df_padj_downregulated = edger_df_padj.copy()

# Get boolean mask of negative fold changes
negative_fc_mask = edger_df_log2FoldChange < 0
positive_fc_mask = edger_df_log2FoldChange > 0

# Set padj values to 1 where fold change is negative
edger_df_padj_upregulated[negative_fc_mask] = 1
edger_df_padj_downregulated[positive_fc_mask] = 1

### Individual response lookup

In [ ]:
# manually look up
strongest_response_in_literature = {}
upregulated_genes = pd.DataFrame(columns=["gene", "cell_type", "cytokine", "response_type"])

def evaluate_cytokine(current_cytokine, strongest_response_in_literature):
    data_long_cytokine = data_long[data_long["cytokine"] == current_cytokine].copy().set_index("cell_type")
    print("There should be strong response in the following cell types:")
    print(strongest_response_in_literature[current_cytokine])
    print("The relative DEGs and Magnitudes are:")
    print(data_long_cytokine[["DEG_rel", "Distance"]])

def print_regulated_genes(upregulated_genes, current_cytokine):
    # Get genes for this cytokine
    cytokine_genes = upregulated_genes[upregulated_genes["cytokine"] == current_cytokine]
    
    # Group by cell type
    for cell_type, cell_genes in cytokine_genes.groupby("cell_type"):
        print(f"{cell_type}")
        print("--------------------------------")
        
        for _, row in cell_genes.iterrows():
            gene = row["gene"]
            if gene in edger_df_log2FoldChange.index:
                fc = edger_df_log2FoldChange.loc[gene, (cell_type, current_cytokine)]
                pval = edger_df_padj.loc[gene, (cell_type, current_cytokine)]
                
                print(f"The fold change for {gene} in response to {current_cytokine} is {fc}")
                print(f"The p-value for {gene} in response to {current_cytokine} is {pval:e}" \
                    if not pd.isna(pval) else f"The p-value for {gene} in response to {current_cytokine} is NaN")
                print("----")
            else:
                if gene in unfiltered_genes:
                    print(f"The gene {gene} is not in padj, but is in the list of unfiltered genes.")
                else:
                    print(f"The gene {gene} is not in padj, and is not in the list of unfiltered genes.")

#### We're interested in these (fairly randomly chosen) cytokines: IL-3, IL-32-beta, GM-CSF, TPO, IL-7, CD40L, IL-4, IL-12

In [ ]:
# --------------------------------------------------
# IL-3: 10.3389/fimmu.2024.1411047
current_cytokine = "IL-3"
print(f"Processing {current_cytokine}")
print("""CD123 (IL3RA) is mainly expressed on hematopoietic cells, including hematopoietic stem and progenitor cells (HSPCs),
basophils, eosinophils, mast cells, non-classical monocytes, macrophages, human plasmacytoid dendritic cells (pDCs)
and activated T and B cells (...) In steady-state, murine monocytes and human classical
monocytes do not express the receptor for IL-3. Only human intermediate and non-classical monocytes express it.""")

strongest_response_in_literature[current_cytokine] = ["HSPC", "CD14 Mono", "CD16 Mono"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

print("The expression in CD14 Monos is curiously high.")

"""IL-3 stimulates the growth and
differentiation of HSPCs from bone marrow cultures and in naive or
sublethally irradiated mice into a large range of cell lineages,
including basophils, neutrophils, eosinophils, macrophages"""

"""IL-3 synergizes with IL-4 to enhance CCL17 expression in CD14+ monocytes """
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'CCL3', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'}, 
    {'gene': 'CCL7', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
    {'gene': 'CCL17', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'}
])])

print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-32-beta: 10.1002/JLB.3MR0620-550R
current_cytokine = "IL-32-beta"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["NK CD56low", "NK CD56hi", "CD14 Mono", "CD16 Mono", "cDC"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# pick some upregulated cytokines
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'IL1B', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'inflammatory'},
    {'gene': 'IL6', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'inflammatory'}
])])

print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# M/GM-CSF: 10.1016_j.immuni.2016.10.026
current_cytokine = "GM-CSF"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["CD14 Mono", "CD16 Mono", "cDC"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# Croxford (2015) - Immunity - The Cytokine GM-CSF (pick a subset of genes for plotting)
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'PYCARD', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    {'gene': 'CLEC7A', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    {'gene': 'CCL17', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
    {'gene': 'CCL24', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
])])
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# TPO: 10.1080/09537104.2021.1925102
current_cytokine = "TPO"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["HSPC"] 
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# de Graaf (2011) - Cell cycle - Thrombopoietin and hematopoietic stem cells
# these genes aren't really mentioned directly, so don't plot this
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'CDKN1C', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'quiescence'},
    {'gene': 'CDKN2D', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'quiescence'},
    {'gene': 'HOXB4', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA5', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA9', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA10', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'BCL2L1', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'survival'}
])])

print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-7: 10.3389/fimmu.2021.747324
current_cytokine = "IL-7"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# IL-7: 10.3389/fimmu.2021.747324
# Don't add CD14 Monos here to avoid clutter
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'BCL2', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'survival'},
    {'gene': 'PIM1', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'survival'},
    {'gene': 'BCL2', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell' if not alt_flag else 'CD4 Naive', 'response_type': 'survival'},
    {'gene': 'PIM1', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell' if not alt_flag else 'CD4 Naive', 'response_type': 'survival'},
    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
])])
print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# CD40L: 10.1111/j.1600-065X.2009.00782.x
current_cytokine = "CD40L"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["B cell", "cDC", "CD14 Mono", "CD16 Mono"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# all of these work, but I'll pick two to actually plot
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'NFKB2', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'NF-kappaB'},
    {'gene': 'TRAF1', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'NF-kappaB'},
])])
print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-4: 10.12703/r/10-71
current_cytokine = "IL-4"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["B cell", "CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg", "NK CD56low", "NK CD56hi", "CD14 Mono", "CD16 Mono", "cDC"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# only plot B cell responses for this one
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'FCER2', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'isotype'}, # isotype switch, IgE receptor
    {'gene': 'IGHE', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'isotype'}, # isotype switch, IgG3
    {'gene': 'IL4I1', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'other'}, ## of recent interest
    {'gene': 'IL4R', 'cytokine': current_cytokine, 'cell_type': 'B cell' if not alt_flag else 'B Naive', 'response_type': 'other'}, ## induce its own receptor
])])
print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-12: 10.17179/excli2020-3104
current_cytokine = "IL-12"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["NK CD56low", "NK CD56hi", "CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
    {'gene': 'TNF', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell' if not alt_flag else "CD4 Naive", 'response_type': 'cytokine production'},
    {'gene': 'GZMB', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytotoxicity'},
    {'gene': 'PRF1', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytotoxicity'}
])])
print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

### Check if expected global impact conforms to expectations

In [ ]:
strongest_response_in_literature["IFN-beta"] = ALL_CELL_TYPES

count_significant_responses = 0
expected_responses = pd.DataFrame(index=strongest_response_in_literature.keys(), columns=ALL_CELL_TYPES, data=0)
for cytokine in strongest_response_in_literature.keys():
    for cell_type in strongest_response_in_literature[cytokine]:
        if cell_type in ALL_CELL_TYPES_EXCLUDE:
            expected_responses.loc[cytokine, cell_type] = 1
            if strong_impact_by_cell_type.loc[cytokine, cell_type]:
                count_significant_responses += 1
        else:
            print(f"The cell type {cell_type} is not in the list of all cell types.")

In [ ]:
no_significant_responses_subset = strong_impact_by_cell_type.loc[expected_responses.index, expected_responses.columns][ALL_CELL_TYPES]
count_significant_responses / expected_responses.sum().sum()

In [ ]:
count_significant_responses / expected_responses.sum().sum()

In [ ]:
ordered_index = [i for i in CYTOKINE_ORDER if i in expected_responses.index]
expected_responses = expected_responses.loc[ordered_index, :]
all_distances_mean_subset = all_normalized_distances_mean.loc[expected_responses.index, expected_responses.columns][ALL_CELL_TYPES]

In [ ]:
plt.figure(figsize=cm_to_inches((5,3.8)))
ax = sns.heatmap(all_distances_mean_subset, vmin=0.0, vmax=1, cmap="coolwarm",
            cbar_kws={"ticks": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0], "label": "response magnitude", "shrink": 0.6}, square=True)

ax = plt.gca()

for cytokine_index, cytokine in enumerate(expected_responses.index):
    row = expected_responses.loc[cytokine]

    # go through contiguous runs of same-color boxes
    prev_color = None
    run_start = None

    for i, cell_type in enumerate(expected_responses.columns):
        if row[cell_type] == 1:
            is_strong = bool(strong_impact_by_cell_type.loc[cytokine, cell_type])
            color = "limegreen" if is_strong else "coral"
        else:
            color = None

        # check if we continue a run
        if color == prev_color and color is not None:
            # still in same-color run
            pass
        else:
            # close previous run
            if prev_color is not None:
                rect = plt.Rectangle(
                    (run_start, cytokine_index),
                    width=i - run_start,
                    height=1,
                    fill=False,
                    edgecolor=prev_color,
                    linewidth=1.3,
                    clip_on=False,
                    ls="--",
                    zorder=2 if prev_color == "lightgreen" else 3
                )
                ax.add_patch(rect)

            # start new run
            if color is not None:
                run_start = i
            else:
                run_start = None

        prev_color = color

    # close final run at row end
    if prev_color is not None and run_start is not None:
        rect = plt.Rectangle(
            (run_start, cytokine_index),
            width=len(expected_responses.columns) - run_start,
            height=1,
            fill=False,
            edgecolor=prev_color,
            linewidth=1.3,
            clip_on=False,
            ls="--",
            zorder=2 if prev_color == "lightgreen" else 3
        )
        ax.add_patch(rect)
            
# # Add legend for green box
# # Create custom legend patch for green box
# green_box = plt.Rectangle((0,0), 1, 1, fill=False, edgecolor="lightgreen", linewidth=2)
# plt.gca().add_artist(plt.legend([green_box], ['expected response'], loc='center left', bbox_to_anchor=(0.5, 0.5), handlelength=1, handletextpad=1))

apply_greek_labels_heatmap(ax)
plt.gca().tick_params(length=0)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_expected_strength_responses_literature.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
plt.figure(figsize=cm_to_inches((5,3.8)))
receptor_expression_by_cytokine_relative_plot = receptor_expression_by_cytokine_relative.copy()
receptor_expression_by_cytokine_relative_plot.loc["IL-32-beta"] = np.nan
ax = sns.heatmap(np.log10(receptor_expression_by_cytokine_relative_plot.loc[expected_responses.index, expected_responses.columns].clip(0.01)), 
                cmap="coolwarm", 
                square=True, 
                cbar_kws={"shrink": 0.6, "label": r"$\log_{10}$(rel. receptor expr.)"})

ax = plt.gca()

for cytokine_index, cytokine in enumerate(expected_responses.index):
    row = expected_responses.loc[cytokine]

    # go through contiguous runs of same-color boxes
    prev_color = None
    run_start = None

    for i, cell_type in enumerate(expected_responses.columns):
        if row[cell_type] == 1:
            is_strong = bool(strong_impact_by_cell_type.loc[cytokine, cell_type])
            color = "limegreen" if is_strong else "coral"
        else:
            color = None

        # check if we continue a run
        if color == prev_color and color is not None:
            # still in same-color run
            pass
        else:
            # close previous run
            if prev_color is not None:
                rect = plt.Rectangle(
                    (run_start, cytokine_index),
                    width=i - run_start,
                    height=1,
                    fill=False,
                    edgecolor=prev_color,
                    linewidth=1.3,
                    clip_on=False,
                    ls="--",
                    zorder=2 if prev_color == "lightgreen" else 3
                )
                ax.add_patch(rect)

            # start new run
            if color is not None:
                run_start = i
            else:
                run_start = None

        prev_color = color

    # close final run at row end
    if prev_color is not None and run_start is not None:
        rect = plt.Rectangle(
            (run_start, cytokine_index),
            width=len(expected_responses.columns) - run_start,
            height=1,
            fill=False,
            edgecolor=prev_color,
            linewidth=1.3,
            clip_on=False,
            ls="--",
            zorder=2 if prev_color == "lightgreen" else 3
        )
        ax.add_patch(rect)

apply_greek_labels_heatmap(ax)
plt.gca().tick_params(length=0)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_receptor_expression_subset.{format}"), 
                bbox_inches="tight", 
                dpi=300)

### Individual gene upregulation

In [ ]:
upregulated_genes = upregulated_genes.reset_index(drop=True)
for index, row in upregulated_genes.iterrows():
    if row["gene"] in edger_df_log2FoldChange.index:
        fc = edger_df_log2FoldChange.loc[row["gene"], (row["cell_type"], row["cytokine"])]
        pval = edger_df_padj.loc[row["gene"], (row["cell_type"], row["cytokine"])]
        upregulated_genes.loc[index, "fc"] = fc
        upregulated_genes.loc[index, "pval"] = pval
upregulated_genes.dropna(inplace=True)
upregulated_genes["cyto_response"] = upregulated_genes["cytokine"] + "\n" + upregulated_genes["response_type"]

In [ ]:
# export things
# upregulated_genes.to_csv(os.path.join("Fig1E", "3.2_upregulated_genes_literature.csv"), index=False)
# all_distances_mean_subset.to_csv(os.path.join("Fig1E", "3AB_distances.csv"))
# expected_responses.to_csv(os.path.join("Fig1E", "3AB_expected_responses.csv"))
# strong_impact_by_cell_type.to_csv(os.path.join("Fig1E", "3AB_strong_impact_by_cell_type.csv"))
# receptor_expression_by_cytokine_relative.to_csv(os.path.join("Fig1E", "3AB_receptor_expression_relative.csv"))

In [ ]:
if not alt_flag:
    cell_types = ["B cell", "CD4 T cell", "NK CD56hi", "CD14 Mono"]
else:
    cell_types = ["B Naive", "CD4 Naive", "NK CD56hi", "CD14 Mono"]
    
for cell_type in cell_types:
    df_subset = upregulated_genes[upregulated_genes["cell_type"] == cell_type]

    # Fixed dimensions for the plot area (in cm)
    plot_width = len(df_subset['cyto_response'].unique())  # Width of plot area
    plot_height = 2  # Height of plot area
    margin_bottom = 2  # Extra space for x-labels
    margin_top = 0.5  # Extra space for title
    total_width = plot_width
    total_height = plot_height + margin_bottom + margin_top

    plt.figure(figsize=cm_to_inches((total_width, total_height)))

    # Create subplot with specific spacing to maintain fixed plot area
    plt.subplots_adjust(bottom=margin_bottom/total_height, 
                    top=1-margin_top/total_height)

    # Create colormap based on -log10(pvalue)
    log_pvals = -np.log10(df_subset['pval'].clip(1E-5))
    norm = plt.Normalize(0, log_pvals.max())
    cmap = plt.cm.viridis

    plt.scatter(df_subset['cyto_response'], df_subset['fc'], 
            c=log_pvals, cmap=cmap, norm=norm,
            alpha=1, s=10)

    # Add colorbar
    cbar = plt.colorbar()
    cbar.set_label('-log10(p-value)')

    # Add gene labels next to points
    for _, row in df_subset.iterrows():
        plt.annotate(row['gene'], 
                    (row['cyto_response'], row['fc']),
                    xytext=(5, 0), 
                    textcoords='offset points',
                    rotation=90,
                    fontsize=6)

    plt.xticks(rotation=90, va='top', ha='center')
    plt.ylabel(r'log$_2$(fold change)')
    plt.title(cell_type, fontsize=7, pad=10)
    plt.xlim(-0.25, len(df_subset['cyto_response'].unique()) - 0.75)
    plt.ylim(min(min(df_subset['fc']) - 0.5, -0.25), max(df_subset['fc']) + 0.5)

    apply_greek_labels_heatmap(plt.gca())

    # Remove top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    for format in ["png", "svg"]:
        plt.savefig(os.path.join(plot_folder, f"3_individual_upregulated_genes_by_cytokine_{cell_type}.{format}"), bbox_inches="tight", dpi=300)

# 4 - Average cytokine production response across cell types

Here, we
1) Get the average and max over all cytokines.
2) Filter to genes where the maximum average expression is at least 4, the maximum maximum expression at least 20, and the total count at least 20
3) Filter the *mean* average expression to these columns.
4) Calculate the relative rpm across cell types

The resultant gene filter (expressed_genes_by_cell_type_dict) is later used to calculate correlation values between expression patterns.

In [ ]:
plot_folder = os.path.join(base_plot_folder, "4_average_cytokine_production_by_cell_type")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# 1) Get the average and max over all cytokine stimulation conditions
sum_over_all_cytokines_mean = average_norm_sum_gene_count_df.T.groupby(level='cell_type').mean().T
sum_over_all_cytokines_max = average_norm_sum_gene_count_df.T.groupby(level='cell_type').max().T
substantial_detection = average_sum_gene_count_df.T.groupby(level=0).max().T # these are total counts

# 2) Calculate the per-gene filter for later use
expressed_genes_by_cell_type_df = (sum_over_all_cytokines_mean > 4) & (sum_over_all_cytokines_max > 20) & (substantial_detection > 20)
expressed_genes_by_cell_type_dict = {cell_type: list(expressed_genes_by_cell_type_df[cell_type][expressed_genes_by_cell_type_df[cell_type]].index) for cell_type in expressed_genes_by_cell_type_df.columns}

In [ ]:
# Calculate the per-cytokine filter (different from per-gene filter because cytokines are made up of multiple genes)
# First we calculate a proxy expression level for each cytokine given all genes that constitute it
cytokine_expression_df_mean = {} # mean across conditions
cytokine_expression_df_max = {} # max across conditions
cytokine_expression_df_all_details = {} # all conditions separately
cytokine_expression_df_all_PBS = {} # PBS only
cytokine_expression_df_substantial = {} # substantial detection

# iterate over each cytokine gene individually
for index, row in cytokine_info.iterrows():
    cytokine_genes = row["gene"].split(",")
    cytokine_genes = [gene.strip() for gene in cytokine_genes]

    cytokine_genes_in_sum = [gene in sum_over_all_cytokines_mean.index for gene in cytokine_genes]
    if not all(cytokine_genes_in_sum): # skip if any of the genes are missing
        continue
    
    # the minimum is taken here under the assumption that all of the individual genes that make up a cytokine need to be expressed
    # e.g. IL23A, IL12B for IL-23
    cytokine_expression_df_all_details[index] = average_norm_sum_gene_count_df.loc[cytokine_genes, ALL_CELL_TYPES].min(axis=0)
    cytokine_expression_df_all_PBS[index] = average_norm_sum_gene_count_df.loc[cytokine_genes, (ALL_CELL_TYPES, "PBS")].min(axis=0)
    
    cytokine_expression_df_mean[index] = sum_over_all_cytokines_mean.loc[cytokine_genes, :].min(axis=0)
    cytokine_expression_df_max[index] = sum_over_all_cytokines_max.loc[cytokine_genes, :].min(axis=0)
    cytokine_expression_df_substantial[index] = substantial_detection.loc[cytokine_genes, :].min(axis=0)
    
cytokine_expression_df_mean = pd.DataFrame(cytokine_expression_df_mean)
cytokine_expression_df_max = pd.DataFrame(cytokine_expression_df_max)
cytokine_expression_df_substantial = pd.DataFrame(cytokine_expression_df_substantial)
cytokine_expression_df_all_details = pd.DataFrame(cytokine_expression_df_all_details)
cytokine_expression_df_all_PBS = pd.DataFrame(cytokine_expression_df_all_PBS).droplevel(1)

In [ ]:
# get a table for individual conditions
# for each cytokine (column), get the index (cell type) and value of the maximum in cytokine_expression_df_all_details
cytokine_expression_df_all_details_filter = cytokine_expression_df_all_details.loc[ALL_CELL_TYPES]

cytokine_max_indices = cytokine_expression_df_all_details_filter.idxmax(axis=0)
cytokine_max_values = cytokine_expression_df_all_details_filter.max(axis=0)

cytokine_max_values = cytokine_max_values[cytokine_max_values > 40] # only plot interesting cytokines with large maximum expression values
cytokine_max_indices = cytokine_max_indices.loc[cytokine_max_values.index]

# Create a DataFrame for better display
cytokine_max_indices_df = pd.DataFrame({
    'produced cytokine': cytokine_max_indices.index,
    'cell-stim': cytokine_max_indices.values,
    'mean cpm': cytokine_max_values.values
})
cytokine_max_indices_df['cell type'] = cytokine_max_indices_df['cell-stim'].apply(lambda x: x[0])
cytokine_max_indices_df['treatment'] = cytokine_max_indices_df['cell-stim'].apply(lambda x: x[1])
cytokine_max_indices_df['mean cpm'] = cytokine_max_indices_df['mean cpm']
cytokine_max_indices_df['mean cpm PBS'] = None
cytokine_max_indices_df['produced cytokine'] = cytokine_max_indices_df['produced cytokine']
for index, row in cytokine_max_indices_df.iterrows():
    treatment = row['treatment']
    cell_type = row['cell type']
    cytokine = row['produced cytokine']
    cytokine_max_indices_df.loc[index, 'mean cpm PBS'] = cytokine_expression_df_all_PBS.loc[cell_type, cytokine]
    
    if not cytokine.replace("IL-32-beta", "IL-32") in cytokine_to_gene:
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = np.nan
        continue
    genes = cytokine_to_gene[cytokine.replace("IL-32-beta", "IL-32")]
    genes = [gene.strip() for gene in genes.split(",")]
    if not all([gene in edger_df_log2FoldChange.index for gene in genes]):
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = np.nan
        continue
    log2fcs = edger_df_log2FoldChange.loc[genes, (cell_type, treatment)].values
    if all(fc > 0 for fc in log2fcs):
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = 2**min(log2fcs)
    elif all(fc < 0 for fc in log2fcs):
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = 2**max(log2fcs)
    else:
        cytokine_max_indices_df.loc[index, 'EdgeR FC'] = np.nan
cytokine_max_indices_df['produced cytokine'] = cytokine_max_indices_df['produced cytokine'].apply(lambda x: x.replace("IL-32-beta", "IL-32"))
cytokine_max_indices_df['mean cpm'] = cytokine_max_indices_df['mean cpm'].astype(int)

# difference for this one is that it doesn't take into account the paired nature of the data very well
cytokine_max_indices_df['FC to PBS'] = cytokine_max_indices_df['mean cpm'] / cytokine_max_indices_df['mean cpm PBS']
cytokine_max_indices_df['FC to PBS'] = cytokine_max_indices_df['FC to PBS'].astype(float).round(1)

# replace 'FC to PBS' values by EdgeR FC values if available
for index, row in cytokine_max_indices_df.iterrows():
    if not pd.isna(row['EdgeR FC']):
        cytokine_max_indices_df.loc[index, 'FC to PBS'] = float(f"{row['EdgeR FC']:.1f}")

# Only use max_expression, cell_type, and treatment columns, and split into two tables with at most 40 entries each
cols_to_show = ['produced cytokine', 'cell type', 'treatment', 'mean cpm', 'FC to PBS']
df_to_show = cytokine_max_indices_df[cols_to_show]
df_to_show = df_to_show.sort_values(by='produced cytokine')
df_to_show['produced cytokine'] = df_to_show['produced cytokine'].apply(lambda x: introduce_greek_letters(x))
df_to_show['treatment'] = df_to_show['treatment'].apply(lambda x: introduce_greek_letters(x))

max_rows_per_table = 40
num_rows = len(df_to_show)
num_tables = (num_rows // max_rows_per_table) + int(num_rows % max_rows_per_table > 0)

for i in range(num_tables):
    start_idx = i * max_rows_per_table
    end_idx = min((i + 1) * max_rows_per_table, num_rows)
    sub_df = df_to_show.iloc[start_idx:end_idx]
    
    fig, ax = plt.subplots(figsize=cm_to_inches((7, num_rows / 22)))
    ax.axis('off')
    table = ax.table(
        cellText=sub_df.values,
        colLabels=sub_df.columns,
        loc='center',
        cellLoc='center'
    )
    table.auto_set_font_size(False)
    table.set_fontsize(6)
    table.auto_set_column_width(col=list(range(len(sub_df.columns))))

    # Set header row (row=0) to bold and light gray background
    header_color = '#fafafa'
    for col in range(len(sub_df.columns)):
        cell = table[0, col]
        cell.set_fontsize(6)
        cell.set_text_props(weight='bold')
        cell.set_facecolor(header_color)
    
    # Set all data cells to light gray background
    for row in range(1, len(sub_df) + 1):
        for col in range(len(sub_df.columns)):
            cell = table[row, col]
            cell.set_facecolor(header_color)
    
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"4_max_cytokine_production_table_{i}.svg")) 

In [ ]:
# Get the actual expression filter for all cytokines and our tested (parse) cytokines
expressed_cytokines_by_cell_type_all_df = (cytokine_expression_df_mean > 4) & (cytokine_expression_df_max > 20) & (cytokine_expression_df_substantial > 20)
expressed_cytokines_by_cell_type_all = {}
for cell_type in expressed_cytokines_by_cell_type_all_df.index:
    series = expressed_cytokines_by_cell_type_all_df.loc[cell_type]
    expressed_cytokines_by_cell_type_all[cell_type] = [cyto for cyto in series.index if series[cyto]]
    
expressed_cytokines_by_cell_type_parse = {}
for cell_type in expressed_cytokines_by_cell_type_all.keys():
    curr_list = expressed_cytokines_by_cell_type_all[cell_type]
    expressed_cytokines_by_cell_type_parse[cell_type] = [cyto for cyto in curr_list if cyto in CYTOKINE_ORDER]

In [ ]:
# test robustness of filter choice
expressed_cytokines_by_filter_combination = {}

for max_val in [5, 10, 20, 30, 50, 100]:
    for mean_val in [1, 2, 4, 8, 16, 32]:
        filter_name = f"mean>{mean_val}_max>{max_val}"
        # Filter to genes where the maximum average expression (across cell types) is at least 4 and the maximum maximum expression (across cell types) at least 20
        cytokine_expression_df_filtered_mean = cytokine_expression_df_mean.loc[:, cytokine_expression_df_mean.loc[ALL_CELL_TYPES].max(axis=0) > mean_val]
        cytokine_expression_df_filtered_max = cytokine_expression_df_max.loc[:, cytokine_expression_df_max.loc[ALL_CELL_TYPES].max(axis=0) > max_val]

        # get the intersection of the two column sets
        columns_mean = cytokine_expression_df_filtered_mean.columns
        columns_max = cytokine_expression_df_filtered_max.columns
        columns_intersection = columns_mean.intersection(columns_max)
        expressed_cytokines_by_filter_combination[filter_name] = len(columns_intersection)

expressed_cytokines_by_filter_combination = pd.DataFrame.from_dict(
    expressed_cytokines_by_filter_combination, orient='index', columns=['num_cytokines']
    ).reset_index()
expressed_cytokines_by_filter_combination[['mean_filter', 'max_filter']] = expressed_cytokines_by_filter_combination['index'].str.extract(r'mean>(\d+)_max>(\d+)').astype(int)
expressed_cytokines_by_filter_combination = expressed_cytokines_by_filter_combination.pivot(index='mean_filter', columns='max_filter', values='num_cytokines')

plt.figure(figsize=cm_to_inches((6,5)))
sns.heatmap(expressed_cytokines_by_filter_combination, annot=True, fmt="d", cmap="YlGnBu", 
            cbar_kws={"label": "number of cytokines expresseed in >=1 cell type"}, square=True)
plt.xlabel("max expression filter")
plt.ylabel("mean expression filter")
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4_expressed_cytokines_by_filter_combination.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
# Do I use this anywhere?
# # Filter to genes where the maximum average expression (across cell types) is at least 4 and the maximum maximum expression (across cell types) at least 20
# cytokine_expression_df_filtered_mean = cytokine_expression_df_mean.loc[:, cytokine_expression_df_mean.loc[ALL_CELL_TYPES].max(axis=0) > 4]
# cytokine_expression_df_filtered_max = cytokine_expression_df_max.loc[:, cytokine_expression_df_max.loc[ALL_CELL_TYPES].max(axis=0) > 20]

# # get the intersection of the two column sets
# columns_mean = cytokine_expression_df_filtered_mean.columns
# columns_max = cytokine_expression_df_filtered_max.columns
# columns_intersection = columns_mean.intersection(columns_max)
# print(len(columns_intersection))

# # 3) Filter the *mean* average expression to these columns.
# cytokine_expression_df_filtered = cytokine_expression_df_filtered_mean.loc[ALL_CELL_TYPES, columns_intersection]

#### Calculate family counts

In [ ]:
cytokines_expressed_anywhere = []
for l in expressed_cytokines_by_cell_type_all.values():
    cytokines_expressed_anywhere.extend(l)
cytokines_expressed_anywhere = list(set(cytokines_expressed_anywhere))
len(cytokines_expressed_anywhere)

cytokine_expression_df_filtered = cytokine_expression_df_mean[cytokines_expressed_anywhere]

In [ ]:
maximum_cell_type_count = {cell_type: 0 for cell_type in ALL_CELL_TYPES}
maximum_cell_type_identity = {}
maximum_cell_type_list = []
for column in cytokine_expression_df_filtered.columns:
    maximum_cell_type = cytokine_expression_df_filtered[column].idxmax()
    row = {
        "cell_type": maximum_cell_type,
        "cytokine": column
    }
    maximum_cell_type_list.append(row)
    maximum_cell_type_count[maximum_cell_type] += 1
    maximum_cell_type_identity[column] = maximum_cell_type

maximum_cell_type_df = pd.DataFrame(maximum_cell_type_list)
maximum_cell_type_df["family"] = maximum_cell_type_df["cytokine"].map(cytokine_name_to_level_2_family)
family_value_counts_df = maximum_cell_type_df.groupby("cell_type")["family"].value_counts().unstack().fillna(0)

# Order the family_value_counts_df by ALL_CELL_TYPES
family_value_counts_df = family_value_counts_df.reindex(ALL_CELL_TYPES).dropna()

available_families = family_value_counts_df.columns
ordered_available_families = [family for family in cytokine_level_2_order if family in available_families]
family_value_counts_df = family_value_counts_df[ordered_available_families]
family_value_counts_df = family_value_counts_df.iloc[:, ::-1]

In [ ]:
max_value_counts = maximum_cell_type_df["cell_type"].value_counts()
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    if not cell_type in max_value_counts.index:
        max_value_counts[cell_type] = 0
max_value_counts = max_value_counts[ALL_CELL_TYPES_EXCLUDE]

plt.figure(figsize=cm_to_inches((4.2,3.2)))
plt.bar(max_value_counts.index, max_value_counts.values, color="black", width=0.65)
plt.title("strongest production", fontsize=7)
plt.xticks(rotation=90)
plt.yticks(rotation=90)
plt.xlim(-0.5, len(max_value_counts)-0.5)
plt.ylim(0, 30)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4_max_cell_type_count.{format}"), dpi=300)

In [ ]:
any_value_counts = expressed_cytokines_by_cell_type_all_df.unstack().to_frame().reset_index()
any_value_counts = any_value_counts.rename(columns={0: "expressed", "level_0": "cytokine"})
any_value_counts["level1_family"] = any_value_counts["cytokine"].map(cytokine_name_to_level_1_family)

any_level1_family_count = any_value_counts.groupby(["cell_type", "level1_family"])["expressed"].sum().to_frame().reset_index()
relevant_families = any_level1_family_count.groupby("level1_family")["expressed"].sum()
relevant_families = relevant_families[relevant_families > 0].index
any_level1_family_count = any_level1_family_count[any_level1_family_count["level1_family"].isin(relevant_families)]

# create a pivot table to get counts by cell type and level1_family
pivot_df = any_level1_family_count.pivot(index="cell_type", columns="level1_family", values="expressed").fillna(0)
pivot_df = pivot_df.reindex(ALL_CELL_TYPES_EXCLUDE).fillna(0)
ordered_families = sorted(pivot_df.columns)

# plotting
ax = pivot_df[ordered_families[::-1]].plot(kind='bar', stacked=True, figsize=cm_to_inches((4.2,4)), width=0.65)
plt.xlabel("")
plt.title("expressed cytokines", fontsize=7)
plt.xticks(rotation=90)
plt.yticks(rotation=90)
plt.tick_params(axis='x', which='both', length=0)

plt.tight_layout()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4_cytokine_family_by_cell_type.{format}"), dpi=300)

In [ ]:
# Transpose the dataframe to plot by cytokine family instead of cell type
family_counts_by_cell_type = family_value_counts_df.T

# Plot each cell type as a separate bar segment
fig, ax = plt.subplots(figsize=cm_to_inches((8, 8)))
bottom = np.zeros(len(family_counts_by_cell_type.index))
for cell_type in family_counts_by_cell_type.columns:
    ax.barh(family_counts_by_cell_type.index, family_counts_by_cell_type[cell_type], 
           left=bottom, label=cell_type, color=CELL_TYPE_COLORS.get(cell_type, 'gray'))
    bottom += family_counts_by_cell_type[cell_type]

ax.set_xlabel('Maximum production per family')
ax.set_ylabel('')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xlim(0, 16)
plt.tight_layout()

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4_maximum_expression_by_family.{format}"), dpi=300)

In [ ]:
# Calculate the relative cpm across cell types (again for the mean expression)
cytokine_expression_df_rel = cytokine_expression_df_filtered.div(cytokine_expression_df_filtered.max(axis=0), axis=1)

# sort the columns by the maximum cell type identity
maximum_cell_type_identity = pd.DataFrame([(k,v) for k,v in maximum_cell_type_identity.items()], columns=['gene', 'cell_type'])

# sort cell type
maximum_cell_type_identity["cell_type"] = pd.Categorical(maximum_cell_type_identity["cell_type"], categories=ALL_CELL_TYPES, ordered=True)
maximum_cell_type_identity = maximum_cell_type_identity.sort_values(by="cell_type")
cytokine_expression_df_rel = cytokine_expression_df_rel.loc[ALL_CELL_TYPES, maximum_cell_type_identity["gene"]]

In [ ]:
# plot in two separate groups (whole plot is too big)
cytokine_groups_here = [['interleukin', 'interferon', 'TNF family', 'complement', 'growth factor', 'other'], ["chemokine"]]
for group_index, family in enumerate(cytokine_groups_here):
    cytokine_expression_df_rel_family = cytokine_expression_df_rel.loc[:, cytokine_info["family_level_1"].isin(family)]
    if group_index == 0:
        plt.figure(figsize=cm_to_inches((6, 16)))
    else:
        plt.figure(figsize=cm_to_inches((6, 10)))

    ax = sns.heatmap(cytokine_expression_df_rel_family.T, cmap="Grays", vmin=0, vmax=1,
                     cbar_kws={"ticks": [0, 0.2, 0.4, 0.6, 0.8, 1], 
                              "label": "relative mean expression", 
                              "shrink": 0.25}, 
                     square=False)


    plt.yticks(rotation=0)
    ax.tick_params(axis='y', labelsize=6)
    tick_positions = np.arange(len(cytokine_expression_df_rel_family.columns))+0.5
    labels = [introduce_greek_letters(label) for label in cytokine_expression_df_rel_family.columns]
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(labels)
    
    ax.tick_params(axis='y', labelsize=6, length=0)
    ax.tick_params(axis='x', length=0)

    plt.xlabel("")
    plt.tight_layout()
    for format in ["png", "svg"]:
        plt.savefig(os.path.join(plot_folder, f"4_average_cytokine_production_by_cell_type_{group_index}.{format}"), 
                    bbox_inches="tight",
                    dpi=300)

In [ ]:
# small sample plot for the main figure
cytokine_expression_df_rel_family = cytokine_expression_df_rel.loc[:, ["IL-32-beta", "CD40L", "IFN-gamma", "TNF-alpha", "IL-10"]]
plt.figure(figsize=cm_to_inches((4.525, 3)))

ax = sns.heatmap(cytokine_expression_df_rel_family.T, cmap="Grays", vmin=0, vmax=1,
                    cbar=False,
                    square=True,
                    linewidths=0,  # Add lines between cells
                    linecolor='black')  # Make lines black


plt.yticks(rotation=0)  # Make labels horizontal
ax.tick_params(axis='y', labelsize=6)
tick_positions = np.arange(len(cytokine_expression_df_rel_family.columns))+0.5
labels = [introduce_greek_letters(label) for label in cytokine_expression_df_rel_family.columns]
ax.set_yticks(tick_positions)
ax.set_yticklabels(labels)
ax.tick_params(axis='y', labelsize=6, length=0)
ax.tick_params(axis='x', length=0)
plt.xlabel("")

# black box
ax.spines['top'].set_visible(True)
ax.spines['right'].set_visible(True)
ax.spines['bottom'].set_visible(True)
ax.spines['left'].set_visible(True)
for spine in ax.spines.values():
    spine.set_color('black')
    spine.set_linewidth(1)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4_average_cytokine_production_by_cell_type_particular.{format}"), 
                bbox_inches="tight",
                dpi=300)

# 5 - Signaling Analysis

## 5.1 - Cytokine to cytokine signaling

In [ ]:
plot_folder = os.path.join(base_plot_folder, "5.1_cytokine_to_cytokine_production")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# constrain to all cytokines that have a calculated fold change for all subunits (here as gene lists per cytokine)
all_cytokines_in_measured_genes = [cytokine for cytokine in cytokine_info["gene"] if all([subunit in edger_df_padj.index for subunit in cytokine.split(", ")])]
cytokines_not_in_measured_genes = [cytokine for cytokine in cytokine_info["gene"] if cytokine not in all_cytokines_in_measured_genes] # for debugging

# get the padj and log2FoldChange values for all necessary genes
all_cytokines_in_measured_genes_flat = [subunit for cytokine in all_cytokines_in_measured_genes for subunit in cytokine.split(", ")]
edger_df_padj_cytokine = edger_df_padj.loc[edger_df_padj.index.isin(all_cytokines_in_measured_genes_flat), :].fillna(1)
edger_df_log2FoldChange_cytokine = edger_df_log2FoldChange.loc[edger_df_padj.index.isin(all_cytokines_in_measured_genes_flat), :].fillna(0)

In [ ]:
cytokine_response_df_all = pd.DataFrame(columns=["cell_type", "treatment", "response", "p_value", "fold_changes"])

for cell_type in ALL_CELL_TYPES:
    for cytokine_treatment in CYTOKINE_ORDER:
        for cytokine_response in all_cytokines_in_measured_genes:
            individual_cytokines = cytokine_response.split(", ")
            
            # skip this response if not all subunits are present (should already have been filtered?)
            skip = False
            for individual_cytokine in individual_cytokines:
                if not individual_cytokine in edger_df_padj_cytokine.index:
                    skip = True
            if skip:
                continue
            
            # max for p value (conservative)
            p_value = max([edger_df_padj_cytokine.loc[individual_cytokine, (cell_type, cytokine_treatment)].astype(float) for individual_cytokine in individual_cytokines])
            # list for fold changes
            fold_changes = [edger_df_log2FoldChange_cytokine.loc[individual_cytokine, (cell_type, cytokine_treatment)].astype(float) for individual_cytokine in individual_cytokines]
            if p_value > 0.05: # only keep significant responses (regardless of fc for now)
                continue
            
            if all(fc < 0 for fc in fold_changes):
                single_fc = min(fold_changes)
            elif all(fc > 0 for fc in fold_changes):
                single_fc = max(fold_changes)
            else:
                single_fc = 0.0 # mixed response
            
            new_row = pd.DataFrame({
                "cell_type": [cell_type], 
                "treatment": [cytokine_treatment], 
                "response": [cytokine_response],
                "p_value": [p_value],
                "fold_changes": [fold_changes],
                "single_fc": [single_fc]
            })
            cytokine_response_df_all = pd.concat([cytokine_response_df_all, new_row], axis=0)

In [ ]:
# convert back to actual cytokine names
cytokine_response_df_all.loc[:, "response"] = cytokine_response_df_all["response"].map(gene_to_cytokine)

# remove duplicates (basically just LTA, LTB because they have the same genes)
print(len(cytokine_response_df_all))
cytokine_response_df_all = cytokine_response_df_all.drop_duplicates(ignore_index=True, subset=["cell_type", "treatment", "response"])
print(len(cytokine_response_df_all))

# add the direction of the regulation
direction = []
for index, row in cytokine_response_df_all.iterrows():
    if all([fc > 1 for fc in row["fold_changes"]]):
        direction.append("upregulated")
    elif all([fc < -1 for fc in row["fold_changes"]]):
        direction.append("downregulated")
    else:
        direction.append("neither")
        
cytokine_response_df_all["direction"] = direction

In [ ]:
# save the results
cytokine_response_df_all.to_csv(os.path.join(output_tables_folder, "Table_4.2_cytokine-cytokine-crosstalk-all.csv"), index=False)

# filter to parse cytokines and save that subset
cytokine_response_df_all_filter = cytokine_response_df_all[cytokine_response_df_all["response"].isin(cytokine_info_parse.index)]
cytokine_response_df_all_filter.to_csv(os.path.join(output_tables_folder, "5.1_cytokine-cytokine-crosstalk-parse.csv"), index=False)

In [ ]:
df = cytokine_response_df_all.copy()
df['treatment'] = df['treatment'].apply(lambda x: introduce_greek_letters(x))
df['response'] = df['response'].apply(lambda x: introduce_greek_letters(x))
df = df.rename(columns={
    'cell_type': 'Cell Type',
    'treatment': 'Cytokine Treatment',
    'response': 'Response Cytokine',
    'fold_changes': 'Log2 Fold Change by subunit',
    'p_value': 'Adjusted P-Value',
    'direction': 'Direction',
    'single_fc': 'Overall Log2 Fold Change'
}
)
write_to_SI_table(df, "cytokine-cytokine-crosstalk")

In [ ]:
cytokine_response_df_all_filter = cytokine_response_df_all[cytokine_response_df_all['direction'].isin(['upregulated', 'downregulated'])]
counts = cytokine_response_df_all_filter.groupby(['treatment'])['direction'].value_counts().unstack().fillna(0)
counts['max'] = counts.max(axis=1)
counts = counts.sort_values(by='max', ascending=False).drop(columns=['max'])

up = counts['upregulated']
down = -counts['downregulated']

fig, ax = plt.subplots(figsize=(6, 3))

x = np.arange(len(counts))
ax.bar(x, up, color='red', label='Upregulated')
ax.bar(x, down, color='blue', label='Downregulated')

ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(counts.index, rotation=90, fontsize=6)
ax.set_ylabel('Number of responses')
ax.legend(frameon=False, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
def cm_to_px(cm):
    return cm * 27.6

def create_sankey_diagram_strength(
    df,
    column1_name=None, column1_order=None,
    column2_name=None, column2_order=None,
    column3_name=None, column3_order=None,
    height_add=0,
    color1="rgba(31, 119, 180, 0.8)",
    color2="rgba(44, 160, 44, 0.8)",
    color3="rgba(255, 127, 14, 0.8)",
    title=None,
    link_color_df=None,
    color_column="color",
    middle_pos_value=None,
):
    """
    Create a Sankey diagram with preserved node ordering and proper column spacing.
    Optionally color specific connections using `link_color_df`.
    """

    # get order of values in the columns
    column1_values = pd.unique(df[column1_name]).tolist()
    column2_values = pd.unique(df[column2_name]).tolist()
    column3_values = pd.unique(df[column3_name]).tolist()
    
    # Use provided orders or by the order of the values in the dataframe
    if column1_order is not None:
        missing_column1_values = set(df[column1_name].unique()) - set(column1_order)
        if missing_column1_values:
            raise ValueError(f"Column1 order missing values: {missing_column1_values}")
        column1_values = [value for value in column1_order if value in df[column1_name].unique()]
        
    if column2_order is not None:
        missing_column2_values = set(df[column2_name].unique()) - set(column2_order)
        if missing_column2_values:
            raise ValueError(f"Column2 order missing values: {missing_column2_values}")
        column2_values = [value for value in column2_order if value in df[column2_name].unique()]
        
    if column3_order is not None:
        missing_column3_values = set(df[column3_name].unique()) - set(column3_order)
        if missing_column3_values:
            raise ValueError(f"Column3 order missing values: {missing_column3_values}")
        column3_values = [value for value in column3_order if value in df[column3_name].unique()]

    # Create mapping dictionaries with ordered indices
    column1_to_idx = {t: idx for idx, t in enumerate(column1_values)}
    column2_to_idx = {c: idx + len(column1_values) for idx, c in enumerate(column2_values)}
    column3_to_idx = {r: idx + len(column1_values) + len(column2_values) for idx, r in enumerate(column3_values)}
    
    # --- NEW: build explicit edge color map if provided
    edge_color_map = {}
    if link_color_df is not None and len(link_color_df) > 0:
        for _, row in link_color_df.iterrows():
            if color_column not in row or pd.isna(row[color_column]):
                continue
            clr = str(row[color_column])

            # map (column1 -> column2) if both present
            if (column1_name in row and column2_name in row and
                pd.notna(row[column1_name]) and pd.notna(row[column2_name])):
                t = row[column1_name]
                c = row[column2_name]
                if t in column1_to_idx and c in column2_to_idx:
                    edge_color_map[(column1_to_idx[t], column2_to_idx[c])] = clr

            # map (column2 -> column3) if both present
            if (column2_name in row and column3_name in row and
                pd.notna(row[column2_name]) and pd.notna(row[column3_name])):
                c = row[column2_name]
                r = row[column3_name]
                if c in column2_to_idx and r in column3_to_idx:
                    edge_color_map[(column2_to_idx[c], column3_to_idx[r])] = clr

    # Calculate link values while preserving order
    links = []
    link_colors = []

    # default/metadata colors (used only if link_color_df is None)
    meta_to_color = {
        'upregulation': 'rgba(220, 50, 50, 0.45)',
        'upregulated':  'rgba(220, 50, 50, 0.45)',
        'downregulation': 'rgba(50, 100, 220, 0.45)',
        'downregulated':  'rgba(50, 100, 220, 0.45)',
    }
    default_link_color = 'rgba(0,0,0,0.15)'

    # First column to second column (treatments to cell types)
    for t in column1_values:
        for c in column2_values:
            count = len(df[(df[column1_name] == t) & (df[column2_name] == c)])
            if count > 0:
                s = column1_to_idx[t]
                tg = column2_to_idx[c]
                links.append({'source': s, 'target': tg, 'value': count})

                # Use explicit color if provided, else default
                link_colors.append(edge_color_map.get((s, tg), default_link_color))

    # Second column to third column (cell types to responses)
    use_explicit = len(edge_color_map) > 0  # if we have any explicit mapping, we use single aggregated edges
    has_metadata = ('metadata' in df.columns) and not use_explicit

    for c in column2_values:
        for r in column3_values:
            sub = df[(df[column2_name] == c) & (df[column3_name] == r)]
            if len(sub) == 0:
                continue

            s = column2_to_idx[c]
            tg = column3_to_idx[r]

            if use_explicit:
                # aggregate into a single edge and color via explicit map (or default)
                links.append({'source': s, 'target': tg, 'value': int(len(sub))})
                link_colors.append(edge_color_map.get((s, tg), default_link_color))
            elif has_metadata:
                # split by metadata (original behavior)
                vc = sub['metadata'].astype(str).value_counts()
                for meta_value, cnt in vc.items():
                    clr = meta_to_color.get(meta_value.lower(), default_link_color)
                    links.append({'source': s, 'target': tg, 'value': int(cnt)})
                    link_colors.append(clr)
            else:
                # fallback single neutral edge
                links.append({'source': s, 'target': tg, 'value': int(len(sub))})
                link_colors.append(default_link_color)

    # Extract sources, targets, and values from links
    sources = [link['source'] for link in links]
    targets = [link['target'] for link in links]
    values  = [link['value']  for link in links]

    # Create color scheme
    n_column1_values = len(column1_values)
    n_column2_values = len(column2_values)
    n_column3_values = len(column3_values)
    
    colors = (
        [color1] * n_column1_values +
        [color2] * n_column2_values +
        [color3] * n_column3_values
    )
    
    # Create node labels in the correct order
    display_labels = column1_values + column2_values + column3_values
    
    # Calculate node positions
    if middle_pos_value:
        x_positions = [0.0]*n_column1_values + [middle_pos_value]*n_column2_values + [1.0]*n_column3_values
    elif n_column1_values > n_column3_values:
        x_positions = [0.0]*n_column1_values + [0.5]*n_column2_values + [1.0]*n_column3_values
    else:
        x_positions = [0.0]*n_column1_values + [0.3]*n_column2_values + [1.0]*n_column3_values
    
    # Create y positions that spread nodes evenly within their columns
    y_column1_values = np.linspace(0.05, 0.95, n_column1_values) if n_column1_values > 1 else [0.5]
    y_column2_values = np.linspace(0.05, 0.95, n_column2_values) if n_column2_values > 1 else [0.5]
    y_column3_values = np.linspace(0.05, 0.95, n_column3_values) if n_column3_values > 1 else [0.5]
    y_positions = list(y_column1_values) + list(y_column2_values) + list(y_column3_values)
    
    # Create Sankey Diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=5,
            thickness=10,
            line=dict(color='black', width=0.5),
            label=display_labels,
            color=colors,
            x=x_positions,
            y=y_positions,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            color=link_colors,
            hovertemplate='Source: %{source.label}<br>Target: %{target.label}<br>Value: %{value}<extra></extra>',
            line=dict(color='rgba(0,0,0,0)', width=0)
        ),
        arrangement='snap'
    )])
    
    height = cm_to_px((max(n_column1_values, n_column2_values, n_column3_values)+height_add)/3.3)
    width = cm_to_px(4)
    fig.update_layout(
        height=height,
        width=width,
        margin=dict(l=2, r=2, t=4, b=4),
        showlegend=False,
        font=dict(family="Helsana", size=6, color="black"),
        title=title
    )
    return fig, height, width

In [ ]:
# special IL-32-beta plot
treatment_order = CYTOKINE_ORDER
response_order = ALL_CYTOKINES_ORDER

treatment_order = [introduce_greek_letters(treatment) for treatment in treatment_order]
response_order = [introduce_greek_letters(response) for response in response_order]

# make sure the fold-change is sufficiently large for all gene products
cytokine_response_df_all_curr = cytokine_response_df_all[cytokine_response_df_all["direction"] != "neither"]
cytokine_response_df_all_curr = cytokine_response_df_all_curr[cytokine_response_df_all_curr["single_fc"].abs() >= 2]
cytokine_response_df_all_curr.loc[:, "response"] = cytokine_response_df_all_curr["response"].apply(lambda x: introduce_greek_letters(x))
cytokine_response_df_all_curr.loc[:, "treatment"] = cytokine_response_df_all_curr["treatment"].apply(lambda x: introduce_greek_letters(x))

current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["treatment"] == "IL-32-β"].copy() 
current_df = current_df[current_df['cell_type'].isin(["CD14 Mono", "CD16 Mono", "cDC"])].copy() # myeloid only

link_color_df = []
for index, row in current_df.iterrows():
    if row['direction'] == 'upregulated':
        link_color_df.append({
            'cell_type': row['cell_type'],
            'response': row['response'],
            'color': 'rgba(220, 50, 50, 0.45)'  # soft red
        })
    elif row['direction'] == 'downregulated':
        link_color_df.append({
            'cell_type': row['cell_type'],
            'response': row['response'],
            'color': 'rgba(50, 100, 220, 0.45)'  # soft blue
        })
link_color_df = pd.DataFrame(link_color_df)

current_df['metadata'] = current_df['direction']
fig, height, width = create_sankey_diagram_strength(
    df=current_df,
    column1_name="treatment",
    column1_order=treatment_order,
    column2_name="cell_type",
    column3_name="response",
    link_color_df=link_color_df,
    column3_order=response_order,
    color1 = "rgba(44, 160, 44, 0.8)",
    color2 = "rgba(31, 119, 180, 0.8)",
    color3 = "rgba(44, 160, 44, 0.8)",
    title=None,
    height_add=-15
)

fig.write_image(
    os.path.join(plot_folder, f"5.1_cytokine_{unidecode("IL-32-beta")}_regulates_others.svg"),
    width=width,
    height=height
    )

In [ ]:
treatment_order = CYTOKINE_ORDER
response_order = ALL_CYTOKINES_ORDER

treatment_order = [introduce_greek_letters(treatment) for treatment in treatment_order]
response_order = [introduce_greek_letters(response) for response in response_order]

# make sure the fold-change is sufficiently large for all gene products
cytokine_response_df_all_curr = cytokine_response_df_all[cytokine_response_df_all["direction"] != "neither"]
cytokine_response_df_all_curr.loc[:, "response"] = cytokine_response_df_all_curr["response"].apply(lambda x: introduce_greek_letters(x))
cytokine_response_df_all_curr.loc[:, "treatment"] = cytokine_response_df_all_curr["treatment"].apply(lambda x: introduce_greek_letters(x))

# now, do this separately for every cytokine
current_plot_folder = os.path.join(plot_folder, "5.1_regulates_others")
os.makedirs(current_plot_folder, exist_ok=True)
for treat_index, current_treatment in enumerate(treatment_order):
    current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["treatment"] == current_treatment]
    if len(current_df) < 4:
        continue
    
    link_color_df = []
    for index, row in current_df.iterrows():
        if row['direction'] == 'upregulated':
            link_color_df.append({
                'cell_type': row['cell_type'],
                'response': row['response'],
                'color': 'rgba(220, 50, 50, 0.45)' # red
            })
        elif row['direction'] == 'downregulated':
            link_color_df.append({
                'cell_type': row['cell_type'],
                'response': row['response'],
                'color': 'rgba(50, 100, 220, 0.45)' # blue
            })
    link_color_df = pd.DataFrame(link_color_df)
    
    fig, height, width = create_sankey_diagram_strength(
        df=current_df,
        column1_name="treatment",
        column1_order=treatment_order,
        column2_name="cell_type",
        column3_name="response",
        column3_order=response_order,
        link_color_df=link_color_df,
        color1 = "rgba(44, 160, 44, 0.8)",
        color2 = "rgba(31, 119, 180, 0.8)",
        color3 = "rgba(44, 160, 44, 0.8)",
        title=None
    )
    
    treat_index_str = str(treat_index).zfill(2)
    fig.write_image(
        os.path.join(current_plot_folder, f"5.1_cytokine_{treat_index_str}_{unidecode(current_treatment)}_regulates_others.svg"),
        width=width,
        height=height
        )

current_plot_folder = os.path.join(plot_folder, "5.1_is_upregulated_by_others")
os.makedirs(current_plot_folder, exist_ok=True)
for current_response in cytokine_response_df_all_curr["response"].unique():
    current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["response"] == current_response]
    current_df = current_df[current_df['direction'] == 'upregulated']
    if len(current_df) < 6:
        continue
    fig, height, width = create_sankey_diagram_strength(
        df=current_df,
        column1_name="treatment",
        column1_order=treatment_order,
        column2_name="cell_type",
        column3_name="response",
        column3_order=response_order,
        color1 = "rgba(44, 160, 44, 0.8)",
        color2 = "rgba(31, 119, 180, 0.8)",
        color3 = "rgba(44, 160, 44, 0.8)",
        title=None
    )
    fig.write_image(
        os.path.join(current_plot_folder, f"5.1_cytokine_{unidecode(current_response)}_is_upregulated_by_others.svg"),
        width=width,
        height=height
    )

In [ ]:
# overview plots
cytokine_response_df_all_current = cytokine_response_df_all[cytokine_response_df_all["direction"].isin(["upregulated", "downregulated"])]

# treatments
treatment_counts = cytokine_response_df_all_current["treatment"].value_counts()
treatment_counts = treatment_counts[treatment_counts > 4]

plt.figure(figsize=cm_to_inches((7, 2.4)))
plt.bar(treatment_counts.index, treatment_counts.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.ylabel('outgoing\nsignaling\nresponses')
apply_greek_labels_heatmap(plt.gca())
plt.margins(x=0.02)
plt.xticks(fontsize=5.9)
plt.tick_params(axis='x', which='both', length=0)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "5.1_stimulation_counts.svg"), bbox_inches="tight")

# responses
response_counts = cytokine_response_df_all_current["response"].value_counts()
response_counts = response_counts[response_counts > 4]

plt.figure(figsize=cm_to_inches((12, 3)))
plt.bar(response_counts.index, response_counts.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.ylabel('incoming\nsignaling responses')
apply_greek_labels_heatmap(plt.gca())
plt.margins(x=0.02)
plt.xticks(fontsize=5.5)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "5.1_response_counts.svg"), bbox_inches="tight")

### More detailed IL-32-beta signaling analysis

In [ ]:
def robust_z_score(df):
    median = df.median(axis=1)
    mad = (df.sub(median, axis=0)).abs().median(axis=1)
    robust_z = df.sub(median, axis=0).div(1.4826 * mad.replace(0, np.nan), axis=0)
    return robust_z

monocyte_diff_marker_panels = {
    "moDC markers": [
        "CD1A",    # lipid antigen presentation
        "CD1C",    # classical moDC marker
        "ITGAM",   # Cd11b
        "HLA-DRA", # MHC-II expansion
        "CIITA",   # MHC-II transactivator
    ],
    "macrophage markers": [ 
        "CD14",    # monocyte/macrophage marker
        "MERTK",   # 
        "MAFB",    # transcription factor for macrophage differentiation
        "CD68",    # macrophage marker
        "CD163",   # macrophage marker
    ],
}

robust_vals_by_panel = {}
for panel_name, genes in monocyte_diff_marker_panels.items():
    plt.figure(figsize=cm_to_inches((10, 8)))
    # we here use raw values because the filters for edgeR means that some genes are missing for some conditions
    # this makes it hard to compute z-scores properly
    robust_vals_by_panel[panel_name] = robust_z_score(np.log2(raw_fc_values.loc[genes]["CD14 Mono"]))
    vals = np.log2(raw_fc_values).loc[
        genes
    ]["CD14 Mono"][
        [
        "IL-4",
        "GM-CSF",
        "M-CSF",
        "IL-10",
        "C5a",
        "IFN-beta",
        "IL-32-beta",
        "VEGF"
        ]
    ]
    
    plt.figure(figsize=cm_to_inches((6, 3.5)))
    sns.heatmap(vals, center=0, cmap="vlag", annot=True, fmt=".1f", cbar_kws={"label": "log2 Fold Change"})
    plt.ylabel(panel_name)
    plt.xlabel("")
    plt.tick_params(axis='x', rotation=90, length=0)
    plt.tick_params(axis='y', rotation=0, length=0)
    apply_greek_labels_heatmap(plt.gca())
    plt.savefig(os.path.join(plot_folder, f"heatmap_{panel_name.replace(' ', '_')}.svg"))

for panel in robust_vals_by_panel:
    vals = robust_vals_by_panel[panel]
    score = vals.mean(axis=0).sort_values(ascending=False)
    top5 = score.head(5)
    bottom5 = score.tail(5)
    
    plt.figure(figsize=cm_to_inches((4, 4.5)))
    indices = np.arange(len(top5) + len(bottom5))
    plt.barh(
        indices[:5],
        top5.values,
        color="tab:green",
    )
    plt.barh(
        indices[5:],
        bottom5.values,
        color="tab:red"
    )
    plt.axvline(0, color="black", linewidth=0.8)
    plt.gca().invert_yaxis()
    plt.yticks(indices, list(top5.index) + list(bottom5.index))
    plt.xlabel("robust Z-score")
    plt.title(f"{panel}\n{', '.join(monocyte_diff_marker_panels[panel])}", fontsize=6.5)
    apply_greek_labels_heatmap(plt.gca())
    plt.savefig(os.path.join(plot_folder, f"barplot_{panel.replace(' ', '_')}.svg"))

In [ ]:
IL32b_response = cytokine_response_df_all[cytokine_response_df_all["response"] == "IL-32"]
IL32b_response.to_csv(os.path.join(output_tables_folder, "Table_4.3_IL-32-beta_response.csv"), index=False)
IL32b_treatment = cytokine_response_df_all[cytokine_response_df_all["treatment"] == "IL-32-beta"]
IL32b_treatment.to_csv(os.path.join(output_tables_folder, "Table_4.3_IL-32-beta_treatment.csv"), index=False)

In [ ]:
IL32b_treatment_myeloid = IL32b_treatment[IL32b_treatment['cell_type'].isin(['cDC', 'CD14 Mono', 'CD16 Mono'])]

In [ ]:
# have a look at all chemokines
chemokines = cytokine_info[cytokine_info['family_level_1'] == "chemokine"].index
detected_in_CD14_Mono = edger_df_log2FoldChange['CD14 Mono'].notna().any(axis=1)
detected_in_CD14_Mono = detected_in_CD14_Mono[detected_in_CD14_Mono].index
chemokines_detected_in_CD14Mono = [c for c in chemokines if c in detected_in_CD14_Mono]
edger_df_log2FoldChange.loc[chemokines_detected_in_CD14Mono]["CD14 Mono"]["IL-32-beta"].sort_values()

In [ ]:
# get median fold-change for specific cytokines (called out in main text)
il32_treat = cytokine_response_df_all[cytokine_response_df_all['treatment'] == 'IL-32-beta']
il32_treat = il32_treat[il32_treat['cell_type'] == 'CD14 Mono']

print(il32_treat[il32_treat['response'].isin(["CXCL1", "CXCL2", "CXCL3", "CXCL5", "CXCL6", "IL-8", "IL-1-alpha", "IL-1-beta"])]['single_fc'].median())
print(il32_treat[il32_treat['response'].isin(["CXCL9", "CXCL10", "CXCL11", "IL-18"])]['single_fc'].median())

In [ ]:
# analyze consensus signaling responses induced by IL-32-beta
chemokine_panels = {
    "Neutrophil": ["CXCL1", "CXCL2", "CXCL3", "CXCL5", "CXCL6", "IL-8"], # CXCR1/2 ligands (direct type III response)
    "Th1": ["CXCL9", "CXCL10", "CXCL11"], # CXCR3 ligands (Th1, CD8+ T cells, NK cells)
    "Th2": ["CCL17", "CCL22"], # CCR4 ligands (Th2 cells, Tregs)
    "type I": ["CCL3", "CCL3L1", "CCL4", "CCL5"],
    "mixed": ["CCL7", "CCL8"],
    "type II": ["CCL13"]
}
other_cytokines = {
    "Survival factors": ["APRIL", "TRAIL"],
    "Growth factors": ["HGF", "PDGF-CC", "BMP6", "VEGF", "TGF-alpha"],
    "IL-1 family": ["IL-1-alpha", "IL-1-beta", "IL-18"],
    "IL-6 family": ["IL-6", "OSM"],
    "IL-10 family": ["IL-10", "IL-19", "IL-24"]
}

for plot_index, response_categories in enumerate([chemokine_panels, other_cytokines]):
    response_to_category = {}
    for cat, responses in response_categories.items():
        for resp in responses:
            response_to_category[resp] = cat

    # Filter IL32b_treatment to only those responses in our categories
    plot_df = cytokine_response_df_all[cytokine_response_df_all['treatment'] == 'IL-32-beta']
    plot_df = plot_df[plot_df['response'].isin(response_to_category.keys())].copy()
    plot_df['category'] = plot_df['response'].map(response_to_category)

    # Prepare x axis
    category_order = list(response_categories.keys())
    response_order = []
    response_to_x = {}
    x_tick_labels = []
    x_tick_positions = []
    x_pos = 0
    for cat in category_order:
        responses = response_categories[cat]
        for resp in responses:
            response_order.append(resp)
            response_to_x[resp] = x_pos
            x_tick_labels.append(resp)
            x_tick_positions.append(x_pos)
            x_pos += 1
        x_pos += 1

    # Plot
    plt.figure(figsize=cm_to_inches((8, 4)))
    offset_width = 0.12
    
    cell_types = ['cDC', 'CD14 Mono', 'CD16 Mono']
    for i, cell_type in enumerate(cell_types):
        cell_type_df = plot_df[plot_df['cell_type'] == cell_type]
        for resp in cell_type_df['response'].unique():
            resp_df = cell_type_df[cell_type_df['response'] == resp]
            x = response_to_x[resp] + (i - (len(cell_types)-1)/2)*offset_width
            plt.scatter(
                np.full(len(resp_df), x),
                resp_df['single_fc'],
                color=CELL_TYPE_COLORS[cell_type],
                label=cell_type if resp == cell_type_df['response'].unique()[0] else "",
                alpha=1,
                s=10,
                edgecolor='none',
                linewidth=0.5
            )
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.8, zorder=0)
    
    # draw category labels
    category_centers = []
    category_labels = []
    last = 0
    for cat in category_order:
        responses = response_categories[cat]
        if len(responses) == 0:
            continue
        first_x = response_to_x[responses[0]]
        last_x = response_to_x[responses[-1]]
        center = (first_x + last_x) / 2
        category_centers.append(center)
        category_labels.append(cat)
        plt.axvspan(first_x-0.5, last_x+0.5, color='lightgrey', alpha=0.08, zorder=-1)

    plt.xticks(x_tick_positions, x_tick_labels, rotation=90, ha='center')
    apply_greek_labels_heatmap(plt.gca())
    plt.ylabel("log2FC")
    plt.ylim(-8,8)

    # Add category labels above the x-axis
    for i, (center, label) in enumerate(zip(category_centers, category_labels)):
        plt.text(center, plt.ylim()[1] + 1 + 1.5*(i%2), label,
                ha='center', va='bottom', fontsize=6, fontweight='bold', color='black')

    # Legend (one entry per cell type)
    handles = [plt.Line2D([0], [0], marker='o', color='w', label=cell_type,
                        markerfacecolor=CELL_TYPE_COLORS[cell_type], markersize=5, markeredgecolor='black')
            for cell_type in cell_types]
    
    plt.legend(handles=handles, bbox_to_anchor=(1.01, 1), loc='upper left', borderaxespad=0.)
    plt.savefig(os.path.join(plot_folder, f"scatter_IL-32-beta_responses_by_category_{plot_index+1}.svg"))

In [ ]:
# get donor-specific values for the response to IL-32-beta in CD14 Monocytes
donor_set_1 = ["D1", "D3", "D4", "D10"] # interferon group
donor_set_2 = ["D5", "D6", "D7", "D8", "D9", "D11", "D12"] # non-interferon group
donor_sets = donor_set_1 + donor_set_2

for plot_index, response_categories in enumerate([chemokine_panels, other_cytokines]):
    # Calculate means and standard deviations for each gene group
    means_set_1 = []
    stds_set_1 = []
    means_set_2 = []
    stds_set_2 = []
    
    for gene_group in response_categories.values():
        # Get values for all genes in the group
        group_vals = []
        all_vals = np.log2(get_donor_vals_ratios('IL-32-beta', "CD14 Mono"))
        for gene in gene_group:
            gene = cytokine_to_gene.get(gene, gene)
            vals = all_vals.loc[gene]
            group_vals.append(vals)
        
        # Calculate mean and std for each donor set across all genes in group
        group_vals_df = pd.DataFrame(group_vals, index=gene_group, columns=DONOR_NAMES)
        
        # Average over donors for each gene individually
        donor_set_1_means = group_vals_df[donor_set_1].mean(axis=1, skipna=True)
        donor_set_2_means = group_vals_df[donor_set_2].mean(axis=1, skipna=True)
        donor_set_1_stds = group_vals_df[donor_set_1].std(axis=1, skipna=True)
        donor_set_2_stds = group_vals_df[donor_set_2].std(axis=1, skipna=True)
        
        # Store individual gene values instead of group averages
        means_set_1.extend(donor_set_1_means.values)
        stds_set_1.extend(donor_set_1_stds.values)
        means_set_2.extend(donor_set_2_means.values)
        stds_set_2.extend(donor_set_2_stds.values)

    # Create labels for individual genes
    all_gene_names = []
    for gene_group in response_categories.values():
        all_gene_names.extend(gene_group)

    # Create bar plot
    x = []
    last = 0
    for group in response_categories.values():
        for _ in group:
            x.append(last)
            last += 1
        last += 1  # add gap between groups
    x = np.array(x)
    width = 0.35

    fig, ax = plt.subplots(figsize=cm_to_inches((10, 5)))

    bars1 = ax.bar(x - width/2, means_set_1, width, yerr=stds_set_1, 
                label='interferon group', capsize=0, alpha=1, color='royalblue')
    bars2 = ax.bar(x + width/2, means_set_2, width, yerr=stds_set_2, 
                label='non-interferon group', capsize=0, alpha=1, color='forestgreen')

    ax.set_ylabel('log2FC')
    ax.set_xticks(x)
    ax.set_xticklabels(all_gene_names, rotation=90)
    apply_greek_labels_heatmap(ax)
    ax.legend()

    # add group labels at the top
    y_max = max(max(means_set_1), max(means_set_2)) + max(max(stds_set_1), max(stds_set_2))
    y_label_height = y_max * 1.1

    # calculate group positions
    group_positions = []
    bars_so_far = 0
    for gene_group in response_categories.values():
        group_size = len(gene_group)
        group_center = (x[bars_so_far] + x[bars_so_far+group_size-1]) / 2
        group_positions.append((group_center, x[bars_so_far], x[bars_so_far + group_size - 1]))
        bars_so_far += group_size

    # add group labels
    names = list(response_categories.keys())
    for i, (center_pos, start_pos, end_pos) in enumerate(group_positions):
        ax.text(center_pos, y_label_height, names[i], 
                ha='center', va='bottom', fontweight='bold', fontsize=7)
        # Add horizontal line to show group span
        ax.plot([start_pos - 0.4, end_pos + 0.4], [y_label_height * 0.95, y_label_height * 0.95], 
                'k-', linewidth=1)
        plt.axvspan(start_pos-0.5, end_pos+0.5, color='lightgrey', alpha=0.08, zorder=-1)
        
    ax.set_ylim(bottom=ax.get_ylim()[0], top=y_label_height * 1.2)
    ax.axhline(y=0, color='black', linewidth=1, linestyle='-')

    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"cytokine_il32_by_donor_group_{plot_index+1}.svg"))

# CONTINUE FROM HERE

## 5.2 - Cell - cell communication

In [ ]:
plot_folder = os.path.join(base_plot_folder, "5.2_cell_cell_communication")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# Get number of expressed tested cytokines for each cell type (max: 70)
num_cytokines = {cell_type: len(cytokines) for cell_type, cytokines in expressed_cytokines_by_cell_type_parse.items()}
num_cytokines = pd.Series(num_cytokines)[ALL_CELL_TYPES]

plt.figure(figsize=cm_to_inches((4,6)))
plt.barh(num_cytokines.index, num_cytokines.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.ylabel('Number of expressed cytokines')
plt.show()

In [ ]:
cell_type_secretion_df = pd.DataFrame(columns=["origin_cell_type", "secretion"])
for origin_cell_type, expressed_cytokines in expressed_cytokines_by_cell_type_parse.items():
    for cytokine in expressed_cytokines:
        cell_type_secretion_df = pd.concat([cell_type_secretion_df,
                                            pd.DataFrame({"origin_cell_type": origin_cell_type, "secretion": cytokine}, index=[0])], axis=0)

In [ ]:
# get strong impact cytokines by cell type
all_normalized_distances_high = strong_impact_by_cell_type.copy()
all_normalized_distances_high["cytokine"] = all_normalized_distances_high.index
all_normalized_distances_high = all_normalized_distances_high.melt(id_vars="cytokine", var_name="cell_type", value_name="significant")
all_normalized_distances_high = all_normalized_distances_high[all_normalized_distances_high["significant"]]

In [ ]:
secretion_to_response_df = pd.DataFrame(columns=["origin_cell_type", "secretion", "response_cell_type"])
for index, row in cell_type_secretion_df.iterrows():
    secretion = row["secretion"]
    response_cell_types = all_normalized_distances_high[all_normalized_distances_high["cytokine"] == secretion]
    for index, response_row in response_cell_types.iterrows():
        secretion_to_response_df = pd.concat(
            [secretion_to_response_df,
            pd.DataFrame({"origin_cell_type": row["origin_cell_type"], "secretion": secretion, "response_cell_type": response_row["cell_type"]}, index=[0])
            ],axis=0)
secretion_to_response_df = secretion_to_response_df.reset_index(drop=True)

In [ ]:
secretion_to_response_df['mean_expression'] = None
secretion_to_response_df['max_expression'] = None
secretion_to_response_df['maximal_treatment'] = None
secretion_to_response_df['response_magnitude'] = None

for index, row in secretion_to_response_df.iterrows():
    condition_for_max = cytokine_expression_df_all_details.loc[(row["origin_cell_type"], slice(None)), row["secretion"]]
    condition_for_max = condition_for_max[row["origin_cell_type"]].sort_values(ascending=False).index[0]
    secretion_to_response_df.loc[index, 'maximal_treatment'] = condition_for_max
    secretion_to_response_df.loc[index, 'mean_expression'] = cytokine_expression_df_mean.loc[row["origin_cell_type"], row["secretion"]]
    secretion_to_response_df.loc[index, 'max_expression'] = cytokine_expression_df_max.loc[row["origin_cell_type"], row["secretion"]]
    secretion_to_response_df.loc[index, 'response_magnitude'] = all_normalized_distances_mean.loc[row['secretion'], row['response_cell_type']]

# we don't actually have isotype information
secretion_to_response_df['secretion'] = secretion_to_response_df['secretion'].replace("IL-32-β", "IL-32")

# save the dataframe
secretion_to_response_df.to_csv(os.path.join(output_tables_folder, "Table_4.1_cell_cell_communication_df.csv"), index=False)

In [ ]:
df = secretion_to_response_df.copy()
df['secretion'] = df['secretion'].apply(lambda x: introduce_greek_letters(x))
df = df.rename(columns={
    'origin_cell_type': 'Origin Cell Type',
    'secretion': 'Secreted Cytokine',
    'response_cell_type': 'Responding Cell Type',
    'mean_expression': 'Mean Expression Across Conditions (cpm)',
    'max_expression': 'Max Expression Across Conditions (cpm)',
    'maximal_treatment': 'Condition of Max Expression',
    'response_magnitude': 'Response Magnitude'
}
)
write_to_SI_table(df, "cell-cell-communication")

In [ ]:
secretion_order = CYTOKINE_ORDER
secretion_order = [introduce_greek_letters(secretion).replace("IL-32-β", "IL-32") for secretion in secretion_order]
secretion_to_response_df["secretion"] = secretion_to_response_df["secretion"].apply(lambda x: introduce_greek_letters(x))
secretion_to_response_df["secretion"] = secretion_to_response_df["secretion"].replace("IL-32-β", "IL-32")
cell_type_order = ALL_CELL_TYPES

cell_type = "cDC"
current_df = secretion_to_response_df[secretion_to_response_df["origin_cell_type"] == cell_type]
column1_name = "origin_cell_type"
column2_name = "secretion"
column3_name = "response_cell_type"



fig, height, width = create_sankey_diagram_strength(
    df=current_df,
    column1_name=column1_name,
    column1_order=cell_type_order,
    column2_name=column2_name,
    column2_order=secretion_order,
    column3_name=column3_name,
    column3_order=cell_type_order,
    color1 = "rgba(31, 119, 180, 0.8)",
    color2 = "rgba(44, 160, 44, 0.8)", 
    color3 = "rgba(31, 119, 180, 0.8)",
    title=None,
    height_add=1.5,
    link_color_df=link_color_df,
)

fig.write_image(
    os.path.join(plot_folder, f"5.2_cell_type_{unidecode(cell_type)}_secretes.svg"),
    width=width,
    height=height
    )

In [ ]:
secretion_order = CYTOKINE_ORDER
secretion_order = [introduce_greek_letters(secretion).replace("IL-32-β", "IL-32") for secretion in secretion_order]
secretion_to_response_df["secretion"] = secretion_to_response_df["secretion"].apply(lambda x: introduce_greek_letters(x))
secretion_to_response_df["secretion"] = secretion_to_response_df["secretion"].replace("IL-32-β", "IL-32")
cell_type_order = ALL_CELL_TYPES

for cell_type in ALL_CELL_TYPES:
    if cell_type == "cDC":
        link_color_df = []
        for index, row in current_df.iterrows():
            if row['secretion'] == "IL-15":
                link_color_df.append({
                    'origin_cell_type': row['origin_cell_type'],
                    'response_cell_type': row['response_cell_type'],
                    'secretion': row['secretion'],
                    'color': 'rgba(220, 50, 50, 0.45)'  # soft red
                })
        link_color_df = pd.DataFrame(link_color_df)
    else:
        link_color_df = None
        
    current_df = secretion_to_response_df[secretion_to_response_df["origin_cell_type"] == cell_type]
    column1_name = "origin_cell_type"
    column2_name = "secretion"
    column3_name = "response_cell_type"

    fig, height, width = create_sankey_diagram_strength(
        df=current_df,
        column1_name=column1_name,
        column1_order=cell_type_order,
        column2_name=column2_name,
        column2_order=secretion_order,
        column3_name=column3_name,
        column3_order=cell_type_order,
        color1 = "rgba(31, 119, 180, 0.8)",
        color2 = "rgba(44, 160, 44, 0.8)", 
        color3 = "rgba(31, 119, 180, 0.8)",
        title=None,
        height_add=1.5,
        link_color_df=link_color_df,
    )

    fig.write_image(
        os.path.join(plot_folder, f"5.2_cell_type_{unidecode(cell_type)}_secretes.svg"),
        width=width,
        height=height
        )

In [ ]:
secretion_connection_counts = secretion_to_response_df.groupby(["origin_cell_type", "response_cell_type"]).size()
secretion_connection_counts = secretion_connection_counts.to_frame().reset_index().pivot(index="origin_cell_type", columns="response_cell_type", values=0)
secretion_connection_counts = secretion_connection_counts.loc[ALL_CELL_TYPES, ALL_CELL_TYPES]

# signaling counts plot
cell_cell_signaling_counts = secretion_to_response_df.groupby("origin_cell_type")["response_cell_type"].value_counts().unstack().fillna(0)
cell_cell_signaling_counts = cell_cell_signaling_counts.loc[ALL_CELL_TYPES, ALL_CELL_TYPES]

fig, ax = plt.subplots(figsize=cm_to_inches((8, 6)))
bottom = np.zeros(len(cell_cell_signaling_counts.index))
for cell_type in cell_cell_signaling_counts.columns:
    ax.barh(cell_cell_signaling_counts.index, cell_cell_signaling_counts[cell_type], 
           left=bottom, label=cell_type, color=CELL_TYPE_COLORS.get(cell_type, 'gray'))
    bottom += cell_cell_signaling_counts[cell_type]

ax.set_xlabel('')
ax.set_ylabel('')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5.2_cell_cell_communication_counts.{format}"), dpi=300)

### Calculate cell-cell communication strength

In [ ]:
# get cytokines with at least one strong response
some_strong_effect = strong_impact_by_cell_type.apply(lambda row: any(row), axis=1)
some_strong_effect = some_strong_effect[some_strong_effect].index

# subset the cytokine expression to that
cytokine_expression_df_rel_reorganized = cytokine_expression_df_rel.T[ALL_CELL_TYPES]
cytokine_expression_df_rel_reorganized = cytokine_expression_df_rel_reorganized[cytokine_expression_df_rel_reorganized.index.isin(some_strong_effect)]

# also get absolute expression values
cytokine_expression_df_mean_reorganized = cytokine_expression_df_mean.T[ALL_CELL_TYPES]
cytokine_expression_df_mean_reorganized = cytokine_expression_df_mean_reorganized[cytokine_expression_df_mean_reorganized.index.isin(some_strong_effect)]
# only keep those with at least 10 cpm in at least one cell type (to avoid relative values for very low expressed cytokines)
cytokine_expression_df_mean_reorganized = cytokine_expression_df_mean_reorganized[cytokine_expression_df_mean_reorganized.max(axis=1) >= 10]

# reorganize distances for merging
all_distances_mean_reorganized = all_normalized_distances_mean.loc[cytokine_expression_df_rel_reorganized.index, ALL_CELL_TYPES]

In [ ]:
df1 = cytokine_expression_df_rel_reorganized.reset_index().melt(id_vars="index").rename(
    columns={
        "index": "gene",
        "value": "value_sender", # relative expression of the cytokine in the sender cell type
        "cell_type": "cell_type_sender"}
    )

df2 = all_distances_mean_reorganized.reset_index().melt(id_vars="index").rename(
    columns={
        "index": "gene",
        "value": "value_receiver", # response magnitude in the receiver cell type
        "variable": "cell_type_receiver"}
    )

df_mean = cytokine_expression_df_mean_reorganized.reset_index().melt(id_vars="index").rename(
    columns={
        "index": "gene",
        "value": "mean_expression", # mean expression of the cytokine in the sender cell type
        "cell_type": "cell_type_sender"}
    )

df1 = df1.merge(
    df_mean,
    on=['gene', 'cell_type_sender'],
    how='inner'
)

merged_df = df1.merge(
    df2,
    on='gene',
    how='inner'
)

# subset to sufficient mean expression
merged_df = merged_df[merged_df['mean_expression'] >= 4].copy()
merged_df["score"] = merged_df["value_sender"] * merged_df["value_receiver"]

In [ ]:
plt.figure(figsize=cm_to_inches((5,3)))

hist_data, bin_edges = np.histogram(merged_df["score"], bins=np.arange(0,1.05,0.05))
log_hist_data = np.log10(hist_data + 1)
plt.bar(bin_edges[:-1], log_hist_data, width=0.05, align='edge', color="skyblue", edgecolor="black")

yticks = [0, 1, 2, 3]
plt.yticks(yticks, [f"$10^{i}$" if i > 0 else "0" for i in yticks])
plt.xlabel("interaction score")
plt.ylabel(r"log$_{10}$(count)")

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5.2_interaction_strength_histogram.{format}"), dpi=400)

In [ ]:
cell_type_communication_strength = merged_df.groupby(["cell_type_sender", "cell_type_receiver"])["score"].sum()
cell_type_communication_strength = cell_type_communication_strength.to_frame().reset_index().pivot(
    index="cell_type_sender",
    columns="cell_type_receiver",
    values="score")
cell_type_communication_strength = cell_type_communication_strength.loc[ALL_CELL_TYPES, ALL_CELL_TYPES]

In [ ]:
fig, ax = plt.subplots(figsize=cm_to_inches((5, 5)))
sns.heatmap(cell_type_communication_strength, vmin=0, vmax=6, cmap="viridis", ax=ax, square=True,
    cbar_kws={"label": "interaction score", "shrink": 0.7})
ax.set_xlabel("receiver cell type")
ax.set_ylabel("sender cell type")

In [ ]:
# Sort merged_df by score within each unique_pair
merged_df["gene"] = merged_df["gene"].apply(introduce_greek_letters)
merged_df["unique_pair"] = merged_df["cell_type_sender"] + "-" + merged_df["cell_type_receiver"]
merged_df = merged_df.sort_values(by=["unique_pair", "score"], ascending=[True, False])

In [ ]:
# no isotype information
merged_df["gene"] = merged_df["gene"].str.replace("IL-32-β", "IL-32")

In [ ]:
def smart_join(gene_series, max_list_length=5):
    # Convert pandas Series to list of genes
    genes = list(gene_series)
    
    if not len(genes):
        return ""
    
    max_line_length = 5  # adjust this threshold as needed
    current_line = []
    current_length = 0
    all_lines = []
    
    for gene in genes[:max_list_length]:
        # Calculate length if we add this gene (including comma and space)
        additional_length = len(gene) + (2 if current_line else 0)  # +2 for ", "
        
        # If adding this gene would exceed threshold
        if current_line and (current_length + additional_length > max_line_length):
            # Store current line and start new one
            all_lines.append(", ".join(current_line))
            current_line = [gene]
            current_length = len(gene)
        else:
            # Add to current line
            current_line.append(gene)
            current_length += additional_length
    
    # Don't forget the last line
    if current_line:
        all_lines.append(", ".join(current_line))
    
    return "\n".join(all_lines)

def top_sum(score_series, top_n=None):
    # use all of the scores by default
    score_series = score_series.sort_values(ascending=False)
    if top_n is not None:
        return score_series.iloc[:top_n].sum()
    else:
        return score_series.sum()

score_cutoff = 0.25

# First, get unique genes for each sender-receiver pair that pass the cutoff (for text annotations)
interaction_genes = merged_df[merged_df["score"] > score_cutoff].groupby(
    ["cell_type_sender", "cell_type_receiver"]
)["gene"].agg(lambda x: smart_join(x)).reset_index()

# Create a pivot table for the scores (will be used for color intensity)
score_matrix = merged_df.groupby(
    ["cell_type_sender", "cell_type_receiver"]
)["score"].agg(lambda x: top_sum(x)).reset_index()
score_pivot = score_matrix.pivot(
    index="cell_type_sender", 
    columns="cell_type_receiver", 
    values="score"
)
score_pivot = score_pivot.fillna(0)

plt.figure(figsize=cm_to_inches((16, 16)))
ax = plt.gca()
sns.heatmap(score_pivot, cmap='YlOrBr_r', annot=False, vmin=0, vmax=3,
    cbar_kws={'label': 'sum of interaction scores (cell-cell connection strength)', 'shrink': 0.3, 'pad': 0})

# Text annotations
for sender in score_pivot.index:
    for receiver in score_pivot.columns:
        genes = interaction_genes[
            (interaction_genes["cell_type_sender"] == sender) & 
            (interaction_genes["cell_type_receiver"] == receiver)
        ]["gene"].values
        
        if len(genes) > 0:
            text = genes[0] if len(genes) > 0 else ""
            score = score_pivot.loc[sender, receiver]
            alpha = min(score / score_cutoff, 1.0) if pd.notnull(score) else 0.3
            
            # Add text with adjusted position and smaller size
            plt.text(
                score_pivot.columns.get_loc(receiver) + 0.5,
                score_pivot.index.get_loc(sender) + 0.5,
                text,
                color='black' if score > 0.75 else "white",
                ha='center',
                va='center',
                alpha=alpha,
                fontsize=5.5,
                bbox=dict(
                    pad=3,
                    facecolor='none',
                    edgecolor='none',
                    alpha=0
                ),
                wrap=True
            )

# formatting
plt.tick_params(axis='both', which='both', length=0)
ax.tick_params(axis='both', which='both', length=0)
plt.xticks(rotation=90, fontsize=6.7)
plt.yticks(rotation=0, fontsize=6.7)
plt.xlabel("Receiver", fontsize=8)
plt.ylabel("Sender", fontsize=8)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5.2_detailed_cell_cell_interactions.{format}"), dpi=300)

In [ ]:
# How does IL-32 mediate cell-cell communication?
merged_df_il32 = merged_df[merged_df["gene"] == "IL-32"]
merged_df_il32 = merged_df_il32[merged_df_il32["score"] > 0.25]

secretion_order = CYTOKINE_ORDER
secretion_order = [introduce_greek_letters(secretion.replace("IL-32-beta", "IL-32")) for secretion in secretion_order]

secretion_to_response_df["secretion"] = secretion_to_response_df["secretion"].apply(lambda x: introduce_greek_letters(x))
cell_type_order = ALL_CELL_TYPES

column1_name = "cell_type_sender"
column2_name = "gene"
column3_name = "cell_type_receiver"

link_color_df = []
for index, row in merged_df_il32.iterrows():
    for index, row in merged_df_il32.iterrows():
        alpha = float(row.get('value_sender', 0.0))
        alpha = max(0.05, min(alpha, 1.0))  # minimum visibility, max 1.0
        link_color_df.append({
            'gene': row['gene'],
            'cell_type_sender': row['cell_type_sender'],
            'color': f'rgba(255,0,0,{alpha*0.75})'  # gray with alpha proportional to value_sender
        })
        
        alpha = float(row.get('value_receiver', 0.0))
        alpha = max(0.05, min(alpha, 1.0))  # minimum visibility, max 1.0
        link_color_df.append({
            'cell_type_receiver': row['cell_type_receiver'],
            'gene': row['gene'],
            'color': f'rgba(255,0,0,{alpha*0.75})'  # gray with alpha proportional to value_receiver
        })
link_color_df = pd.DataFrame(link_color_df)

fig, height, width = create_sankey_diagram_strength(
    df=merged_df_il32,
    column1_name=column1_name,
    column1_order=cell_type_order,
    column2_name=column2_name,
    column2_order=secretion_order,
    column3_name=column3_name,
    column3_order=cell_type_order,
    color1 = "rgba(31, 119, 180, 0.8)",
    color2 = "rgba(44, 160, 44, 0.8)", 
    color3 = "rgba(31, 119, 180, 0.8)",
    title=None,
    height_add=3,
    link_color_df=link_color_df,
    middle_pos_value=0.4
)

fig.write_image(
    os.path.join(plot_folder, f"5.2_IL32_comm.svg"),
    width=width,
    height=height
    )

In [ ]:
# Plot overall interaction sankey diagram
# Create a pivot table for the scores (will be used for color intensity)
score_matrix = merged_df[merged_df["score"] > score_cutoff].groupby(
    ["cell_type_sender", "cell_type_receiver"]
)["score"].agg(lambda x: top_sum(x)).reset_index()

# Calculate max and min connection strength for each sender cell type
max_values = score_matrix.groupby("cell_type_sender")["score"].max().reset_index()
min_values = score_matrix.groupby("cell_type_sender")["score"].min().reset_index()

# Merge the max and min values back to the original dataframe
score_matrix = score_matrix.merge(max_values, on="cell_type_sender", suffixes=("", "_max"))
score_matrix = score_matrix.merge(min_values, on="cell_type_sender", suffixes=("", "_min"))

score_matrix["connection_strength"] = score_matrix["score"] / score_matrix["score_max"]

In [ ]:
import matplotlib.colors as mcolors
def to_int_rgb(rgb_tuple):
    return tuple(int(255 * comp) for comp in rgb_tuple)
int_rgb = {name: to_int_rgb(mcolors.to_rgb(val)) for name, val in CELL_TYPE_COLORS.items()}

def tuple_to_rgba_string(color):
    if isinstance(color, str):
        return color
    color = f"rgba({color[0]}, {color[1]}, {color[2]}, 0.8)"
    return color

def create_sankey_diagram_two_column(df, column1_name=None, column1_order=None, column2_name=None, column2_order=None,
                          strength_column=None, color1 = "rgba(31, 119, 180, 0.8)", color2 = "rgba(44, 160, 44, 0.8)", 
                          link_color_scheme = (31, 119, 180),
                          title="Cytokine crosstalk between treatments and responses"):
    """
    Create a Sankey diagram with two columns and connection strengths.
    
    Parameters:
    df: pandas DataFrame with three columns: two entity columns and one strength column
    column1_name: name of the first column
    column1_order: list of column1 names in desired order (optional)
    column2_name: name of the second column
    column2_order: list of column2 names in desired order (optional)
    color1: color for first column (can be string or dict mapping values to colors)
    color2: color for second column (can be string or dict mapping values to colors)
    title: title of the diagram
    """
    
    # Get order of values in the columns
    column1_values = pd.unique(df[column1_name]).tolist()
    column2_values = pd.unique(df[column2_name]).tolist()
    
    if column1_order is not None:
        # Ensure that all nodes in the DataFrame are included in the order
        extra = set(df[column1_name].unique()) - set(column1_order)
        if extra:
            raise ValueError(f"Column1 order missing values for: {extra}")
        column1_values = column1_order  # use the full provided order even if some nodes are absent from df
        
    if column2_order is not None:
        # Ensure that all nodes in the DataFrame are included in the order
        extra = set(df[column2_name].unique()) - set(column2_order)
        if extra:
            raise ValueError(f"Column1 order missing values for: {extra}")
        column2_values = column2_order  # use the full provided order even if some nodes are absent from df
        
    # Create mapping dictionaries with ordered indices
    column1_to_idx = {t: idx for idx, t in enumerate(column1_values)}
    column2_to_idx = {c: idx + len(column1_values) for idx, c in enumerate(column2_values)}
    
    # Calculate link values while preserving order
    links = []
    
    # Get all strength values for normalization
    all_strengths = df[strength_column].values
    min_strength = all_strengths.min()
    max_strength = all_strengths.max()

    for t in column1_values:
        for c in column2_values:
            strength = df[(df[column1_name] == t) & (df[column2_name] == c)][strength_column].values
            if len(strength) > 0:
                # Normalize strength to opacity range [0.1, 0.8]
                normalized_strength = (strength[0] - min_strength) / (max_strength - min_strength)
                opacity = 0.1 + (0.9 * normalized_strength)  # Scale to [0.2, 1.0]
                
                links.append({
                    'source_label': t,
                    'source': column1_to_idx[t],
                    'target': column2_to_idx[c],
                    'value': strength[0],
                    'opacity': opacity
                })

    # Extract sources, targets, values and opacities from links
    sources = [link['source'] for link in links]
    targets = [link['target'] for link in links]
    values = [link['value'] for link in links]
    opacities = [link['opacity'] for link in links]

    # Handle colors (either string or dictionary)
    if isinstance(color1, dict):
        colors_column1 = [tuple_to_rgba_string(color1.get(val, "rgba(31, 119, 180, 0.8)")) for val in column1_values]
    else:
        colors_column1 = [color1] * len(column1_values)
        
    if isinstance(color2, dict):
        colors_column2 = [tuple_to_rgba_string(color2.get(val, "rgba(44, 160, 44, 0.8)"))
                          for val in column2_values]
    else:
        colors_column2 = [color2] * len(column2_values)

    # Combine colors for all nodes
    colors = colors_column1 + colors_column2
    display_labels = [
        "" for lbl in column1_values
    ] + [
        "" for lbl in column2_values
    ]
    
    # Calculate node positions
    x_positions = [0.05] * len(column1_values) + [0.95] * len(column2_values)
    
    # Create y positions that spread nodes evenly within their columns
    y_column1_values = np.linspace(0.05, 0.95, len(column1_values)) if len(column1_values) > 1 else [0.5]
    y_column2_values = np.linspace(0.05, 0.95, len(column2_values)) if len(column2_values) > 1 else [0.5]
    y_positions = list(y_column1_values) + list(y_column2_values)
    
    # Compute link colors using the computed opacity values
    if type(link_color_scheme) == tuple:
        link_colors_base = [link_color_scheme for link in links]
    else:
        link_colors_base = [link_color_scheme[link['source_label']] for link in links]
    
    link_colors = [f"rgba({base_link_color[0]}, {base_link_color[1]}, {base_link_color[2]}, {opac})" for base_link_color, opac in zip(link_colors_base, opacities)]

    # print(list(zip(x_positions, y_positions, display_labels)))
    
    # Create Sankey Diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=5,
            thickness=10,
            line=dict(color='black', width=0.5),
            label=display_labels,
            color=colors,
            x=x_positions,
            y=y_positions,
        ),
            link=dict(
            source=sources,
            target=targets,
            value=values,
            hovertemplate='Source: %{source.label}<br>Target: %{target.label}<br>Strength: %{value}<extra></extra>',
            line=dict(color='rgba(0,0,0,0)', width=0),
            color=link_colors
        ),
        arrangement='fixed'
    )])
    
    height = cm_to_px(4.92)
    width = cm_to_px(2.6)
    # Dynamic height scaling and improved margins
    fig.update_layout(
        height=180,
        width=90,
        margin=dict(
            l=2,  # Left margin
            r=2,  # Right margin
            t=3,  # Top margin
            b=3   # Bottom margin to prevent cutoff
        ),
        showlegend=False,
        font=dict(
            color='black',
            size=6,
            family='Helvetica'
            ),
        title=title
    )
    return fig, height, width

fig, height, width = create_sankey_diagram_two_column(
    score_matrix.copy(),
    column1_name="cell_type_sender",
    column1_order=ALL_CELL_TYPES.copy(),
    color1=int_rgb,
    column2_name="cell_type_receiver",
    column2_order=ALL_CELL_TYPES.copy(),
    color2=int_rgb, # color2 = "rgba(128, 128, 128, 0.8)",
    strength_column="connection_strength",
    link_color_scheme=int_rgb,
    title=None
)

fig.write_image(
    os.path.join(plot_folder, f"5.2_cell_cell_connection.svg"),
    width=width,
    height=height
    )

## 5.3 - Family - family communication

In [ ]:
plot_folder = os.path.join(base_plot_folder, "5.3_family_counts")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
family_level = "level2"

if family_level == "level2":
    cytokine_response_df_all["family"] = cytokine_response_df_all["response"].map(cytokine_name_to_level_2_family)
elif family_level == "level1":
    cytokine_response_df_all["family"] = cytokine_response_df_all["response"].map(cytokine_name_to_level_1_family)
else:
    raise Exception("Invalid level choice!")

cytokine_response_df_all["response_plus_cell_type"] = cytokine_response_df_all["response"] + "_" + cytokine_response_df_all["cell_type"]

In [ ]:
family_value_counts_positive = {}
for treatment in cytokine_response_df_all["treatment"].unique():
    response_df_filter = cytokine_response_df_all[cytokine_response_df_all["treatment"] == treatment]
    response_df_filter = response_df_filter[response_df_filter["fold_changes"].apply(lambda x: all(fc > 1 for fc in x))]
    family_value_counts_positive[treatment] = response_df_filter["family"].value_counts()
    
family_value_counts_positive = pd.DataFrame(family_value_counts_positive)
family_value_counts_positive.fillna(0, inplace=True)
family_value_counts_positive = family_value_counts_positive.astype(int)

family_value_counts_negative = {}
for treatment in cytokine_response_df_all["treatment"].unique():
    response_df_filter = cytokine_response_df_all[cytokine_response_df_all["treatment"] == treatment]
    response_df_filter = response_df_filter[response_df_filter["fold_changes"].apply(lambda x: all(fc < -1 for fc in x))]
    family_value_counts_negative[treatment] = response_df_filter["family"].value_counts()
    
family_value_counts_negative = pd.DataFrame(family_value_counts_negative)
family_value_counts_negative.fillna(0, inplace=True)
family_value_counts_negative = family_value_counts_negative.astype(int)

In [ ]:
# inspect the results manually
subset = cytokine_response_df_all[cytokine_response_df_all["family"] == "IL-10"]
subset = subset[subset["direction"] == "upregulated"]
subset["treatment"].value_counts()

In [ ]:
# add 0 counts for all missing families
if family_level == "level2":
    family_value_counts_positive = family_value_counts_positive.reindex(cytokine_level_2_order, fill_value=0)
    family_value_counts_negative = family_value_counts_negative.reindex(cytokine_level_2_order, fill_value=0)
if family_level == "level1":
    family_value_counts_positive = family_value_counts_positive.reindex(cytokine_level_1_order, fill_value=0)
    family_value_counts_negative = family_value_counts_negative.reindex(cytokine_level_1_order, fill_value=0)

In [ ]:
# total counts by cytokine treatment
total_significant_interactions_cytokine = family_value_counts_positive + family_value_counts_negative
total_significant_interactions_cytokine = total_significant_interactions_cytokine.sum(axis=0)

plt.figure(figsize=cm_to_inches((4, 3)))
plt.hist(total_significant_interactions_cytokine, bins=np.arange(0, 250, 10))
plt.xlabel("significant changes")
plt.ylabel("stimulation conditions")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_total_significant_interactions_cytokine.svg"), bbox_inches="tight")

In [ ]:
# total counts by cytokine response family
total_significant_interactions_family = family_value_counts_positive + family_value_counts_negative
total_significant_interactions_family = total_significant_interactions_family.sum(axis=1)

plt.figure(figsize=cm_to_inches((4, 3)))
plt.hist(total_significant_interactions_family, bins=np.arange(0, 250, 10))
plt.xlabel("significant changes")
plt.ylabel("family response")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_total_significant_interactions_family.svg"), bbox_inches="tight")

In [ ]:
# filter to those with more than 3 interactions for plotting purposes
filter_cytokine_interactions = total_significant_interactions_cytokine[total_significant_interactions_cytokine > 3]
filter_family_interactions = total_significant_interactions_family[total_significant_interactions_family > 3]
family_value_counts_positive = family_value_counts_positive.loc[filter_family_interactions.index, filter_cytokine_interactions.index]
family_value_counts_negative = family_value_counts_negative.loc[filter_family_interactions.index, filter_cytokine_interactions.index]

# order columns by all_known_cytokine_and_chemokine_names
family_value_counts_positive = family_value_counts_positive.reindex(columns=[col for col in CYTOKINE_ORDER if col in family_value_counts_positive.columns])
family_value_counts_negative = family_value_counts_negative.reindex(columns=[col for col in CYTOKINE_ORDER if col in family_value_counts_negative.columns])
family_value_counts_both = family_value_counts_positive - family_value_counts_negative
family_value_counts_abs = family_value_counts_positive + family_value_counts_negative

family_value_counts_positive_rel = family_value_counts_positive.divide(family_value_counts_positive.max(axis=1), axis=0)
family_value_counts_negative_rel = family_value_counts_negative.divide(family_value_counts_negative.max(axis=1), axis=0)
family_value_counts_both_rel = family_value_counts_both.divide(abs(family_value_counts_both).max(axis=1), axis=0)

In [ ]:
# This is done to plot overview labels on the y-axis
counts_per_level1_family = {}
for level2_family in family_value_counts_positive_rel.index:
    level1_family = level2_to_level1.get(level2_family)
    if level1_family is not None:
        counts_per_level1_family[level1_family] = counts_per_level1_family.get(level1_family, 0) + 1
        
# invert the order of the dictionary
counts_per_level1_family = dict(reversed(counts_per_level1_family.items()))        

In [ ]:
if family_level == "level2":
    fig = plt.figure(figsize=cm_to_inches((14, 8)))
else:
    fig = plt.figure(figsize=cm_to_inches((12, 10)))

ax = sns.heatmap(family_value_counts_positive_rel, vmin=0, vmax=1, cmap="Reds",
            cbar_kws={"label": "relative number of upregulated cytokines", "shrink": 0.6 if family_level == "level2" else 0.15}, square=True)

plot_overview_labels_along_y_axis(fig, ax, counts_per_level1_family.keys(),
                                  counts_per_level1_family, -10, len(family_value_counts_positive_rel.index), stagger=1.5)

plt.xlabel("cytokine treatment")
plt.ylabel("")
plt.gca().tick_params(axis='both', length=0)
apply_greek_labels_heatmap(plt.gca())

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_family_value_counts_positive_{family_level}.svg"), bbox_inches="tight")

In [ ]:
if family_level == "level2":
    fig = plt.figure(figsize=cm_to_inches((14, 8)))
else:
    fig = plt.figure(figsize=cm_to_inches((12, 10)))

ax = sns.heatmap(family_value_counts_negative_rel, vmin=0, vmax=1, cmap="Blues",
            cbar_kws={"label": "relative number of downregulated cytokines", "shrink": 0.6 if family_level == "level2" else 0.15}, square=True)


plot_overview_labels_along_y_axis(fig, ax, counts_per_level1_family.keys(),
                                  counts_per_level1_family, -10, len(family_value_counts_positive_rel.index), stagger=1.5)

plt.ylabel("")
apply_greek_labels_heatmap(plt.gca())
plt.xlabel("cytokine treatment")
plt.gca().tick_params(axis='both', length=0)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_family_value_counts_negative_{family_level}.svg"), bbox_inches="tight")


In [ ]:
if family_level == "level2":
    fig = plt.figure(figsize=cm_to_inches((12, 7)))
else:
    fig = plt.figure(figsize=cm_to_inches((12, 10)))

df = family_value_counts_both_rel

ax = sns.heatmap(df, vmin=df.min().min(), vmax=df.max().max(), cmap="coolwarm",
            cbar_kws={"label": "relative number of regulated cytokines", "shrink": 0.6 if family_level == "level2" else 0.15}, square=True)


plot_overview_labels_along_y_axis(fig, ax, counts_per_level1_family.keys(),
                                  counts_per_level1_family, -10, len(family_value_counts_positive_rel.index), stagger=1.5, fontsize=6.5)


plt.xticks(fontsize=6)
plt.yticks(fontsize=6)
plt.ylabel("")
apply_greek_labels_heatmap(plt.gca())
plt.xlabel("cytokine treatment")
plt.gca().tick_params(axis='both', length=0)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_family_value_counts_both_{family_level}.svg"), bbox_inches="tight")

# 6 - Cytokine similarity and grouping (Fig. 5a-c)

In [ ]:
plot_folder = os.path.join(base_plot_folder, "6.1_within_cell_similarity")
os.makedirs(plot_folder, exist_ok=True)

from scipy.spatial.distance import pdist, squareform
use_spatial_gene_set = False
spatial_gene_set = {}

In [ ]:
# can verify that spatial gene sets do not change the results significantly despite having a much smaller number of genes
# spatial_gene_set = pd.read_csv("../annotations/spatial_genes.csv")["gene_name"].tolist()
# use_spatial_gene_set = False

# cytokine_genes = cytokine_info_parse["gene"]
# cytokine_genes_in_spatial = []
# for gene in cytokine_genes:
#     gene = gene.split(", ")
#     present = all([g in spatial_gene_set for g in gene])
#     if present:
#         cytokine_genes_in_spatial.append(', '.join(gene) if len(gene) > 1 else gene[0])
# print(len(cytokine_genes_in_spatial))

In [ ]:
# FOR DEBUGGING/TESTING THE SIMILARITY CODE
cell_type = "CD14 Mono"
all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
existing_fold_change_vals = edger_df_log2FoldChange[cell_type].dropna(how='all').index
all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]
if use_spatial_gene_set:
    all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in spatial_gene_set]
cell_type_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, cell_type]

cytokine1 = "GM-CSF"
cytokine2 = "IL-3"

cell_data_1 = cell_type_log2[cytokine1].dropna()
cell_data_2 = cell_type_log2[cytokine2].dropna()

common_index = cell_data_1.index.intersection(cell_data_2.index)
cell_data_1 = cell_data_1[common_index]
cell_data_2 = cell_data_2[common_index]

r = np.corrcoef(cell_data_1, cell_data_2)[0, 1]
r

In [ ]:
cytokine_combs = list(combinations_with_replacement(CYTOKINE_ORDER, 2))

if os.path.exists(os.path.join(intermediate_output_folder, f"6.1_r_vals_by_cell_type{'_spatial' if use_spatial_gene_set else ''}.pkl")):
    with open(os.path.join(intermediate_output_folder, f"6.1_r_vals_by_cell_type{'_spatial' if use_spatial_gene_set else ''}.pkl"), "rb") as f:
        r_vals_by_cell_type = pickle.load(f)
else:
    r_vals_by_cell_type = {}
    cosine_distances_by_cell_type = {}
    euclidean_distances_by_cell_type = {}
    
    for cell_type in ALL_CELL_TYPES:
        r_vals_by_cytokine = pd.DataFrame(index=[cytokine for cytokine in CYTOKINE_ORDER],
                                             columns=[cytokine for cytokine in CYTOKINE_ORDER])
        cosine_distance_by_cytokine = pd.DataFrame(index=[cytokine for cytokine in CYTOKINE_ORDER],
                                             columns=[cytokine for cytokine in CYTOKINE_ORDER])
        print(cell_type)
        i = 0
        
        # first, use the pre-calculated filter based on expression values
        all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
        
        # then, subset to those where we actually have fold change values for this cell type
        existing_fold_change_vals = edger_df_log2FoldChange[cell_type].dropna(how='all').index
        all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]

        if use_spatial_gene_set:
            all_inclusive_filter = [gene for gene in all_inclusive_filter  if gene in spatial_gene_set]
        
        print(len(all_inclusive_filter))
        cell_type_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, cell_type]
        for cytokine1, cytokine2 in cytokine_combs:
            if cytokine1 == cytokine2:
                r_vals_by_cytokine.loc[cytokine1, cytokine2] = 1.0
                continue
            
            # get values for both cytokines and drop na
            cell_data_1 = cell_type_log2[cytokine1].dropna()
            cell_data_2 = cell_type_log2[cytokine2].dropna()

            if len(cell_data_1) == 0 or len(cell_data_2) == 0:
                r_vals_by_cytokine.loc[cytokine1, cytokine2] = np.nan
                continue
            
            # get the intersection of the two cytokines
            common_index = cell_data_1.index.intersection(cell_data_2.index)
            cell_data_1 = cell_data_1[common_index]
            cell_data_2 = cell_data_2[common_index]
            
            r = np.corrcoef(cell_data_1, cell_data_2)[0, 1]
            r_vals_by_cytokine.loc[cytokine1, cytokine2] = r
            
        # fill in the other half of the correlation matrix
        r_vals_by_cytokine = r_vals_by_cytokine.fillna(r_vals_by_cytokine.T).astype(float)
        r_vals_by_cell_type[cell_type] = r_vals_by_cytokine.copy()

    os.makedirs(intermediate_output_folder, exist_ok=True)
    with open(os.path.join(intermediate_output_folder, f"6.1_r_vals_by_cell_type{'_spatial' if use_spatial_gene_set else ''}.pkl"), "wb") as f:
        pickle.dump(r_vals_by_cell_type, f)

In [ ]:
# Create multi-index DataFrame
index = pd.MultiIndex.from_product([CYTOKINE_ORDER, CYTOKINE_ORDER], names=['cytokine1', 'cytokine2'])
r_vals_by_cell_type_df = pd.DataFrame(index=index, columns=ALL_CELL_TYPES)
for cell_type in ALL_CELL_TYPES:
    r_matrix = r_vals_by_cell_type[cell_type]
    for cyt1 in CYTOKINE_ORDER:
        for cyt2 in CYTOKINE_ORDER:
            r_vals_by_cell_type_df.loc[(cyt1, cyt2), cell_type] = r_matrix.loc[cyt1, cyt2]

### Get correlations for spatial gene subset (Here, we can verify the spatial genes as being meaningful)

In [ ]:
# with open(os.path.join("output", f"6.1_r_vals_by_cell_type.pkl"), "rb") as f:
#     r_vals_by_cell_type_full = pickle.load(f)
# with open(os.path.join("output", f"6.1_r_vals_by_cell_type_spatial.pkl"), "rb") as f:
#     r_vals_by_cell_type_spatial = pickle.load(f)

In [ ]:
# # Create figure with subplots
# fig, axes = plt.subplots(3, 5, figsize=cm_to_inches((16, 10)))
# axes = axes.flatten()

# for i, cell_type in enumerate(ALL_CELL_TYPES):
#     df_full = r_vals_by_cell_type_full[cell_type]
#     df_spatial = r_vals_by_cell_type_spatial[cell_type]

#     df_full = df_full.reset_index().melt(id_vars=["index"]).rename(columns={"index": "cytokine_1", "variable": "cytokine_2", "value": "r"})
#     df_spatial = df_spatial.reset_index().melt(id_vars=["index"]).rename(columns={"index": "cytokine_1", "variable": "cytokine_2", "value": "r"})

#     # drop diagonal
#     df_full = df_full[df_full["cytokine_1"] != df_full["cytokine_2"]]
#     df_spatial = df_spatial[df_spatial["cytokine_1"] != df_spatial["cytokine_2"]]

#     # make identifier
#     df_full["identifier"] = df_full["cytokine_1"] + "_" + df_full["cytokine_2"]
#     df_spatial["identifier"] = df_spatial["cytokine_1"] + "_" + df_spatial["cytokine_2"]

#     # set as index
#     df_full.set_index("identifier", inplace=True)
#     df_spatial.set_index("identifier", inplace=True)

#     # drop na
#     df_full = df_full.dropna()
#     df_spatial = df_spatial.dropna()

#     # harmonize index
#     df_spatial = df_spatial.loc[df_full.index]

#     r = np.corrcoef(df_full["r"], df_spatial["r"])[0, 1]
    
#     # Plot on appropriate subplot
#     axes[i].scatter(df_full["r"], df_spatial["r"], rasterized=True, color="skyblue", s=5, alpha=0.5)
#     axes[i].plot([-1,1], [-1,1], color="black")
#     axes[i].text(0.05, 0.95, f'r = {r:.2f}', transform=axes[i].transAxes, 
#                 verticalalignment='top')
#     axes[i].set_title(cell_type)
#     axes[i].set_xlabel("Pearson r\nall genes")
#     axes[i].set_ylabel("Pearson r\nspatial genes")
#     axes[i].label_outer()
#     axes[i].set_xlim(-1,1)
#     axes[i].set_ylim(-1,1)

# # Remove empty subplots
# for i in range(len(ALL_CELL_TYPES), len(axes)):
#     fig.delaxes(axes[i])

# plt.tight_layout()
# for format in ["png", "svg"]:
#     plt.savefig(os.path.join(plot_folder, f"6.1_full_vs_spatial_correlation.{format}"), dpi=400)

### Explore how correlation values depend on the impact classification

In [ ]:
# Apply logical AND across the index to find cell types where both IL-3 and GM-CSF have significant impact
both_strong = strong_impact_by_cell_type.loc["IL-3", ALL_CELL_TYPES_EXCLUDE] & strong_impact_by_cell_type.loc["GM-CSF", ALL_CELL_TYPES_EXCLUDE]
both_strong_columns = both_strong[both_strong].index
not_both_columns = both_strong[~both_strong].index

In [ ]:
not_both_columns

In [ ]:
print(r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), both_strong_columns].mean())
print(r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), both_strong_columns].std())

In [ ]:
print(r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), not_both_columns].mean())
print(r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), not_both_columns].std())

### create scatter plots

In [ ]:
def get_lower_triangle(df):
    """Get values from the lower triangle of a DataFrame (excluding the diagonal)."""
    return df.where(np.tril(np.ones(df.shape), k=-1).astype(bool))

def find_pairs_thresh(r_vals_by_cytokine, threshold):
    pairs = []
    for cytokine1, cytokine2 in cytokine_combs:
        if threshold > 0:
            if r_vals_by_cytokine.loc[cytokine1, cytokine2] > threshold:
                if cytokine1 != cytokine2:
                    pairs.append((cytokine1, cytokine2))
        if threshold < 0:
            if r_vals_by_cytokine.loc[cytokine1, cytokine2] < threshold:
                if cytokine1 != cytokine2:
                    pairs.append((cytokine1, cytokine2))
    return pairs

In [ ]:
all_pairs_df_high = []
for cell_type in ALL_CELL_TYPES:
    r_vals = r_vals_by_cell_type[cell_type]
    pairs = find_pairs_thresh(r_vals, threshold=0.8)
    # sort the pairs by name
    pairs = sorted(pairs, key=lambda x: (x[0], x[1]))
    for cytokine1, cytokine2 in pairs:
        new_row = {"cell_type": cell_type, "cytokine1": cytokine1, "cytokine2": cytokine2, "r": r_vals.loc[cytokine1, cytokine2]}
        new_row = pd.DataFrame([new_row])
        if len(new_row) > 0:
            all_pairs_df_high.append(new_row)
all_pairs_df_high = pd.concat(all_pairs_df_high, ignore_index=True)

all_pairs_df_low = []
for cell_type in ALL_CELL_TYPES:
    r_vals = r_vals_by_cell_type[cell_type]
    pairs = find_pairs_thresh(r_vals, threshold=-0.4)
    # sort the pairs by name
    pairs = sorted(pairs, key=lambda x: (x[0], x[1]))
    for cytokine1, cytokine2 in pairs:
        new_row = {"cell_type": cell_type, "cytokine1": cytokine1, "cytokine2": cytokine2, "r": r_vals.loc[cytokine1, cytokine2]}
        new_row = pd.DataFrame([new_row])
        if len(new_row) > 0:
            all_pairs_df_low.append(new_row)
all_pairs_df_low = pd.concat(all_pairs_df_low, ignore_index=True)
    
all_pairs_df_high["cytokine_pair"] = all_pairs_df_high["cytokine1"] + "_" + all_pairs_df_high["cytokine2"]
all_pairs_df_low["cytokine_pair"] = all_pairs_df_low["cytokine1"] + "_" + all_pairs_df_low["cytokine2"]       

In [ ]:
pairs_to_plot = [
    ("IL-15", "IL-2"),
    ("IL-4", "IL-13"),
    ("IL-15", "IL-10"),
    ("IL-3", "GM-CSF"),
    ("IL-15", "APRIL"),
    ("IL-15", "VEGF")
]

from scipy.stats import gaussian_kde
corr_plot_folder = os.path.join(plot_folder, "correlation_plots")
os.makedirs(corr_plot_folder, exist_ok=True)

density_plot_flag = False
        
# ------------------------- Large plot -------------------------
fig, axs = plt.subplots(nrows=4, ncols=6, figsize=cm_to_inches((14, 8)), sharex=True, sharey=True)
axs = axs.flatten()
counter = 0

for cell_type in ALL_CELL_TYPES:
    if not cell_type in ["B cell", "CD14 Mono", "cDC", "CD4 T cell"]:
        continue
    print(cell_type)
    
    r_vals = r_vals_by_cell_type[cell_type]
    #pairs = find_pairs_thresh(r_vals, threshold=-0.4)
    #print(cell_type, pairs)
    pairs = pairs_to_plot
    
    all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
    existing_fold_change_vals = edger_df_log2FoldChange[cell_type].dropna(how='all').index
    all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]
    print(len(all_inclusive_filter))
    cell_type_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, cell_type].dropna()
    
    if use_spatial_gene_set:
        all_inclusive_filter = [gene for gene in cell_type_log2.index if gene in spatial_gene_set]
        cell_type_log2_old = cell_type_log2.copy()
        cell_type_log2 = cell_type_log2.loc[all_inclusive_filter]
    
    for cytokine1, cytokine2 in pairs:
        x = cell_type_log2.loc[:, cytokine1]
        y = cell_type_log2.loc[:, cytokine2]
        r = np.corrcoef(x, y)[0, 1]
        if use_spatial_gene_set: 
            x_old = cell_type_log2_old.loc[:, cytokine1]
            y_old = cell_type_log2_old.loc[:, cytokine2]
            r_old = np.corrcoef(x_old, y_old)[0, 1]
        
        # ------------------------- Small plot -------------------------
        plt.figure(figsize=cm_to_inches((3, 3)))
        plt.scatter(x, y, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=2, label=f"r={r:.2f}")

        plt.ylim(-4, 4)
        plt.xticks([-4, 0, 4], fontsize=6)
        plt.yticks([-4, 0, 4], fontsize=6) 

        plt.xlabel(f"log$_{2}$FC", fontsize=6)
        plt.ylabel(f"log$_{2}$FC", fontsize=6)
        
        plt.title(f'{cytokine1} vs {cytokine2}', fontsize=6.5)
        plt.legend(loc="upper left", fontsize=6)
        
        plt.tight_layout()
        for format in ["png", "svg"]:
            plt.savefig(os.path.join(corr_plot_folder, f"6.1_small_corr_{cell_type}_{cytokine1}_vs_{cytokine2}{'_spatial' if use_spatial_gene_set else ''}.{format}"),
                            bbox_inches="tight", dpi=400)
        plt.close()
        
        # ---- subplots of large plot ----
        ax = axs[counter]
        
        # create a cross at the origin
        ax.plot([-5, 5], [0, 0], color="black", linewidth=1, zorder=0, alpha=0.5)
        ax.plot([0, 0], [-5, 5], color="black", linewidth=1, zorder=0, alpha=0.5)
        ax.scatter(x, y, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=2, color="red" if use_spatial_gene_set else "tab:blue")
        if use_spatial_gene_set:
            ax.scatter(x_old, y_old, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=1, color="gray")
        
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_xticks([-4, -2, 0, 2, 4])
        ax.set_yticks([-4, -2, 0, 2, 4]) 
        ax.set_xlabel(f"{introduce_greek_letters(cytokine1)}", fontsize=6)
        ax.set_ylabel(f"{introduce_greek_letters(cytokine2)}", fontsize=6)

        if use_spatial_gene_set:
            correlation_text = f"r$_\\mathrm{{spatial}}$={r:.2f}\nr$_\\mathrm{{full}}$={r_old:.2f}"
        else:
            correlation_text = f"r={r:.2f}"
        ax.text(0.05 if "IL-10" not in [cytokine1, cytokine2] else 0.4, 0.95, correlation_text, transform=ax.transAxes, 
                fontsize=5, verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))
        ax.tick_params(axis='both', which='both', length=0)
        
        counter += 1
        if not ax.get_subplotspec().is_first_col():
            plt.setp(ax.get_yticklabels(), visible=False)
        
        if not ax.get_subplotspec().is_last_row():
            plt.setp(ax.get_xticklabels(), visible=False)

plt.tight_layout(pad=0.5, h_pad=0.5, w_pad=0.5)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(corr_plot_folder, f"6.1_large_corr_panel{'_spatial' if use_spatial_gene_set else ''}.{format}"),
                    bbox_inches="tight", dpi=400)
plt.close()

In [ ]:
if use_spatial_gene_set:
    raise Exception("Deactivate spatial gene set to continue!")

In [ ]:
# write out similarity results to SI table
r_vals_by_cell_type_df.to_excel(os.path.join(output_tables_folder, "6.1_r_vals_by_cell_type_df.xlsx"))
df = r_vals_by_cell_type_df.copy()
df = df.reset_index()
df['cytokine1'] = df['cytokine1'].apply(introduce_greek_letters)
df['cytokine2'] = df['cytokine2'].apply(introduce_greek_letters)
write_to_SI_table(df, "cytokine_similarity_pearson_r")

In [ ]:
# Create a subplot grid based on number of cell types
n_cell_types = len(r_vals_by_cell_type.keys())
n_rows = 3
n_cols = 5

fig, axes = plt.subplots(n_rows, n_cols, figsize=cm_to_inches((16, 8)))
axes = axes.flatten()

# Plot histogram for each cell type
for idx, (cell_type, ax) in enumerate(zip(r_vals_by_cell_type.keys(), axes)):
    df_subset = r_vals_by_cell_type[cell_type].copy()
    # set all diagonals to NaN
    df_subset.values[np.diag_indices_from(df_subset)] = np.nan
    flattened = pd.Series(df_subset.values.flatten()).dropna()
    ax.hist(flattened, bins=np.arange(-1, 1.01, 0.1), density=True, color="skyblue")
    mean_val = flattened.mean()
    
    ax.axvline(mean_val, color='black', linestyle='dashed', linewidth=1)
    ax.text(mean_val+0.3, 3, f'{mean_val:.2f}', color='black', ha='center')
    ax.set_title(cell_type)
    ax.set_ylim(0, 3.5)
    ax.set_xlim(-1, 1)
    
    ax.set_ylabel("density")
    ax.set_xlabel("Pearson r")
    ax.label_outer()

for idx in range(len(r_vals_by_cell_type.keys()), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"6.1_corr_histograms.{format}"), dpi=300, bbox_inches="tight")

## Dendrogram clustering

#### This isn't actually being used in the paper

In [ ]:
def plot_heatmap_with_clustering(distance_matrix, cell_type, plot_name, heatmap_label, communities=None):
    """This plots a heatmap for the Supplementary Information."""
    if communities is not None:
        communities_flat = [item for sublist in communities for item in sublist]
        distance_matrix = distance_matrix.loc[communities_flat, communities_flat]

    plt.figure(figsize=(len(distance_matrix)*0.1+1, len(distance_matrix)*0.1+1))
    ax = sns.heatmap(distance_matrix, cmap="coolwarm", center=0, annot=False, fmt=".2f", vmin=-1, vmax=1, cbar=True, square=True,
                cbar_kws={'label': heatmap_label, 'shrink': 0.6})
    
    # plot black boxes around the communities
    if communities is not None:
        current_pos = 0
        for community in communities:
            num_entries = len(community)
            rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='black', linewidth=1.5, clip_on=False)
            plt.gca().add_patch(rect)
            current_pos += num_entries
    
    # set font sizes to 5.5
    ax.set_xticks(np.arange(0.5, len(distance_matrix)+0.5))
    ax.set_yticks(np.arange(0.5, len(distance_matrix)+0.5))
    ax.set_xticklabels(distance_matrix.index, fontsize=5.5, rotation=90)  # Rotate for better visibility
    ax.set_yticklabels(distance_matrix.index, fontsize=5.5)
    # set the tick length to 0
    ax.tick_params(length=0)
    
    apply_greek_labels_heatmap(ax)

    plt.title(f"{cell_type}", fontsize=7)
    plt.tight_layout()

    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_{plot_name}_{cell_type}.{format}", bbox_inches="tight")
        
        
def plot_heatmap_with_clustering_small(distance_matrix, cell_type, plot_name, heatmap_label, communities=None):
    """This plots a heatmap for the Main Text."""
    if communities is not None:
        communities_flat = [item for sublist in communities for item in sublist]
        distance_matrix = distance_matrix.loc[communities_flat, communities_flat]

    plt.figure(figsize=cm_to_inches((8.5,8.5)))
    ax = sns.heatmap(
        distance_matrix, cmap="coolwarm", center=0, annot=False, fmt=".2f",
        vmin=-1, vmax=1, cbar=True, square=True,
        cbar_kws={'label': heatmap_label, 'shrink': 0.45}
    )
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=5.5)  # Reduce the fontsize of the colorbar labels
    
    # plot black boxes around the communities
    if communities is not None:
        current_pos = 0
        for community in communities:
            num_entries = len(community)
            rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries,
                                 fill=False, color='black', linewidth=1.5, clip_on=False)
            plt.gca().add_patch(rect)
            current_pos += num_entries
    
    # Instead of the default y-axis labels, we will stagger them in 3 positions.
    # First, remove the default labels:
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # Define 3 staggered horizontal offsets (in axis coordinates: 0=left, 1=right).
    # Adjust these values as needed for your plot.
    stagger_offsets = [-0.01, -0.15, -0.19]
    
    # Manually add the y-axis labels at staggered positions.
    for i, label in enumerate(distance_matrix.index):
        offset = stagger_offsets[i % 2]
        ax.text(
            offset, i + 0.2, introduce_greek_letters(label),
            ha='right', va='center', fontsize=5.5,
            transform=ax.get_yaxis_transform()  # x in axis fraction, y in data coordinates
        )
    
    # remove tick marks
    ax.tick_params(length=0)
    
    apply_greek_labels_heatmap(ax)
    
    plt.title(f"{cell_type}", fontsize=7)
    plt.tight_layout()

    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_small_{plot_name}_{cell_type}.{format}", bbox_inches="tight")

In [ ]:
%%capture output
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform

def dendrogram_clustering(distance_matrix):
    columns = distance_matrix.columns
    distance_matrix_original = distance_matrix.copy()
    
    # distance matrix
    distance_matrix = np.nan_to_num(distance_matrix)

    # condense the distance matrix
    condensed_distance = squareform(distance_matrix)

    # hierarchical clustering 
    linkage = sch.linkage(condensed_distance, method='ward')

    # get the dendrogram and ordered indices
    dendro = sch.dendrogram(linkage, labels=columns, no_plot=True)
    ordered_indices = dendro['leaves']

    # compute cophenetic distances between leaves
    _, cophenetic_distances = sch.cophenet(linkage, condensed_distance)
    # convert to a square matrix
    cophenetic_matrix = squareform(cophenetic_distances)
    cophenetic_df = pd.DataFrame(cophenetic_matrix, index=columns, columns=columns)
    cophenetic_df = cophenetic_df.iloc[ordered_indices, ordered_indices]
    cophenetic_df = cophenetic_df/cophenetic_df.max().max()
    
    return distance_matrix_original.iloc[ordered_indices, ordered_indices], cophenetic_df, linkage

# for cell_type in r_vals_by_cell_type.keys():
#     cytokine_filter = strong_impact_by_cell_type[cell_type][strong_impact_by_cell_type[cell_type]].index
#     distance_matrix = 1 - r_vals_by_cell_type[cell_type].loc[cytokine_filter, cytokine_filter].astype(float)
#     ordered_distance_vals, _, _ = dendrogram_clustering(distance_matrix)
#     plot_heatmap_with_clustering(1-ordered_distance_vals, cell_type, plot_name="cosine_dendrogram", heatmap_label="cosine similarity")

## Group cytokines by highly similar correlation values

In [ ]:
# get the correlation values in long format
all_r_vals_long = (
    r_vals_by_cell_type_df
    .stack()
    .reset_index()
    .rename(columns={
        "level_2": "cell_type",
        0: "r_val"
    })
)

# check if both cytokines have a strong impact in a given cell type
all_r_vals_long["strong_impact"] = False
for index, row in all_r_vals_long.iterrows():
    cytokine_1 = row["cytokine1"]
    cytokine_2 = row["cytokine2"]
    cell_type = row["cell_type"]
    strong_impact_1 = strong_impact_by_cell_type.loc[cytokine_1, cell_type]
    strong_impact_2 = strong_impact_by_cell_type.loc[cytokine_2, cell_type]
    if strong_impact_1 and strong_impact_2:
        all_r_vals_long.loc[index, "strong_impact"] = True
        
# subset to strong impact and exclude unwanted cell types
all_r_vals_long = all_r_vals_long[all_r_vals_long["strong_impact"]]

# Create a unique identifier and drop duplicates (Pearson r is symmetric)
all_r_vals_long["unique_identifier"] = all_r_vals_long.apply(
    lambda x: "_".join(sorted([str(x.get("cytokine1", "")), str(x.get("cytokine2", "")), str(x.get("cell_type", ""))])), 
    axis=1
)
all_r_vals_long = all_r_vals_long.drop_duplicates(subset=["unique_identifier"])

In [ ]:
# subset to those that have values in at least three cell types
all_r_vals_long["cyto1_cyto2"] = all_r_vals_long["cytokine1"] + "_" + all_r_vals_long["cytokine2"]
val_counts = all_r_vals_long["cyto1_cyto2"].value_counts()

# subset to those measured in at least three cell types (otherwise an overall consensus value is not meaningful)
val_counts = val_counts[val_counts > 2]
all_r_vals_long = all_r_vals_long[all_r_vals_long["cyto1_cyto2"].isin(val_counts.index)]

# calculate the mean value
mean_r = all_r_vals_long.groupby(["cytokine1", "cytokine2"])["r_val"].mean()
mean_r = mean_r.unstack()

In [ ]:
# threshold at mean values
group_by_cytokine = pd.DataFrame(columns=["cytokine1", "cytokine2"])
threshold = 0.7
for cytokine_1 in mean_r.index:
    cytokine_r = mean_r.loc[cytokine_1]
    cytokine_r = cytokine_r[cytokine_r > threshold]
    for cytokine_2 in cytokine_r.index:
        group_by_cytokine.loc[len(group_by_cytokine)] = [cytokine_1, cytokine_2]

group_by_cytokine = group_by_cytokine[group_by_cytokine["cytokine1"] != group_by_cytokine["cytokine2"]]

In [ ]:
# Create pair column by sorting cytokine names alphabetically and joining with underscore
group_by_cytokine["pair"] = group_by_cytokine.apply(lambda x: f"{min(x['cytokine1'], x['cytokine2'])}_{max(x['cytokine1'], x['cytokine2'])}", axis=1)

# Remove duplicate pairs
group_by_cytokine = group_by_cytokine.drop_duplicates(subset=["pair"])

In [ ]:
def sort_list_by_reference(input_list, reference):
    """
    Sort a list based on the order of elements in a reference list.
    
    Parameters:
    -----------
    input_list : list
        The list to be sorted. Should be a subset of the reference list.
    reference : list
        The reference list that defines the desired order.
        
    Returns:
    --------
    list
        A new list with elements from input_list sorted according to their order in reference.
    """
    # Create a dictionary mapping each element in reference to its position
    reference_positions = {element: i for i, element in enumerate(reference)}
    
    # Sort the input list based on the positions in the reference list
    sorted_list = sorted(input_list, key=lambda x: reference_positions.get(x, float('inf')))
    
    return sorted_list

In [ ]:
# Create an undirected graph
G = ig.Graph()

# Add vertices (unique cytokines)
unique_cytokines = set(group_by_cytokine['cytokine1'].unique()) | set(group_by_cytokine['cytokine2'].unique())
G.add_vertices(list(unique_cytokines))

# Add edges from the dataframe
edges = list(zip(group_by_cytokine['cytokine1'], group_by_cytokine['cytokine2']))
G.add_edges(edges)

# Find connected components
components = G.components()

# Print each group
all_close_groups = []
for i, component in enumerate(components, 1):
    # Get the vertex names for this component
    group = [G.vs[idx]['name'] for idx in component]
    sorted_group = sort_list_by_reference(group, list(cytokine_info.index))
    print(f"Group {i}: {sorted_group}")
    all_close_groups.append(sorted_group)

In [ ]:
with open(os.path.join(intermediate_output_folder, "6_highly_similar_groups.pkl"), "wb") as f:
    pickle.dump(all_close_groups, f)
    
# write out the groups to a table
df = all_close_groups.copy()
df = pd.Series([", ".join(group) for group in df])
df = df.apply(introduce_greek_letters)
df = df.to_frame(name="Highly similar cytokine groups")
df['Group ID'] = df.index + 1
df = df[['Group ID', 'Highly similar cytokine groups']]
write_to_SI_table(df, "highly_similar_cytokine_groups")    

## Leiden clustering


#### Clustering in each cell type separately

In [ ]:
%%capture output
all_communities_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    # subset to cytokines with strong impact in this cell type
    # could deactivate this but it makes the plots too large and largely includes noise
    cytokine_filter = strong_impact_by_cell_type[cell_type][strong_impact_by_cell_type[cell_type]].index
    adjacency_matrix = r_vals_by_cell_type[cell_type].loc[cytokine_filter, cytokine_filter].astype(float)
    
    # Get the distance matrix.
    df = adjacency_matrix.copy()
    # Ensure that the diagonal is zero (i.e., no self-loops)
    np.fill_diagonal(df.values, 0)

    all_communities, _ = leiden_clustering(
        adjacency_matrix,
        0.5,
        False,
        invert_distance=False)
    mean_r_to_first_community = []
    for community in all_communities:
        r_vals = adjacency_matrix.loc[all_communities[0], community]
        # set diagonal to nan
        np.fill_diagonal(r_vals.values, np.nan)
        mean_r = r_vals.stack().mean()
        mean_r_to_first_community.append(mean_r)

    sort_index = np.argsort(mean_r_to_first_community)[::-1]
    all_communities = [all_communities[i] for i in sort_index]
    all_communities_by_cell_type[cell_type] = all_communities

    plot_heatmap_with_clustering(adjacency_matrix, cell_type, plot_name="pearson_leiden", heatmap_label="Pearson r", communities=all_communities)
    if cell_type == "CD14 Mono":
        plot_heatmap_with_clustering_small(adjacency_matrix, cell_type, plot_name="pearson_leiden", heatmap_label="Pearson r", communities=all_communities)

In [ ]:
all_communities_by_cell_type_df = {}
for cell_type in ALL_CELL_TYPES:
    comms = all_communities_by_cell_type[cell_type]
    cytokine_filter = strong_impact_by_cell_type[cell_type][strong_impact_by_cell_type[cell_type]].index
    grouped = pd.Series(index=pd.MultiIndex.from_product([cytokine_filter, cytokine_filter]), dtype=int, data=0)
    for comm_id, community in enumerate(comms):
        for cytokine1 in community:
            for cytokine2 in community:
                grouped.loc[(cytokine1, cytokine2)] = 1         
    all_communities_by_cell_type_df[cell_type] = grouped
all_communities_by_cell_type_df = pd.DataFrame(all_communities_by_cell_type_df)

In [ ]:
# all_communities_by_cell_type_df.to_excel(os.path.join(output_tables_folder, "DATE_6.1_community_assignments_by_cell_type.xlsx"))

#### Clustering across cell types

In [ ]:
myeloid_cells = ["CD14 Mono", "CD16 Mono", "cDC"]
lymphoid_cells = ["B cell", "CD4 T cell", "CD8 T cell", "NK CD56low", "NK CD56hi"]

In [ ]:
def get_combined_communities(cell_types, all_communities_by_cell_type):
    # create a distance matrix by normalizing clustering counts
    grouped_together_total = pd.DataFrame(0, index=CYTOKINE_ORDER, columns=CYTOKINE_ORDER)

    for cell_type in cell_types:
        communities = all_communities_by_cell_type[cell_type]
        for community in communities:
            for cytokine_1 in community:
                for cytokine_2 in community:
                    grouped_together_total.loc[cytokine_1, cytokine_2] += 1
                    
    # discard those with 0 in the diagonal
    # Keep only cytokines that have at least 2 interactions
    non_zero_diag = grouped_together_total.index[grouped_together_total.values.diagonal() > 1]
    grouped_together_total = grouped_together_total.loc[non_zero_diag, non_zero_diag]

    # Compute the row sums
    row_sums = grouped_together_total.sum(axis=1)

    # Compute the inverse square roots of the row sums
    eps = 1e-10
    inv_sqrt = 1 / np.sqrt(row_sums + eps)

    # Get the normalization factors
    norm_factors = pd.Series(inv_sqrt, index=grouped_together_total.index)

    # Perform the symmetric normalization
    # Multiply rows by the normalization factors, then columns.
    normalized_matrix = grouped_together_total.mul(norm_factors, axis=0)\
                                            .mul(norm_factors, axis=1)

    # round to 5 decimal places
    normalized_matrix = normalized_matrix.round(5)
    cosine_distances = squareform(pdist(normalized_matrix.values, metric='cosine'))
    cosine_distances = pd.DataFrame(cosine_distances, index=normalized_matrix.index, columns=normalized_matrix.index)
    
    # perform Leiden clustering (so far: resolution 0.5)
    all_communities, _ = leiden_clustering(cosine_distances, resolution_parameter=0.7, print_communities=False, invert_distance=True)

    # reorder communities internally to group highly similar cytokines
    reordered_communities = []

    for community in all_communities:
        community_list = list(community)
        
        # track which elements were moved to maintain order of non-subgroup elements
        moved_elements = set()
        
        # process each group that's a subset of this community in all_close_groups
        for group in all_close_groups: # this was calculated above by grouping on correlation
            if set(group).issubset(set(community)):
                # find the earliest index of any group member in the community
                earliest_idx = min(community_list.index(member) for member in group)
                
                # remove group members from their current positions
                # (if they haven't been moved by a previously)
                for member in group:
                    if member not in moved_elements:
                        community_list.remove(member)
                        moved_elements.add(member)
                
                # insert group members together at the earliest possible index
                for i, member in enumerate(group):
                    if member not in community_list:  # avoid duplicates (do I need this?)
                        community_list.insert(earliest_idx + i, member)
        
        reordered_communities.append(community_list)

    # replace all_communities with the reordered version
    all_communities = reordered_communities
    communities_flat = [item for sublist in all_communities for item in sublist]      
    
    return all_communities, communities_flat, grouped_together_total, normalized_matrix


cell_subset_labels = ['myeloid', 'lymphoid', 'all']
cell_subsets = [myeloid_cells, lymphoid_cells, ALL_CELL_TYPES_EXCLUDE]

for i in range(len(cell_subset_labels)):
    cell_subset_label = cell_subset_labels[i]
    cell_types = cell_subsets[i]
    
    all_communities, communities_flat, grouped_together_total, normalized_matrix = get_combined_communities(cell_types, all_communities_by_cell_type)

    # -------------------- Plotting --------------------
    df_to_plot = grouped_together_total.loc[communities_flat, communities_flat]
    plt.figure(figsize=cm_to_inches((12, 12)))

    ax = sns.heatmap(df_to_plot, square=True, vmin=0, vmax=df_to_plot.max().max(), cbar=True,
                cbar_kws={'label': 'grouping count', 'shrink': 0.6})

    # plot white boxes around the communities
    current_pos = 0
    for community in all_communities:
        num_entries = len(community)
        rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='white', linewidth=1.5, clip_on=False)
        plt.gca().add_patch(rect)
        current_pos += num_entries

    # Plot additional boxes around highly similar groups
    if cell_subset_label == 'all':
        for group in all_close_groups:
            group_indices = [communities_flat.index(member) for member in group if member in communities_flat]
            if group_indices:
                min_idx = min(group_indices)
                max_idx = max(group_indices)
                rect = plt.Rectangle((min_idx, min_idx), max_idx - min_idx + 1, max_idx - min_idx + 1, fill=False, color='skyblue', ls="--", linewidth=1, clip_on=False)
                plt.gca().add_patch(rect)

    # set font sizes to 5.5
    ax.set_xticks(np.arange(0.5, len(df_to_plot)+0.5))
    ax.set_yticks(np.arange(0.5, len(df_to_plot)+0.5))
    ax.set_xticklabels(df_to_plot.index, fontsize=5.5, rotation=90)  # Rotate for better visibility
    ax.set_yticklabels(df_to_plot.index, fontsize=5.5)
    # set the tick length to 0
    ax.tick_params(length=0)

    apply_greek_labels_heatmap(ax)

    plt.tight_layout()
    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_grouped_together_total_{cell_subset_label}.{format}", bbox_inches="tight")


    # -------------------- Plotting ALT --------------------
    df_to_plot = grouped_together_total.loc[communities_flat, communities_flat]
    plt.figure(figsize=cm_to_inches((8.5, 8.5)))

    ax = sns.heatmap(df_to_plot, square=True, vmin=0, vmax=df_to_plot.max().max(), cbar=True,
                cbar_kws={'label': 'grouping count', 'shrink': 0.45})

    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=5.5)  # Reduce the fontsize of the colorbar labels

    # plot white boxes around the communities
    current_pos = 0
    for community in all_communities:
        num_entries = len(community)
        rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='white', linewidth=1, clip_on=False)
        plt.gca().add_patch(rect)
        current_pos += num_entries

    # Plot additional boxes around highly similar groups
    if cell_subset_label == 'all':
        for group in all_close_groups:
            group_indices = [communities_flat.index(member) for member in group if member in communities_flat]
            if group_indices:
                min_idx = min(group_indices)
                max_idx = max(group_indices)
                rect = plt.Rectangle((min_idx, min_idx), max_idx - min_idx + 1, max_idx - min_idx + 1, fill=False, color='skyblue', ls="--", linewidth=1, clip_on=False)
                plt.gca().add_patch(rect)


    # Instead of the default y-axis labels, we will stagger them in 3 positions.
    # First, remove the default labels:
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # Define 3 staggered horizontal offsets (in axis coordinates: 0=left, 1=right).
    # Adjust these values as needed for your plot.
    stagger_offsets = [-0.01, -0.15, -0.19]

    # Manually add the y-axis labels at staggered positions.
    # Here, the y coordinate is the same as the tick (i + 0.5) and the x position is determined by the stagger.
    for i, label in enumerate(df_to_plot.index):
        offset = stagger_offsets[i % 2]
        ax.text(
            offset, i + 0.2, introduce_greek_letters(label),
            ha='right', va='center', fontsize=5.5,
            transform=ax.get_yaxis_transform()  # x in axis fraction, y in data coordinates
        )

    # remove tick marks
    ax.tick_params(length=0)
    
    if cell_subset_label == "all":
        ax.set_title(f"grouping across cell types", fontsize=7)
    else:
        ax.set_title(f"{cell_subset_label} cells ({", ".join(cell_types)})", fontsize=7)
    apply_greek_labels_heatmap(ax)

    plt.tight_layout()
    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_grouped_together_total_alt_{cell_subset_label}.{format}", bbox_inches="tight")

In [ ]:
# # write out the groups to a table
# df = all_communities.copy()
# df = pd.Series([", ".join(group) for group in df])
# df = df.apply(introduce_greek_letters)
# df = df.to_frame(name="Cytokine Groups by Leiden Clustering")
# df['Group ID'] = df.index + 1
# df = df[['Group ID', 'Cytokine Groups by Leiden Clustering']]
# write_to_SI_table(df, "cytokine_groups")    

In [ ]:
# save the results
with open(os.path.join(intermediate_output_folder, "6_all_communities_cyto_cyto.pkl"), "wb") as f:
    pickle.dump(all_communities, f)

In [ ]:
all_communities_str = [", ".join(l) for l in all_communities]
all_communities_df = pd.DataFrame(all_communities_str).rename(columns={0: "communities_across_cells"})
all_communities_df["close_groups_across_cells"] = ""

for l in all_close_groups:
    mapping_index = [i for i, community in enumerate(all_communities) if all(cyto in community for cyto in l)]
    if mapping_index:
        if len(all_communities_df.loc[mapping_index[0], "close_groups_across_cells"]) == 0:
            all_communities_df.loc[mapping_index[0], "close_groups_across_cells"] = ", ".join(l)
        else:
            all_communities_df.loc[mapping_index[0], "close_groups_across_cells"] = \
                all_communities_df.loc[mapping_index[0], "close_groups_across_cells"] + "; " + ", ".join(l)
all_communities_df.to_csv(os.path.join(output_tables_folder, "6.1_across_cell_communities.csv"))

In [ ]:
cytokine_to_across_cell_group = {}
for index, row in all_communities_df.iterrows():
    community = row["communities_across_cells"].split(", ")
    for cytokine in community:
        cytokine_to_across_cell_group[cytokine] = f"Group {'0' if index+1<10 else ''}{index+1}" if len(community)>1 else "Ungrouped"

In [ ]:
all_communities_by_cell_type_df = {}
cytokine_to_group_by_cell_type = {}
group_to_cytokines_by_cell_type = {}
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    cytokine_to_group = {}
    group_to_cytokines = {}
    communities = all_communities_by_cell_type[cell_type]
    output_df = pd.DataFrame(index=CYTOKINE_ORDER, columns=CYTOKINE_ORDER, data=np.nan)
    
    # set everything with strong impact to 0
    strong_impact = strong_impact_by_cell_type[cell_type]
    strong_impact = strong_impact[strong_impact].index
    output_df.loc[strong_impact, strong_impact] = 0

    for i, community in enumerate(communities):
        combinations = list(itertools.product(community, community))
        for c1, c2 in combinations:
            output_df.loc[c1, c2] = 1
        for entry in community:
            cytokine_to_group[entry] = f"Group {'0' if index+1<10 else ''}{i+1}" if len(community)>1 else "Ungrouped"
        group_to_cytokines[f"Group {i+1}"] = community
    group_to_cytokines_by_cell_type[cell_type] = group_to_cytokines
    cytokine_to_group_by_cell_type[cell_type] = cytokine_to_group
    all_communities_by_cell_type_df[cell_type] = output_df
all_communities_by_cell_type_df = pd.concat(all_communities_by_cell_type_df, axis=1, keys=ALL_CELL_TYPES_EXCLUDE)
all_communities_by_cell_type_df.to_csv(os.path.join(output_tables_folder, "6.1_within_cell_communities.csv"))

In [ ]:
os.makedirs("group_exploration", exist_ok=True)
with open("group_exploration/group_to_cytokines_by_cell_type.pkl", 'wb') as f:
    pickle.dump(group_to_cytokines_by_cell_type, f)
with open("group_exploration/cytokine_to_group_by_cell_type.pkl", 'wb') as f:
    pickle.dump(cytokine_to_group_by_cell_type, f)
with open("group_exploration/cytokine_to_across_cell_group.pkl", 'wb') as f:
    pickle.dump(cytokine_to_across_cell_group, f)

## 6.3 - Secondy responses

In [ ]:
plot_folder = os.path.join(base_plot_folder, "6.3_secondary_responses")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# inputs calculated above for receptor expression vs. response strength
# outliers_rec_by_cell_type_high_high: High receptor expression - strong response (primary response?)
# outliers_rec_by_cell_type_low_high: Low receptor expression - strong response (secondary response?)

all_secondary_responses_pre_filter = {}
all_secondary_responses = {}
if 'cytokine_response_df_all' in locals():
    test_cytokines = CYTOKINE_ORDER
    
    for test_cytokine in test_cytokines: # secondary responses for a specific cytokine
        # all cell types that have a strong response to the primary cytokine and high receptor expression
        base_cell_types = list(outliers_rec_by_cell_type_high_high[outliers_rec_by_cell_type_high_high["index"] == test_cytokine]["cell_type"])
        
        # all cell types that have a strong response to the primary cytokine and low receptor expression
        additional_cell_types = list(outliers_rec_by_cell_type_low_high[outliers_rec_by_cell_type_low_high["index"] == test_cytokine]["cell_type"])
        
        # get upregulated cytokines in primary response cell types for response to test_cytokine
        df = cytokine_response_df_all[(cytokine_response_df_all["treatment"] == test_cytokine) & (cytokine_response_df_all["cell_type"].isin(base_cell_types))].sort_values(by="fold_changes")
        df["fold_changes"] = df["fold_changes"].apply(min) # this is to get the lowest fold-change for multi-subunit cytokines (only care about upregulation here so this works)
        df = df[df["fold_changes"] > 1] # subset on those that are substantially upregulated
        df = df[~(df["response"] == test_cytokine)] #  We don't care about self-interaction here
        secondary_cytokines = df["response"].unique()
        
        # get potential secondary responses
        secondary_responses = {}
        all_secondary_responses_pre_filter[test_cytokine] = []
        for secondary_cytokine in secondary_cytokines:
            # which cell types secrete this cytokine?
            secreting_cell_types = df[df['response'] == secondary_cytokine]['cell_type'].unique()
            
            # strong enough response to primary cytokine
            strong_responders = list(outliers_rec_by_cell_type_high_high[outliers_rec_by_cell_type_high_high["index"] == secondary_cytokine]["cell_type"]) 
            strong_responders = [c for c in strong_responders if c in additional_cell_types] # only look at those where we don't already have a primary response
            
            if len(strong_responders) > 0:
                all_secondary_responses_pre_filter[test_cytokine].append(secondary_cytokine)
                # get sufficiently similar responses
                responder_similarities = r_vals_by_cell_type_df.loc[test_cytokine]
                responder_similarities = responder_similarities.loc[secondary_cytokine, strong_responders]
                responder_similarities = responder_similarities[responder_similarities > 0.5]
                if len(responder_similarities) > 0:
                    secondary_responses[secondary_cytokine] = {}
                    secondary_responses[secondary_cytokine]['similarity'] = responder_similarities
                    secondary_responses[secondary_cytokine]['secreters'] = secreting_cell_types
        if len(secondary_responses) > 0:
            all_secondary_responses[test_cytokine] = secondary_responses

In [ ]:
rows_list = []

# Check if all_secondary_responses exists and is a dictionary
if 'all_secondary_responses' in locals() and isinstance(all_secondary_responses, dict):
    for primary_cytokine, secondary_map in all_secondary_responses.items():
        for secondary_cytokine, details_dict in secondary_map.items():
            similarity_series = details_dict.get('similarity')
            secreters_array = details_dict.get('secreters')

            # Ensure that 'similarity' and 'secreters' keys exist and their values are not None
            if similarity_series is not None and secreters_array is not None:
                # Convert numpy array of secreting cell types to a list
                secreting_cell_types_list = list(secreters_array)
                
                # Iterate through the similarity Series (index=responding_cell_type, value=similarity_score)
                for responding_cell_type, similarity_score in similarity_series.items():
                    rows_list.append({
                        'primary_cytokine': primary_cytokine,
                        'secondary_cytokine': secondary_cytokine,
                        'responding_cell_type': responding_cell_type,
                        'similarity_score': similarity_score,
                        'secreting_cell_types': secreting_cell_types_list
                    })

# Define column names for the DataFrame
column_names = [
    'primary_cytokine', 
    'secondary_cytokine', 
    'responding_cell_type', 
    'similarity_score', 
    'secreting_cell_types'
]

# Create the DataFrame from the list of rows
df_all_secondary_responses = pd.DataFrame(rows_list, columns=column_names)
df_all_secondary_responses = df_all_secondary_responses.explode("secreting_cell_types")

# Display the resulting DataFrame
df_all_secondary_responses['secondary_cytokine'] = df_all_secondary_responses['secondary_cytokine'].apply(introduce_greek_letters)
df_all_secondary_responses['primary_cytokine'] = df_all_secondary_responses['primary_cytokine'].apply(introduce_greek_letters)

In [ ]:
def create_parcats_diagram(df,
                           column_names, # List of column names in order
                           column_orders=None, # List of order lists, or None
                           title=None,
                           color_column=None, # Optional: column name to use for coloring paths
                           colorscale=None,  # Optional: Plotly colorscale for path_color_column)
                           width=400,
                           height=160,
    ):
    """
    Create a Parallel Categories (Parcats) diagram.

    Parameters:
    df (pd.DataFrame): DataFrame containing the data. Each row is an observation/path.
    column_names (list): List of strings, names of the columns to include as dimensions.
    column_orders (list of lists, optional): List where each element is a list of unique values 
                                             for the corresponding column in column_names, in desired order.
                                             If None, order is inferred from data.
    title (str, optional): Title of the diagram.
    color_column (str, optional): Name of a column in df whose values will be used to color the paths.
                                  This column's values should ideally be numeric for continuous coloring,
                                  or can be categorical if a suitable categorical colorscale is provided
                                  or if Plotly handles it by default.
    colorscale (str or list, optional): Plotly colorscale name (e.g., 'Viridis') or custom scale.
    category_spacing (float, optional): Spacing between categories on an axis (normalized, 0 to 1).
                                        Default is 0.03. Smaller values reduce space.
    """
    if len(column_names) < 2:
        raise ValueError("At least two columns must be specified for a Parcats diagram.")
    if column_orders and len(column_names) != len(column_orders):
        raise ValueError("Length of column_names and column_orders must match.")
    
    def _adaptive_spacing(df, cols, gap_total=0.25):
        n_cats  = {c: df[c].nunique() for c in cols}
        return {c: 0 if n_cats[c] == 1
                else gap_total/(n_cats[c]-1)
                for c in cols}
    spacings = _adaptive_spacing(df, column_names, gap_total=5)    
    print(spacings)
    
    dimensions = []
    for i, col_name in enumerate(column_names):
        if col_name not in df.columns:
            raise ValueError(f"Column '{col_name}' not found in DataFrame.")
        
        dim = dict(
            label=None,  # No explicit label for the dimension axis itself
            values=df[col_name],
            categoryspacing=spacings[col_name]  # Reduce vertical distance between categories
        )
        if column_orders and column_orders[i] is not None:
            unique_vals_in_df = df[col_name].unique()
            ordered_categories = [val for val in column_orders[i] if val in unique_vals_in_df]
            for val in unique_vals_in_df:
                if val not in ordered_categories:
                    ordered_categories.append(val)
            dim['categoryarray'] = ordered_categories
        dimensions.append(dim)
    
    if 'count' not in df.columns: 
        grouped_df = df.groupby(column_names, observed=True).size().reset_index(name='_parcats_count_')
        # Rebuild dimensions based on grouped_df
        dimensions = []
        for i, col_name in enumerate(column_names):
            dim = dict(
                label=None, 
                values=grouped_df[col_name],
            )
            if column_orders and column_orders[i] is not None:
                unique_vals_in_grouped_df = grouped_df[col_name].unique()
                ordered_categories = [val for val in column_orders[i] if val in unique_vals_in_grouped_df]
                for val in unique_vals_in_grouped_df:
                    if val not in ordered_categories:
                        ordered_categories.append(val)
                dim['categoryarray'] = ordered_categories
            dimensions.append(dim)
        counts_values = grouped_df['_parcats_count_']
    else: 
        counts_values = df['count']
        
    parcats_data = dict(
        dimensions=dimensions,
        counts=counts_values, 
        line=dict(shape='hspline') 
    )

    if color_column:
        # Determine which DataFrame to use for color data
        color_data_source_df = grouped_df if 'grouped_df' in locals() else df
        
        if color_column not in color_data_source_df.columns:
             raise ValueError(f"Color column '{color_column}' not found in the relevant DataFrame.")
        
        color_data_values = color_data_source_df[color_column]

        if pd.api.types.is_numeric_dtype(color_data_values):
            parcats_data['line']['color'] = color_data_values
            parcats_data['line']['edgecolor'] = "none"
            parcats_data['line']['showscale'] = True 
            if colorscale:
                parcats_data['line']['colorscale'] = colorscale
        else: 
            unique_color_categories = color_data_values.astype('category')
            parcats_data['line']['color'] = unique_color_categories.cat.codes
            if colorscale: 
                 parcats_data['line']['colorscale'] = colorscale


    fig = go.Figure(data=[go.Parcats(parcats_data)])
    
    fig.update_layout(
        title=title,
        margin=dict(l=2.5, r=2.5, t=20 if title else 2.5, b=2.5),
        width=width,
        height=height,
        font=dict(size=18, family="Helvetica", color='black') # Global font color
    )
    return fig

# Define the full list of orderings for Parcats
column_order = ["primary_cytokine", "secreting_cell_types", "secondary_cytokine", "responding_cell_type"]
all_column_orders = [
    ["IL-12", "IL-2", "IL-15", "TSLP", "IFN-γ"], 
    ['NK CD56hi', 'NK CD56low', "cDC", "HSPC"],
    ["IFN-γ", "TL1A", "IL-15"], 
    #["IFN-γ"], 
    ['CD14 Mono', 'CD16 Mono','B cell', 'CD8 T cell', 'Treg', 'NKT', 'cDC', 'HSPC', 'CD4 T cell', 'MAIT', 'NK CD56hi', 'NK CD56low'],
]

# Pre-order the DataFrame based on the column orders
df_all_secondary_responses_filter = df_all_secondary_responses.copy()
# Create ordered categorical columns for each dimension
for col, order in zip(column_order, all_column_orders):
    df_all_secondary_responses_filter[col] = pd.Categorical(
        df_all_secondary_responses_filter[col],
        categories=order,
        ordered=True
    )
# Sort the DataFrame based on the ordered columns
df_all_secondary_responses_filter = df_all_secondary_responses_filter.sort_values(by=column_order)

fig_parcats = create_parcats_diagram(
    df=df_all_secondary_responses_filter,
    column_names=column_order,
    column_orders=all_column_orders,
    color_column=column_order[0],
    colorscale=[ [0.0, "#440154"], [0.7, "#21918c"], [0.92, "#fde725"], [1.0, "#b5bd00"] ],  # Custom Viridis: last color is darker yellow-green
    width=500,
    height=240,
)
fig_parcats.show()
fig_parcats.write_image(os.path.join(plot_folder, f"6.1.5_secondary_response_parcats_all.svg"), engine="kaleido", width=400, height=160)
fig_parcats.write_image(os.path.join(plot_folder, f"6.1.5_secondary_response_parcats_all.png"), engine="kaleido", width=400, height=160)

In [ ]:
# test everything that's a potential secondary response
test_cytokines = CYTOKINE_ORDER
#test_cytokines = ["IL-12"]

for test_cytokine in test_cytokines:
    base_cell_types = list(outliers_rec_by_cell_type_high_high[outliers_rec_by_cell_type_high_high["index"] == test_cytokine]["cell_type"])
    additional_cell_types = list(outliers_rec_by_cell_type_low_high[outliers_rec_by_cell_type_low_high["index"] == test_cytokine]["cell_type"])
    
    # no additional cell types to look at
    if len(additional_cell_types) == 0:
        continue
    
    # this is all sufficiently upregulated cytokines that produce a strong response in at least one additional cell type
    if len(all_secondary_responses_pre_filter[test_cytokine]) == 0:
        continue
    
    # receptor expression
    receptor_cytokines = [test_cytokine] + list(all_secondary_responses_pre_filter[test_cytokine])
    receptor_subset = receptor_expression_by_cytokine_relative.loc[receptor_cytokines, ALL_CELL_TYPES]
            
    # similarities
    r_val_df_subset = r_vals_by_cell_type_df.loc[test_cytokine]
    if len(receptor_cytokines) > 1:
        r_val_df_subset = r_val_df_subset.loc[receptor_cytokines[1:], ALL_CELL_TYPES]

    # get values for plotting
    expr = np.log10(receptor_subset.clip(0.01,1)).values
    rv   = r_val_df_subset.values 
    response_magnitude_cytokine = all_normalized_distances_mean.loc[test_cytokine, ALL_CELL_TYPES].values

    M, N = expr.shape
    M += 1
    
    # colormaps and norms
    cmap_expr = matplotlib.colormaps.get_cmap("coolwarm")
    norm_expr = matplotlib.colors.Normalize(vmin=np.nanmin(expr), vmax=np.nanmax(expr))

    cmap_r    = matplotlib.colormaps.get_cmap("viridis")
    r_min = 0.3
    r_max = 0.9
    norm_r    = matplotlib.colors.Normalize(vmin=r_min, vmax=r_max)

    cmap_response_magnitude = matplotlib.colormaps.get_cmap("rocket")
    response_magnitude_min = 0
    response_magnitude_max = 1
    norm_response_magnitude = matplotlib.colors.Normalize(vmin=response_magnitude_min, vmax=response_magnitude_max)
    
    # actual plot
    fig, ax = plt.subplots(figsize=(cm_to_inches((4.5, 3+0.5*M))))
    ax.set_xlim(0, N)
    ax.set_ylim(0, M)
    ax.invert_yaxis()
    ax.set_aspect("equal")
    for i in range(M):
        for j in range(N):
            x0, y0 = j, i

            if i == 0:
                # full square for row 0
                color = cmap_expr(norm_expr(expr[i, j]))
                rect = plt.Rectangle(
                    (x0, y0), 1, 1,
                    facecolor=color,
                    edgecolor=None,      # no border
                    linewidth=0,
                    antialiased=False
                )
                ax.add_patch(rect)
            elif i == 1:
                # full square for row 1
                color = cmap_response_magnitude(norm_response_magnitude(response_magnitude_cytokine[j]))
                rect = plt.Rectangle(
                    (x0, y0), 1, 1,
                    facecolor=color,
                    edgecolor=None,
                    linewidth=0,
                    antialiased=False
                )
                ax.add_patch(rect)
            else:
                # top half (expr)
                c1 = cmap_expr(norm_expr(expr[i-1, j]))
                top_rect = plt.Rectangle(
                    (x0, y0), 1, 0.5,
                    facecolor=c1,
                    edgecolor=None,
                    linewidth=0,
                    antialiased=False
                )
                ax.add_patch(top_rect)

                # bottom half (r‐value)
                c2 = cmap_r(norm_r(rv[i-2, j]))
                bot_rect = plt.Rectangle(
                    (x0, y0+0.5), 1, 0.5,
                    facecolor=c2,
                    edgecolor=None,
                    linewidth=0,
                    antialiased=False
                )
                ax.add_patch(bot_rect)

    # ticks and labels
    ax.set_xticks(np.arange(N)+0.5)
    ax.set_xticklabels(receptor_subset.columns, rotation=90, fontsize=5.5)
    y_labels = list(receptor_subset.index)
    y_labels = [y_labels[0]] + y_labels 
    ax.set_yticks(np.arange(len(y_labels)) + 0.5)
    ax.set_yticklabels(y_labels, fontsize=5.5)
    
    # horizontal separators
    ax.axhline(2, color="white", linewidth=1, solid_capstyle="butt")
    for i in range(len(receptor_subset) - 2):
        ax.axhline(i+3, color="white", linewidth=0.75, solid_capstyle="butt")
        
    # create colorbars
    sm1 = matplotlib.cm.ScalarMappable(norm=norm_expr, cmap=cmap_expr)
    sm1.set_array([])
    cax1 = fig.add_axes([0.92, 0.3, 0.03, 0.4])
    fig.colorbar(sm1, cax=cax1, label=r"$\log_{10}$(rel. receptor expr.)")

    sm2 = matplotlib.cm.ScalarMappable(norm=norm_response_magnitude, cmap=cmap_response_magnitude)
    sm2.set_array([])
    cax2 = fig.add_axes([0.96, 0.3, 0.03, 0.4])
    fig.colorbar(sm2, cax=cax2, label="response magnitude")

    sm3 = matplotlib.cm.ScalarMappable(norm=norm_r, cmap=cmap_r)
    sm3.set_array([])
    cax3 = fig.add_axes([0.96, 0.3, 0.03, 0.4])
    fig.colorbar(sm3, cax=cax3, label="response similarity (r)")
    
    # ----------------------------------------------------------------------------------------
    def style_heatmap_labels(ax, base_cell_types, additional_cell_types):
        """
        Style the heatmap labels with different formatting based on cell type categories.
        """
        # Get the column indices for base_cell_types and additional_cell_types
        all_cell_types = ax.get_xticklabels()
        all_cell_type_texts = [label.get_text() for label in all_cell_types]
        
        # Create a list to store the font properties for each x-tick label
        fontprops = []
        for cell_type in all_cell_type_texts:
            if cell_type in base_cell_types:
                # Bold font for base_cell_types
                fontprops.append({'weight': 'heavy', 'color': 'black'})
            elif cell_type in additional_cell_types:
                # Italic font for additional_cell_types
                fontprops.append({'style': 'italic', 'color': 'black'})
            else:
                # Grey color for other cell types
                fontprops.append({'color': 'grey'})

        # Apply the font properties to x-tick labels
        plt.setp(ax.get_xticklabels(), **{'fontsize': 5.5, 'rotation': 90})
        for i, label in enumerate(ax.get_xticklabels()):
            if i < len(fontprops):
                label.set_fontweight(fontprops[i].get('weight', 'normal'))
                label.set_fontstyle(fontprops[i].get('style', 'normal'))
                label.set_color(fontprops[i].get('color', 'black'))

        # Make the first y-tick label bold
        y_labels = ax.get_yticklabels()
        if len(y_labels) > 0:
            y_labels[0].set_fontweight('bold')
            
    # Call the function with current parameters
    style_heatmap_labels(ax, base_cell_types, additional_cell_types)
    
    # ----------------------------------------------------------------------------------------
    # Create a legend to explain the font styles
    # legend_elements = [
    #     matplotlib.lines.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=0, 
    #         label='Primary targets', markeredgecolor='black'),
    #     matplotlib.lines.Line2D([0], [0], marker='o', color='w', markerfacecolor='black', markersize=0, 
    #         label='Secondary targets', markeredgecolor='black', linestyle='None'),
    #     matplotlib.lines.Line2D([0], [0], marker='o', color='w', markerfacecolor='grey', markersize=0, 
    #         label='Non-targeted', markeredgecolor='grey')
    # ]

    # Create a separate legend axis
    # legend_ax = fig.add_axes([0.2, 0.2, 0.0, 0.1])  # Moved up by increasing y position from 0.02 to 0.7
    # legend_ax.axis('off')

    # # Add the legend to this axis
    # legend = legend_ax.legend(handles=legend_elements, loc='center', 
    #                         fontsize=6, frameon=False)

    # # Style the legend text to match the actual formatting
    # legend_texts = legend.get_texts()
    # legend_texts[0].set_fontweight('heavy')
    # legend_texts[1].set_fontstyle('italic')
    # legend_texts[2].set_color('grey')
    
    # # Add a legend for the black rectangle (r > 0.5)
    # r_value_legend = [
    #     matplotlib.patches.Patch(
    #         edgecolor='black',
    #         facecolor='none',
    #         linewidth=1,
    #         label='(r > 0.5)&(strong impact)&(FC>2)\n&(receptor expressed)'
    #     )
    # ]
    
    # # Create a separate legend axis for r-value
    # r_value_legend_ax = fig.add_axes([0.2, 0.1, 0.0, 0.1])
    # r_value_legend_ax.axis('off')
    
    # # Add the legend to this axis
    # r_value_legend = r_value_legend_ax.legend(
    #     handles=r_value_legend,
    #     loc='center',
    #     fontsize=6,
    #     frameon=False
    # )
    # ----------------------------------------------------------------------------------------
    
    # plot black rectangles around cells that meet criteria
    for cell_idx, cell_type in enumerate(ALL_CELL_TYPES):
        r_vals = r_vals_by_cell_type[cell_type]
        for cytokine in receptor_cytokines[1:]:
            potential_cell_types = df_all_secondary_responses[df_all_secondary_responses["primary_cytokine"] == introduce_greek_letters(test_cytokine)]
            potential_cell_types = potential_cell_types[potential_cell_types["secondary_cytokine"] == introduce_greek_letters(cytokine)]['responding_cell_type'].unique()
            if cell_type in potential_cell_types:
                cytokine_idx = receptor_cytokines.index(cytokine)
                rect = plt.Rectangle(
                    (cell_idx, cytokine_idx+1),
                    1, 1,
                    fill=False,
                    edgecolor='black',
                    linewidth=1,
                    linestyle='-',
                    clip_on=False,
                    zorder=100
                )
                ax.add_patch(rect)
                rect.set_visible(True)
    
    # ----------------------------------------------------------------------------------------

    apply_greek_labels_heatmap(ax)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis='y', length=0)
    ax.tick_params(axis='x', rotation=90, labelsize=5.5, length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)
    
    for format in ["png", "svg"]:
        plt.savefig(os.path.join(plot_folder, f"6.1.2_receptor_expression_{test_cytokine}.{format}"), 
                    bbox_inches="tight", 
                    dpi=300)